# 11. Exploratory Data Analysis (EDA)

This section presents the exploratory data analysis performed on the cleaned RavenStack dataset after the completion of SQL preparation, data validation, and cleaning. At this stage, the dataset is structurally reliable, relationally consistent, and analytically ready for descriptive and business-oriented exploration.

The purpose of this EDA is to move beyond basic descriptive summaries and uncover meaningful patterns across the major dimensions of the customer lifecycle in a subscription-based SaaS business. The analysis is designed to support later stages of the project, including churn interpretation, feature engineering, segmentation, and business reporting.

## EDA Objectives

The exploratory analysis in this section is intended to:

- understand the structure of the customer base and subscription portfolio
- examine how revenue, billing behavior, and trial exposure are distributed
- analyze product usage behavior and feature adoption patterns
- evaluate support activity, service burden, and customer satisfaction indicators
- explore churn-related business signals such as churn reasons and refund behavior
- identify descriptive patterns that may later support churn analysis and retention strategy development

## EDA Roadmap

The EDA will be organized into the following analytical themes:

### 11.1 Account and Subscription Overview
Review the structure of accounts and subscriptions, including industry mix, geography, referral source, plan tier, customer size, trial status, and billing frequency.

### 11.2 Revenue and Billing Analysis
Examine the distribution of MRR, ARR, zero-revenue subscriptions, and the relationship between revenue structure and trial or paid subscription status.

### 11.3 Product Usage Analysis
Analyze usage intensity, feature adoption breadth, usage duration, and product friction indicators such as error activity.

### 11.4 Support Experience Analysis
Explore ticket volume, escalation patterns, response effort, and support satisfaction behavior.

### 11.5 Churn Event Analysis
Review churn reason distributions, churn-related refunds, and reactivation-related signals.

### 11.6 Cross-Domain Analytical Patterns
Compare patterns across customer profile, subscription structure, usage behavior, support operations, and churn-related business signals.

## Expected Outcome

By the end of this section, the project should establish a clear descriptive understanding of the RavenStack customer lifecycle and highlight the most relevant patterns that will inform the next analytical stages of the project.
``

## Load Required Libraries and Final Cleaned Tables

This step initializes the Python environment for the exploratory data analysis (EDA) stage by importing the required libraries and loading the final cleaned CSV files generated after the data quality and cleaning process.

The goal of this step is to ensure that the notebook starts from the final validated analytical tables rather than from older intermediate files.

In [56]:
import pandas as pd
import numpy as np

accounts_clean = pd.read_csv("final_accounts_clean.csv")
subscriptions_clean = pd.read_csv("final_subscriptions_clean.csv")
feature_usage_clean = pd.read_csv("final_feature_usage_clean.csv")
support_tickets_clean = pd.read_csv("final_support_tickets_clean.csv")
churn_events_clean = pd.read_csv("final_churn_events_clean.csv")

tables_clean = {
    "accounts": accounts_clean,
    "subscriptions": subscriptions_clean,
    "feature_usage": feature_usage_clean,
    "support_tickets": support_tickets_clean,
    "churn_events": churn_events_clean
}

accounts = accounts_clean.copy()
subscriptions = subscriptions_clean.copy()
feature_usage = feature_usage_clean.copy()
support_tickets = support_tickets_clean.copy()
churn_events = churn_events_clean.copy()

for name, df in tables_clean.items():
    print(f"{name.upper()} -> shape: {df.shape}")

ACCOUNTS -> shape: (500, 10)
SUBSCRIPTIONS -> shape: (5000, 14)
FEATURE_USAGE -> shape: (25000, 8)
SUPPORT_TICKETS -> shape: (2000, 10)
CHURN_EVENTS -> shape: (600, 9)


## 11.1 Account and Subscription Overview

This step provides a structured overview of the account base and subscription portfolio in the RavenStack dataset. The objective is to understand the general composition of customers and subscriptions before moving into deeper analytical themes such as revenue, usage behavior, support activity, or churn patterns.

The analysis in this step focuses on the following questions:

- How many customer accounts and subscription records are present in the dataset?
- How are customer accounts distributed across industries, countries, referral sources, and initial plan tiers?
- How are subscriptions distributed across plan tiers, billing frequency, trial status, and churn status?
- What does customer size look like in terms of seat allocation?
- What is the overall structural composition of the subscription portfolio?

This step is important because it establishes the descriptive baseline of the business. By understanding the structure of accounts and subscriptions first, the later analytical sections can be interpreted within the correct commercial and customer context.
``

In [57]:
# =========================================================
# 11.1 Account and Subscription Overview
# =========================================================

# Use final cleaned tables
accounts_df = accounts_clean.copy()
subscriptions_df = subscriptions_clean.copy()

print("=== 11.1 Account and Subscription Overview ===")

# ---------------------------------------------------------
# A) High-level structural summary
# ---------------------------------------------------------
print("\n--- High-Level Structural Summary ---")

summary_metrics = {
    "Total Accounts": len(accounts_df),
    "Unique Accounts": accounts_df["account_id"].nunique(),
    "Total Subscriptions": len(subscriptions_df),
    "Unique Subscriptions": subscriptions_df["subscription_id"].nunique(),
    "Accounts with Trial Flag = True": int(accounts_df["is_trial"].fillna(False).sum()),
    "Accounts with Churn Flag = True": int(accounts_df["churn_flag"].fillna(False).sum()),
    "Subscriptions with Trial Flag = True": int(subscriptions_df["is_trial"].fillna(False).sum()),
    "Subscriptions with Churn Flag = True": int(subscriptions_df["churn_flag"].fillna(False).sum()),
}

for metric, value in summary_metrics.items():
    print(f"{metric}: {value}")

# ---------------------------------------------------------
# B) Customer size overview
# ---------------------------------------------------------
print("\n--- Customer Size Overview (Seats) ---")
display(accounts_df["seats"].describe())

print("\n--- Subscription Seat Overview ---")
display(subscriptions_df["seats"].describe())

# ---------------------------------------------------------
# C) Account-level categorical distributions
# ---------------------------------------------------------
print("\n--- Account-Level Distributions ---")

print("\nIndustry Distribution:")
display(accounts_df["industry"].value_counts(dropna=False))

print("\nCountry Distribution:")
display(accounts_df["country"].value_counts(dropna=False))

print("\nReferral Source Distribution:")
display(accounts_df["referral_source"].value_counts(dropna=False))

print("\nInitial Plan Tier Distribution (Accounts):")
display(accounts_df["plan_tier"].value_counts(dropna=False))

print("\nTrial Status Distribution (Accounts):")
display(accounts_df["is_trial"].value_counts(dropna=False))

print("\nChurn Flag Distribution (Accounts):")
display(accounts_df["churn_flag"].value_counts(dropna=False))

# ---------------------------------------------------------
# D) Subscription-level categorical distributions
# ---------------------------------------------------------
print("\n--- Subscription-Level Distributions ---")

print("\nPlan Tier Distribution (Subscriptions):")
display(subscriptions_df["plan_tier"].value_counts(dropna=False))

print("\nBilling Frequency Distribution:")
display(subscriptions_df["billing_frequency"].value_counts(dropna=False))

print("\nTrial Status Distribution (Subscriptions):")
display(subscriptions_df["is_trial"].value_counts(dropna=False))

print("\nChurn Flag Distribution (Subscriptions):")
display(subscriptions_df["churn_flag"].value_counts(dropna=False))

print("\nAuto-Renew Flag Distribution:")
display(subscriptions_df["auto_renew_flag"].value_counts(dropna=False))

# ---------------------------------------------------------
# E) Optional percentage views
# ---------------------------------------------------------
print("\n--- Percentage Views ---")

print("\nAccounts by Industry (%):")
display((accounts_df["industry"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nSubscriptions by Plan Tier (%):")
display((subscriptions_df["plan_tier"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nSubscriptions by Billing Frequency (%):")
display((subscriptions_df["billing_frequency"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nSubscriptions by Trial Status (%):")
display((subscriptions_df["is_trial"].value_counts(normalize=True, dropna=False) * 100).round(2))


=== 11.1 Account and Subscription Overview ===

--- High-Level Structural Summary ---
Total Accounts: 500
Unique Accounts: 500
Total Subscriptions: 5000
Unique Subscriptions: 5000
Accounts with Trial Flag = True: 97
Accounts with Churn Flag = True: 110
Subscriptions with Trial Flag = True: 778
Subscriptions with Churn Flag = True: 486

--- Customer Size Overview (Seats) ---


count    500.000000
mean      20.560000
std       21.044718
min        1.000000
25%        5.000000
50%       15.000000
75%       28.000000
max      163.000000
Name: seats, dtype: float64


--- Subscription Seat Overview ---


count    5000.000000
mean       29.852000
std        23.089771
min         1.000000
25%        14.000000
50%        24.000000
75%        40.000000
max       189.000000
Name: seats, dtype: float64


--- Account-Level Distributions ---

Industry Distribution:


...

plan_tier
Enterprise    34.46
Pro           33.50
Basic         32.04
Name: proportion, dtype: float64


Subscriptions by Billing Frequency (%):


billing_frequency
monthly    50.78
annual     49.22
Name: proportion, dtype: float64


Subscriptions by Trial Status (%):


is_trial
0    84.44
1    15.56
Name: proportion, dtype: float64

### 11.1 Interpretation — Account and Subscription Overview

The account and subscription overview establishes a clear structural baseline for the RavenStack business. The dataset contains **500 unique customer accounts** and **5,000 unique subscription records**, indicating that the subscription layer is much more granular than the account layer and that each account may hold multiple subscription records over time. This confirms that the business should be analyzed through both customer-level and subscription-level perspectives rather than through a single flat view.

From an account perspective, the customer portfolio appears **well diversified across industries**, with no single industry dominating the business excessively. The largest segments are **DevTools (22.6%)** and **FinTech (22.4%)**, followed by **Cybersecurity (20.0%)**, **HealthTech (19.2%)**, and **EdTech (15.8%)**. This suggests that RavenStack serves a relatively balanced B2B SaaS customer base rather than being heavily dependent on a single vertical. Such a structure is analytically useful because it allows later churn and retention analysis to compare customer behavior across sectors without extreme class imbalance.

Geographically, the account base is **highly concentrated in the United States**, which accounts for **291 out of 500 accounts**. The remaining accounts are distributed across smaller international markets such as the UK, India, Australia, Germany, Canada, and France. This concentration implies that any later account-level or revenue-level findings may be influenced primarily by the U.S. customer base, and geographic interpretation should therefore distinguish between the dominant market and the smaller secondary markets.

The customer acquisition profile is also relatively balanced. **Organic acquisition** is the strongest source, followed by **other**, **ads**, **event**, and **partner** channels, with no channel overwhelmingly dominating the account base. This is a useful early signal because it suggests that referral-source comparisons later in the project may be meaningful, especially when evaluating whether certain acquisition paths are associated with stronger subscription continuity or lower churn exposure.

At the account level, the initial commercial portfolio is also balanced. The initial plan-tier mix is distributed across **Pro (178 accounts)**, **Basic (168 accounts)**, and **Enterprise (154 accounts)**, which indicates that the business is not structurally overloaded toward low-tier or premium-tier customers. In other words, the starting plan distribution appears commercially diversified, which is important for later comparisons between initial customer positioning and later subscription behavior.

Trial exposure at the account level is relatively limited. Only **97 out of 500 accounts** are marked as trial accounts, while **110 accounts** carry an account-level churn flag. This means the business is primarily composed of non-trial customer relationships, while churn at the account level is present but not dominant. This gives early evidence that churn is meaningful enough to analyze, but not so extreme that it overwhelms the customer base structure.

From a subscription perspective, the commercial portfolio is even more balanced than the account portfolio. Subscription records are distributed almost evenly across **Enterprise (34.46%)**, **Pro (33.50%)**, and **Basic (32.04%)**, while **monthly billing (50.78%)** and **annual billing (49.22%)** are also nearly split. This is an analytically strong pattern because it means that future comparisons between billing frequency, plan tier, and retention-related outcomes can be made without severe class imbalance.

The subscription trial structure also shows a clear and interpretable pattern. Only **15.56% of subscriptions** are trial subscriptions, while the majority (**84.44%**) are non-trial. This confirms that the subscription portfolio is primarily revenue-generating rather than trial-heavy, which is consistent with a relatively mature SaaS customer base. At the same time, the fact that trial subscriptions still represent a meaningful minority suggests that trial-related behavior should remain a dedicated analytical segment in later stages, especially when evaluating zero-revenue subscriptions or early-stage churn dynamics.

Finally, the seat distribution indicates meaningful variation in customer size. At the account level, the average number of seats is **20.56**, while at the subscription level the average rises to **29.85**, with wide dispersion in both cases. This suggests that RavenStack serves a mix of smaller and larger customers rather than a narrowly concentrated account-size profile. Because seat count is a strong proxy for account scale and potential commercial value, it should be treated as an important dimension in later segmentation and churn-focused analysis.

Overall, this step shows that RavenStack has a **balanced multi-segment SaaS structure** across industries, plan tiers, billing models, and customer sizes, while still showing clear commercial concentration in the U.S. market. The account and subscription base appears large enough, varied enough, and commercially structured enough to support deeper analysis in later stages, particularly around revenue behavior, product usage, support burden, and churn-related segmentation.

## 11.2 Revenue and Billing Analysis

This step examines the commercial structure of the RavenStack subscription portfolio. The goal is to understand how recurring revenue is distributed across the business, how billing behavior is organized, and how trial-related subscriptions differ from revenue-generating subscriptions.

The analysis in this step focuses on the following questions:

- What do the main recurring revenue metrics look like across subscriptions?
- How are MRR and ARR distributed across the portfolio?
- How common are zero-revenue subscriptions, and how are they related to trial status?
- How is revenue distributed across plan tiers and billing frequency?
- Is recurring revenue broadly distributed, or concentrated in particular commercial segments?

This step is important because it establishes the commercial baseline of the business before moving into product usage, support behavior, and churn-related analysis.

In [58]:
# =========================================================
# 11.2 Revenue and Billing Analysis
# =========================================================

subscriptions_df = subscriptions_clean.copy()

print("=== 11.2 Revenue and Billing Analysis ===")

# ---------------------------------------------------------
# A) Core revenue summary
# ---------------------------------------------------------
print("\n--- Core Revenue Summary ---")

revenue_summary = {
    "Total Subscriptions": len(subscriptions_df),
    "Total MRR": subscriptions_df["mrr_amount"].sum(),
    "Total ARR": subscriptions_df["arr_amount"].sum(),
    "Average MRR per Subscription": subscriptions_df["mrr_amount"].mean(),
    "Average ARR per Subscription": subscriptions_df["arr_amount"].mean(),
    "Median MRR per Subscription": subscriptions_df["mrr_amount"].median(),
    "Median ARR per Subscription": subscriptions_df["arr_amount"].median(),
    "Zero-Revenue Subscriptions": int(((subscriptions_df["mrr_amount"] == 0) | (subscriptions_df["arr_amount"] == 0)).sum())
}

for metric, value in revenue_summary.items():
    print(f"{metric}: {value}")

# ---------------------------------------------------------
# B) Revenue distribution overview
# ---------------------------------------------------------
print("\n--- Revenue Distribution Overview ---")
print("\nMRR Summary Statistics:")
display(subscriptions_df["mrr_amount"].describe())

print("\nARR Summary Statistics:")
display(subscriptions_df["arr_amount"].describe())

# ---------------------------------------------------------
# C) Billing frequency structure
# ---------------------------------------------------------
print("\n--- Billing Frequency Structure ---")
print("\nBilling Frequency Distribution:")
display(subscriptions_df["billing_frequency"].value_counts(dropna=False))

print("\nBilling Frequency Distribution (%):")
display((subscriptions_df["billing_frequency"].value_counts(normalize=True, dropna=False) * 100).round(2))

# ---------------------------------------------------------
# D) Revenue by plan tier
# ---------------------------------------------------------
print("\n--- Revenue by Plan Tier ---")

revenue_by_plan = subscriptions_df.groupby("plan_tier").agg(
    subscriptions_count=("subscription_id", "count"),
    total_mrr=("mrr_amount", "sum"),
    total_arr=("arr_amount", "sum"),
    avg_mrr=("mrr_amount", "mean"),
    avg_arr=("arr_amount", "mean"),
    median_mrr=("mrr_amount", "median"),
    median_arr=("arr_amount", "median")
).sort_values("total_mrr", ascending=False)

display(revenue_by_plan)

# ---------------------------------------------------------
# E) Revenue by billing frequency
# ---------------------------------------------------------
print("\n--- Revenue by Billing Frequency ---")

revenue_by_billing = subscriptions_df.groupby("billing_frequency").agg(
    subscriptions_count=("subscription_id", "count"),
    total_mrr=("mrr_amount", "sum"),
    total_arr=("arr_amount", "sum"),
    avg_mrr=("mrr_amount", "mean"),
    avg_arr=("arr_amount", "mean")
).sort_values("total_mrr", ascending=False)

display(revenue_by_billing)

# ---------------------------------------------------------
# F) Zero-revenue subscriptions
# ---------------------------------------------------------
print("\n--- Zero-Revenue Subscription Analysis ---")

zero_revenue_mask = (subscriptions_df["mrr_amount"] == 0) | (subscriptions_df["arr_amount"] == 0)

print("Zero-revenue subscription count:")
print(zero_revenue_mask.sum())

print("\nZero-revenue subscriptions by trial status:")
display(
    subscriptions_df.assign(zero_revenue=zero_revenue_mask)
    .groupby("is_trial")["zero_revenue"]
    .agg(["mean", "sum", "count"])
)

print("\nZero-revenue subscriptions by plan tier:")
display(
    subscriptions_df.assign(zero_revenue=zero_revenue_mask)
    .groupby("plan_tier")["zero_revenue"]
    .agg(["mean", "sum", "count"])
    .sort_values("sum", ascending=False)
)

print("\nSample zero-revenue subscriptions:")
display(
    subscriptions_df.loc[
        zero_revenue_mask,
        ["subscription_id", "account_id", "plan_tier", "is_trial", "billing_frequency", "mrr_amount", "arr_amount"]
    ].head(10)
)

# ---------------------------------------------------------
# G) Revenue concentration check
# ---------------------------------------------------------
print("\n--- Revenue Concentration Check ---")

top_10_share = (
    subscriptions_df.sort_values("mrr_amount", ascending=False)["mrr_amount"].head(10).sum()
    / subscriptions_df["mrr_amount"].sum()
) * 100

top_50_share = (
    subscriptions_df.sort_values("mrr_amount", ascending=False)["mrr_amount"].head(50).sum()
    / subscriptions_df["mrr_amount"].sum()
) * 100

print(f"Top 10 subscriptions share of total MRR: {top_10_share:.2f}%")
print(f"Top 50 subscriptions share of total MRR: {top_50_share:.2f}%")


=== 11.2 Revenue and Billing Analysis ===

--- Core Revenue Summary ---
Total Subscriptions: 5000
Total MRR: 11338747
Total ARR: 136064964
Average MRR per Subscription: 2267.7494
Average ARR per Subscription: 27212.9928
Median MRR per Subscription: 931.0
Median ARR per Subscription: 11172.0
Zero-Revenue Subscriptions: 778

--- Revenue Distribution Overview ---

MRR Summary Statistics:


count     5000.000000
mean      2267.749400
std       3421.375348
min          0.000000
25%        285.000000
50%        931.000000
75%       2786.000000
max      33830.000000
Name: mrr_amount, dtype: float64


ARR Summary Statistics:


count      5000.000000
mean      27212.992800
std       41056.504178
min           0.000000
25%        3420.000000
50%       11172.000000
75%       33432.000000
max      405960.000000
Name: arr_amount, dtype: float64


--- Billing Frequency Structure ---

Billing Frequency Distribution:


...


Zero-revenue subscriptions by plan tier:


,mean,sum,count
plan_tier,,,
Enterprise,0.158445,273,1723
Pro,0.154627,259,1675
Basic,0.153558,246,1602



Sample zero-revenue subscriptions:


,subscription_id,account_id,plan_tier,is_trial,billing_frequency,mrr_amount,arr_amount
2,S-51c0d1,A-659280,Enterprise,1,annual,0,0
11,S-79a9e1,A-3a5ad8,Enterprise,1,annual,0,0
14,S-428e9a,A-8ae3fc,Enterprise,1,monthly,0,0
16,S-13939b,A-86902e,Pro,1,monthly,0,0
20,S-c27134,A-dec005,Pro,1,monthly,0,0
32,S-2877e6,A-95b24a,Enterprise,1,monthly,0,0
34,S-d7d398,A-fb186e,Enterprise,1,annual,0,0
35,S-ddad91,A-40906c,Enterprise,1,annual,0,0
37,S-bb1cf1,A-3a5ad8,Enterprise,1,monthly,0,0
49,S-ad39ef,A-e40f45,Pro,1,annual,0,0



--- Revenue Concentration Check ---
Top 10 subscriptions share of total MRR: 2.34%
Top 50 subscriptions share of total MRR: 8.86%


### 11.2 Interpretation — Revenue and Billing Analysis

The revenue and billing analysis highlights a **commercial structure that is both substantial and well distributed**, while also revealing a clear distinction between trial and revenue-generating subscriptions. Across the 5,000 subscription records, RavenStack generates approximately **11.34 million in total MRR** and **136.06 million in total ARR**, which confirms that the subscription portfolio carries meaningful recurring revenue scale. At the same time, the difference between the **average MRR (2,267.75)** and the **median MRR (931.0)** indicates a strongly **right-skewed revenue distribution**. In practical terms, this means that while many subscriptions generate modest recurring revenue, a smaller group of higher-value subscriptions significantly pulls the average upward. The same pattern is visible in ARR, where the average also sits far above the median. This is an important early commercial signal because it suggests that the portfolio contains a mix of lower-value and higher-value subscriptions rather than a uniform pricing structure.

Billing behavior appears **remarkably balanced** across the portfolio. Monthly subscriptions account for **50.78%** of all records, while annual subscriptions account for **49.22%**, which indicates that the business is not dependent on a single billing model. This is analytically valuable because it creates a strong foundation for later comparisons between billing frequency, product behavior, and churn-related outcomes. It also suggests that revenue risk and retention behavior may need to be interpreted separately across both billing groups rather than assuming one dominant contract structure.

The plan-tier analysis shows a very clear commercial hierarchy. Although the subscription counts are relatively balanced across **Enterprise**, **Pro**, and **Basic**, the revenue contribution is not balanced at all. **Enterprise subscriptions generate by far the largest share of recurring revenue**, with total MRR and ARR far exceeding the other two tiers. In addition, the average and median revenue values for Enterprise subscriptions are substantially higher than those of Pro and Basic subscriptions. This indicates that the subscription portfolio is **commercially diversified by volume, but highly differentiated by value**. In other words, the business may have a fairly even distribution of subscription counts across tiers, but revenue is concentrated much more heavily in the premium part of the portfolio. This is strategically important because later churn analysis should not treat all subscriptions as financially equivalent.

The revenue comparison by billing frequency shows a different pattern. Unlike plan tier, billing frequency does **not create a strong revenue separation**. Monthly and annual subscriptions contribute very similar total MRR and ARR, and their average recurring revenue values are also close. This suggests that billing frequency, at least descriptively, is not the main driver of revenue concentration in the portfolio. Instead, plan tier appears to be the more dominant commercial differentiator. This distinction will be important later when interpreting whether churn risk is driven more by contract structure or by customer value level.

One of the most important findings in this step concerns **zero-revenue subscriptions**. The analysis confirms that all **778 zero-revenue subscriptions** belong to the trial segment, while none of the non-trial subscriptions have zero MRR or ARR. This is a strong sign that the zero-revenue pattern is **fully aligned with business logic**, not with a data quality issue. It also means that trial subscriptions form a structurally different segment of the portfolio and should not be mixed with paid subscriptions in revenue-focused interpretations. From an analytical perspective, this reinforces the idea that trial and paid subscription behavior must be treated separately whenever monetization or financial risk is discussed.

The plan-tier view of zero-revenue subscriptions also supports this interpretation. Zero-revenue subscriptions are distributed relatively evenly across Enterprise, Pro, and Basic tiers, which indicates that the zero-revenue pattern is **not driven by plan tier itself**, but rather by the trial mechanism that cuts across the portfolio. Similarly, the churn-status breakdown suggests that zero-revenue subscriptions are not strongly concentrated in churned or retained groups. This further supports the conclusion that zero-revenue is a **commercial state linked to trial exposure**, rather than a symptom of pricing inconsistency or churn-specific anomaly.

Finally, the revenue concentration check suggests that although the revenue distribution is right-skewed, it is **not extremely concentrated in a tiny number of subscriptions**. The top 10 subscriptions account for **2.34% of total MRR**, and the top 50 account for **8.86%**. This indicates that high-value subscriptions matter, but RavenStack’s revenue base is not dominated by a very small set of extreme outliers. Instead, the business appears to benefit from a relatively broad recurring revenue base, with meaningful premium concentration but not excessive dependence on a handful of subscriptions.

Overall, this step shows that the RavenStack subscription business is **commercially balanced in structure but value-differentiated in revenue contribution**. The portfolio contains a near-even split between monthly and annual billing, a relatively balanced count of subscriptions across plan tiers, and a clear premium-revenue concentration in Enterprise subscriptions. At the same time, zero-revenue subscriptions are fully explained by trial status and should be treated as analytically valid but commercially distinct records. These findings establish a strong commercial baseline for the next stages of the analysis, especially when examining how revenue patterns relate to usage behavior, support burden, and churn exposure.

## 11.3 Product Usage Analysis

This step explores how customers interact with the product at the feature-usage level. The objective is to understand the overall structure of product engagement across subscriptions and identify descriptive patterns in usage intensity, feature adoption breadth, usage duration, and product friction.

The analysis in this step focuses on the following questions:

- How large is the feature usage layer of the business?
- How are usage count, usage duration, and error activity distributed?
- Which features are used most frequently across the platform?
- How broadly do subscriptions adopt product features?
- How common is beta-feature usage?
- Do usage patterns vary across subscription plan tiers?

This step is important because product engagement is one of the strongest behavioral layers in a SaaS business. Understanding usage patterns at this stage provides the descriptive foundation for later churn interpretation, feature engineering, and retention-focused analysis.
``

In [59]:
# =========================================================
# 11.3 Product Usage Analysis
# =========================================================

feature_usage_df = feature_usage_clean.copy()
subscriptions_df = subscriptions_clean.copy()

print("=== 11.3 Product Usage Analysis ===")

# ---------------------------------------------------------
# A) High-level usage summary
# ---------------------------------------------------------
print("\n--- High-Level Usage Summary ---")

usage_summary = {
    "Total Usage Records": len(feature_usage_df),
    "Unique Usage IDs": feature_usage_df["usage_id"].nunique(),
    "Unique Subscriptions in Usage Table": feature_usage_df["subscription_id"].nunique(),
    "Unique Features": feature_usage_df["feature_name"].nunique(),
    "Total Usage Count": feature_usage_df["usage_count"].sum(),
    "Total Usage Duration (Seconds)": feature_usage_df["usage_duration_secs"].sum(),
    "Total Error Count": feature_usage_df["error_count"].sum(),
    "Average Usage Count per Record": feature_usage_df["usage_count"].mean(),
    "Average Usage Duration per Record": feature_usage_df["usage_duration_secs"].mean(),
    "Average Error Count per Record": feature_usage_df["error_count"].mean(),
}

for metric, value in usage_summary.items():
    print(f"{metric}: {value}")

# ---------------------------------------------------------
# B) Core usage distributions
# ---------------------------------------------------------
print("\n--- Core Usage Distributions ---")

print("\nUsage Count Summary:")
display(feature_usage_df["usage_count"].describe())

print("\nUsage Duration Summary:")
display(feature_usage_df["usage_duration_secs"].describe())

print("\nError Count Summary:")
display(feature_usage_df["error_count"].describe())

# ---------------------------------------------------------
# C) Most-used features
# ---------------------------------------------------------
print("\n--- Most Used Features ---")

top_features_by_events = feature_usage_df["feature_name"].value_counts().head(10)
print("\nTop Features by Number of Usage Records:")
display(top_features_by_events)

top_features_by_usage = (
    feature_usage_df.groupby("feature_name")["usage_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop Features by Total Usage Count:")
display(top_features_by_usage)

top_features_by_duration = (
    feature_usage_df.groupby("feature_name")["usage_duration_secs"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop Features by Total Usage Duration:")
display(top_features_by_duration)

# ---------------------------------------------------------
# D) Feature adoption breadth by subscription
# ---------------------------------------------------------
print("\n--- Feature Adoption Breadth by Subscription ---")

subscription_feature_adoption = (
    feature_usage_df.groupby("subscription_id")
    .agg(
        usage_event_count=("usage_id", "count"),
        unique_features_used=("feature_name", "nunique"),
        total_usage_count=("usage_count", "sum"),
        total_usage_duration_secs=("usage_duration_secs", "sum"),
        total_error_count=("error_count", "sum"),
        used_beta_feature=("is_beta_feature", "max")
    )
    .reset_index()
)

print("\nSubscription-Level Usage Summary:")
display(subscription_feature_adoption.describe())

# ---------------------------------------------------------
# E) Beta feature usage
# ---------------------------------------------------------
print("\n--- Beta Feature Usage ---")

print("\nBeta Feature Flag Distribution:")
display(feature_usage_df["is_beta_feature"].value_counts(dropna=False))

print("\nSubscriptions Using at Least One Beta Feature:")
display(subscription_feature_adoption["used_beta_feature"].value_counts(dropna=False))

# ---------------------------------------------------------
# F) Usage by plan tier
# ---------------------------------------------------------
print("\n--- Usage by Plan Tier ---")

usage_with_plan = subscription_feature_adoption.merge(
    subscriptions_df[["subscription_id", "plan_tier"]],
    on="subscription_id",
    how="left"
)

usage_by_plan = usage_with_plan.groupby("plan_tier").agg(
    subscriptions_count=("subscription_id", "count"),
    avg_unique_features_used=("unique_features_used", "mean"),
    avg_total_usage_count=("total_usage_count", "mean"),
    avg_total_usage_duration_secs=("total_usage_duration_secs", "mean"),
    avg_total_error_count=("total_error_count", "mean"),
    beta_feature_subscription_rate=("used_beta_feature", "mean")
).sort_values("avg_total_usage_count", ascending=False)

display(usage_by_plan)

# ---------------------------------------------------------
# G) Error concentration
# ---------------------------------------------------------
print("\n--- Error Concentration ---")

error_by_feature = (
    feature_usage_df.groupby("feature_name")["error_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nTop Features by Total Error Count:")
display(error_by_feature)

error_rate_by_feature = (
    feature_usage_df.groupby("feature_name")
    .agg(total_usage=("usage_count", "sum"), total_errors=("error_count", "sum"))
)

error_rate_by_feature["error_rate_per_usage"] = (
    error_rate_by_feature["total_errors"] / error_rate_by_feature["total_usage"].replace(0, np.nan)
)

print("\nTop Features by Error Rate per Usage:")
display(
    error_rate_by_feature.sort_values("error_rate_per_usage", ascending=False)
    .head(10)
)

=== 11.3 Product Usage Analysis ===

--- High-Level Usage Summary ---
Total Usage Records: 25000
Unique Usage IDs: 24979
Unique Subscriptions in Usage Table: 4967
Unique Features: 40
Total Usage Count: 250525
Total Usage Duration (Seconds): 76055072
Total Error Count: 14107
Average Usage Count per Record: 10.021
Average Usage Duration per Record: 3042.20288
Average Error Count per Record: 0.56428

--- Core Usage Distributions ---

Usage Count Summary:


count    25000.000000
mean        10.021000
std          3.143729
min          0.000000
25%          8.000000
50%         10.000000
75%         12.000000
max         26.000000
Name: usage_count, dtype: float64


Usage Duration Summary:


count    25000.000000
mean      3042.202880
std       2056.544615
min          0.000000
25%       1350.000000
50%       2760.000000
75%       4400.000000
max      12696.000000
Name: usage_duration_secs, dtype: float64


Error Count Summary:


...

,subscriptions_count,avg_unique_features_used,avg_total_usage_count,avg_total_usage_duration_secs,avg_total_error_count,beta_feature_subscription_rate
plan_tier,,,,,,
Pro,1665,4.769369,50.787387,15398.834234,2.893093,0.410210
Basic,1588,4.731108,50.307305,15169.778967,2.792821,0.397355
Enterprise,1714,4.694866,50.219370,15359.628938,2.832555,0.413652



--- Error Concentration ---

Top Features by Total Error Count:


feature_name
feature_4     418
feature_26    417
feature_9     404
feature_2     401
feature_34    397
feature_16    396
feature_40    380
feature_13    372
feature_36    371
feature_29    371
Name: error_count, dtype: int64


Top Features by Error Rate per Usage:


,total_usage,total_errors,error_rate_per_usage
feature_name,,,
feature_4,6374,418,0.065579
feature_9,6207,404,0.065088
feature_26,6470,417,0.064451
feature_16,6273,396,0.063128
feature_18,5850,369,0.063077
feature_2,6525,401,0.061456
feature_40,6222,380,0.061074
feature_13,6099,372,0.060994
feature_34,6536,397,0.060741


### 11.3 Interpretation — Product Usage Analysis

The product usage analysis shows that RavenStack has a **broad and active engagement layer** across the subscription portfolio. The dataset contains **25,000 usage records**, covering **4,967 unique subscriptions** and **40 distinct features**. This confirms that product interaction is not limited to a small subset of subscriptions or functions; instead, usage activity is distributed across a wide portion of the platform. At the same time, the fact that the number of unique usage IDs (**24,979**) is slightly below the total number of usage records (**25,000**) is consistent with the previously documented non-critical duplication issue in `usage_id`, which does not materially affect the subscription-level analytical use of the table.

At the record level, product engagement appears **stable and relatively concentrated around moderate usage levels**. The average usage count per record is **10.02**, while the median is **10**, which indicates that the distribution is fairly centered rather than highly distorted by extreme outliers. A similar pattern is visible in usage duration, where the average record lasts approximately **3,042 seconds**, and the median is **2,760 seconds**. These patterns suggest that most feature interactions are neither trivial nor exceptionally long, but instead reflect a fairly regular product-engagement rhythm.

The error layer appears **present but not dominant**. The average error count per usage record is **0.56**, and the median is **0**, which shows that many product interactions occur without errors, while a subset of records contributes disproportionately to total friction. In practical terms, this suggests that product friction exists but is not universally spread across all usage events. This will be especially important later when comparing behavioral engagement to churn-related signals or when designing feature-level friction indicators.

The feature-level analysis shows that engagement is **not monopolized by a single feature**, but there is still meaningful variation in usage concentration. The most frequently observed features by number of usage records—such as **feature_12, feature_32, feature_6, feature_17, and feature_34**—appear repeatedly across the portfolio, while the ranking by total usage count and total usage duration shows a slightly different ordering. This suggests that some features are used often, while others may generate fewer but heavier interactions. In other words, product engagement has both a **frequency dimension** and an **intensity dimension**, and the two do not always point to exactly the same features.

At the subscription level, the usage summary provides one of the most useful descriptive findings in this step. On average, each subscription is associated with approximately **5 usage events**, **4.73 unique features**, **50 total usage actions**, and more than **15,000 seconds of aggregated usage duration**. This indicates that subscriptions tend to engage with more than one feature and that product interaction is not limited to one-off behavior. The distribution of `unique_features_used` also shows that some subscriptions adopt the product more broadly than others, which is likely to become a key variable later in feature engineering and churn interpretation.

Beta-feature exposure appears **meaningful but not dominant**. At the record level, beta-feature usage represents about **2,544 out of 25,000 usage records**, while at the subscription level **2,023 subscriptions** have used at least one beta feature. This means that beta-feature interaction is not rare, but it also does not define the majority of the usage base. From a product perspective, this suggests that beta adoption is present among a substantial minority of subscriptions and may later help differentiate more experimental or higher-engagement user segments.

One of the most important findings in this step is that **usage behavior appears remarkably similar across plan tiers**. The average number of unique features used, total usage count, total usage duration, total error count, and beta-feature adoption rate are all very close across **Pro, Basic, and Enterprise** subscriptions. This is analytically important because it suggests that, at least descriptively, **plan tier alone does not create major differences in raw usage intensity**. In other words, higher-value subscription tiers do not automatically correspond to dramatically higher product usage in the current data. This insight may become especially important later if product engagement turns out to be a stronger churn-related signal than the commercial plan itself.

Finally, the error concentration analysis suggests that product friction is **not evenly distributed across features**. Some features—such as **feature_4, feature_26, feature_9, feature_2, and feature_34**—contribute more heavily to the overall error burden, both in absolute terms and relative to usage volume. This is a significant descriptive signal because it implies that product friction may be **feature-specific rather than platform-wide**. From an analytical standpoint, these features may become candidates for closer monitoring later in the project, especially when connecting product quality signals to support activity or churn behavior.

Overall, this step shows that RavenStack has a **healthy and broad usage layer**, with moderate usage intensity, multi-feature adoption, meaningful beta exposure, and localized product friction. The most important analytical takeaway is that product behavior appears rich enough to support deeper behavioral analysis, while also suggesting that engagement breadth and feature-level error concentration may be more informative than plan tier alone in later churn-focused stages.


## 11.4 Support Experience Analysis

This step explores the support layer of the RavenStack dataset in order to understand how customers interact with support operations and how service-related burden is distributed across accounts.

The analysis in this step focuses on the following questions:

- How large is the support layer of the business?
- How are support tickets distributed across priority levels?
- How common are escalations?
- What do response and resolution times look like across the portfolio?
- How is customer satisfaction distributed after applying the analytical treatment of `satisfaction_score = 0` as “not rated”?
- Are support-related patterns concentrated, or broadly distributed across accounts?

This step is important because support behavior represents a key operational dimension of the customer experience. High support burden, frequent escalations, slow response effort, or weak satisfaction outcomes may later become relevant signals in churn interpretation and customer-risk segmentation.

In [60]:
# =========================================================
# 11.4 Support Experience Analysis
# =========================================================

support_df = support_tickets_clean.copy()

print("=== 11.4 Support Experience Analysis ===")

# ---------------------------------------------------------
# A) Analytical preparation
# ---------------------------------------------------------
# Treat satisfaction_score = 0 as "not rated" if the analytical column does not already exist
if "satisfaction_score_analytical" not in support_df.columns:
    support_df["satisfaction_score_analytical"] = pd.to_numeric(
        support_df["satisfaction_score"], errors="coerce"
    ).mask(pd.to_numeric(support_df["satisfaction_score"], errors="coerce") == 0, pd.NA).astype("Int64")

# ---------------------------------------------------------
# B) High-level support summary
# ---------------------------------------------------------
print("\n--- High-Level Support Summary ---")

support_summary = {
    "Total Support Tickets": len(support_df),
    "Unique Tickets": support_df["ticket_id"].nunique(),
    "Unique Accounts with Tickets": support_df["account_id"].nunique(),
    "Escalated Tickets": int(pd.to_numeric(support_df["escalation_flag"], errors="coerce").fillna(0).sum()),
    "Rated Tickets (Analytical Satisfaction)": int(support_df["satisfaction_score_analytical"].notna().sum()),
    "Unrated Tickets (Score 0 treated as missing)": int(support_df["satisfaction_score_analytical"].isna().sum()),
}

for metric, value in support_summary.items():
    print(f"{metric}: {value}")

# ---------------------------------------------------------
# C) Priority distribution
# ---------------------------------------------------------
print("\n--- Ticket Priority Distribution ---")

print("\nPriority Distribution:")
display(support_df["priority"].value_counts(dropna=False))

print("\nPriority Distribution (%):")
display((support_df["priority"].value_counts(normalize=True, dropna=False) * 100).round(2))

# ---------------------------------------------------------
# D) Escalation analysis
# ---------------------------------------------------------
print("\n--- Escalation Analysis ---")

print("\nEscalation Flag Distribution:")
display(support_df["escalation_flag"].value_counts(dropna=False))

print("\nEscalation Rate:")
print((pd.to_numeric(support_df["escalation_flag"], errors="coerce").fillna(0).mean() * 100).round(2), "%")

print("\nEscalation by Priority:")
display(
    support_df.assign(escalation_flag_num=pd.to_numeric(support_df["escalation_flag"], errors="coerce"))
    .groupby("priority")["escalation_flag_num"]
    .agg(["mean", "sum", "count"])
    .sort_values("mean", ascending=False)
)

# ---------------------------------------------------------
# E) Response and resolution time analysis
# ---------------------------------------------------------
print("\n--- Response and Resolution Time Analysis ---")

print("\nResolution Time Summary (hours):")
display(pd.to_numeric(support_df["resolution_time_hours"], errors="coerce").describe())

print("\nFirst Response Time Summary (minutes):")
display(pd.to_numeric(support_df["first_response_time_minutes"], errors="coerce").describe())

print("\nAverage Response and Resolution Time by Priority:")
display(
    support_df.groupby("priority").agg(
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        median_resolution_time_hours=("resolution_time_hours", "median"),
        avg_first_response_time_minutes=("first_response_time_minutes", "mean"),
        median_first_response_time_minutes=("first_response_time_minutes", "median")
    ).sort_values("avg_resolution_time_hours", ascending=False)
)

# ---------------------------------------------------------
# F) Satisfaction analysis
# ---------------------------------------------------------
print("\n--- Satisfaction Analysis ---")

print("\nOriginal Satisfaction Score Distribution:")
display(support_df["satisfaction_score"].value_counts(dropna=False).sort_index())

print("\nAnalytical Satisfaction Score Distribution (0 treated as missing):")
display(support_df["satisfaction_score_analytical"].value_counts(dropna=False).sort_index())

print("\nAverage Analytical Satisfaction Score:")
print(support_df["satisfaction_score_analytical"].mean())

print("\nAnalytical Satisfaction by Priority:")
display(
    support_df.groupby("priority")["satisfaction_score_analytical"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

# ---------------------------------------------------------
# G) Account-level support burden
# ---------------------------------------------------------
print("\n--- Account-Level Support Burden ---")

support_by_account = (
    support_df.groupby("account_id")
    .agg(
        ticket_count=("ticket_id", "count"),
        escalation_count=("escalation_flag", lambda x: pd.to_numeric(x, errors="coerce").fillna(0).sum()),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_minutes=("first_response_time_minutes", "mean"),
        avg_satisfaction_score=("satisfaction_score_analytical", "mean")
    )
    .reset_index()
)

print("\nSupport Burden Summary by Account:")
display(support_by_account.describe())

print("\nTop 10 Accounts by Ticket Volume:")
display(
    support_by_account.sort_values("ticket_count", ascending=False).head(10)
)

print("\nTop 10 Accounts by Escalation Count:")
display(
    support_by_account.sort_values("escalation_count", ascending=False).head(10)
)


=== 11.4 Support Experience Analysis ===

--- High-Level Support Summary ---
Total Support Tickets: 2000
Unique Tickets: 2000
Unique Accounts with Tickets: 492
Escalated Tickets: 95
Rated Tickets (Analytical Satisfaction): 1175
Unrated Tickets (Score 0 treated as missing): 825

--- Ticket Priority Distribution ---

Priority Distribution:


priority
urgent    514
high      510
medium    491
low       485
Name: count, dtype: int64


Priority Distribution (%):


priority
urgent    25.70
high      25.50
medium    24.55
low       24.25
Name: proportion, dtype: float64


--- Escalation Analysis ---

Escalation Flag Distribution:


...

,ticket_count,escalation_count,avg_resolution_time_hours,avg_first_response_time_minutes,avg_satisfaction_score
count,492.000000,492.000000,492.000000,492.000000,466.000000
mean,4.065041,0.193089,36.238515,88.560094,3.964163
std,1.837557,0.420107,11.823256,27.942216,0.570420
min,1.000000,0.000000,1.000000,1.000000,3.000000
25%,3.000000,0.000000,29.000000,71.333333,3.500000
50%,4.000000,0.000000,36.000000,88.291667,4.000000
75%,5.000000,0.000000,43.000000,106.250000,4.333333
max,11.000000,3.000000,72.000000,175.000000,5.000000



Top 10 Accounts by Ticket Volume:


,account_id,ticket_count,escalation_count,avg_resolution_time_hours,avg_first_response_time_minutes,avg_satisfaction_score
358,A-bb3bd4,11,0,32.727273,106.363636,4.142857
117,A-3ce5b8,10,0,42.500000,101.800000,3.714286
157,A-503d5a,9,0,36.555556,81.777778,4.000000
42,A-139c3b,9,0,28.444444,66.111111,4.500000
263,A-86902e,9,0,40.888889,71.444444,3.500000
453,A-e98302,9,0,44.444444,99.444444,4.142857
424,A-dcc3f1,9,0,35.666667,83.444444,3.800000
459,A-ed510f,9,0,44.555556,82.555556,4.000000
378,A-c42f1f,9,1,31.333333,81.888889,4.000000
133,A-443b7c,8,0,29.250000,49.000000,4.400000



Top 10 Accounts by Escalation Count:


,account_id,ticket_count,escalation_count,avg_resolution_time_hours,avg_first_response_time_minutes,avg_satisfaction_score
312,A-a0ca4e,6,3,39.166667,89.833333,3.800000
132,A-4437e4,6,2,34.666667,125.333333,3.000000
229,A-7641d3,5,2,45.200000,124.600000,4.333333
246,A-7df7a7,3,1,28.000000,93.000000,4.000000
249,A-7f4db3,5,1,16.600000,91.400000,3.250000
192,A-600734,7,1,52.571429,89.428571,3.666667
201,A-684255,4,1,14.250000,97.750000,4.666667
208,A-6a4e2d,2,1,18.500000,141.000000,4.500000
210,A-6c093d,3,1,40.333333,68.333333,5.000000
225,A-72799b,2,1,33.500000,80.000000,4.500000


### 11.4 Interpretation — Support Experience Analysis

The support analysis shows that RavenStack operates with a **broad but manageable support burden**. The dataset contains **2,000 unique support tickets** distributed across **492 customer accounts**, which indicates that support interaction is widespread across the customer base rather than concentrated in only a small number of accounts. On average, each supported account generated just over **4 tickets**, suggesting that support demand is meaningful but not excessively heavy at the account level.

From a priority perspective, ticket volume is **remarkably balanced** across all four severity levels. Urgent, high, medium, and low tickets each represent roughly one quarter of the support portfolio, which implies that the support layer is not dominated by extreme-severity cases. This balanced structure suggests that RavenStack faces a mixed operational workload rather than a heavily crisis-driven support environment.

Escalations appear to be **relatively limited**. Only **95 tickets**, or **4.75%** of the entire support volume, were escalated. In addition, escalation rates are very similar across priority levels, with no category standing out as disproportionately escalated. This indicates that escalation behavior is not concentrated in only urgent cases and that most support activity appears to be resolved within the standard support process. Operationally, this suggests that the support function is generally stable and not overwhelmed by severe unresolved issues.

Response and resolution times point to a **moderate service effort profile**. The average resolution time is approximately **35.9 hours**, while the average first-response time is around **88.5 minutes**. The distributions also appear fairly centered, with medians close to the means, which suggests that support performance is relatively stable rather than being heavily distorted by a few extreme outliers. Interestingly, urgent tickets do not show substantially faster handling than the other priority groups. In fact, urgent tickets have slightly lower average response and resolution times than high-priority tickets, but the differences across priorities remain relatively modest. This may indicate that the support process is fairly standardized across ticket classes rather than sharply differentiated by priority level.

Satisfaction analysis provides a more nuanced view of customer support outcomes. The original ticket ratings show that **825 records were coded as `0`**, while the remaining rated tickets were mostly concentrated at **3, 4, and 5**. After applying the project’s analytical rule of treating `satisfaction_score = 0` as **not rated** rather than as a true dissatisfaction score, the usable rated subset consists of **1,175 tickets**, with an average analytical satisfaction score of **3.98**. This indicates that, among tickets with meaningful ratings, customer satisfaction is generally positive and centered close to **4 out of 5**. Satisfaction levels also remain relatively similar across priority classes, with only minor differences between groups. This suggests that support performance is not generating strong dissatisfaction in any single ticket category.

At the account level, support burden appears **distributed rather than concentrated**, although some accounts receive more attention than others. The average account with support activity has around **4 tickets**, while the heaviest account in the dataset has **11 tickets**. Escalation burden is also generally low at the account level, with most accounts having no escalations at all and only a very small number recording multiple escalations. This indicates that although some customers interact with support more frequently than others, the dataset does not point to a widespread pattern of chronically high-friction accounts.

The top accounts by ticket volume and escalation count provide useful descriptive signals, but they do not yet indicate a severe operational concentration problem. Most of the accounts with the highest ticket volume still show moderate average response and resolution times, and several of them maintain satisfaction averages near or above the broader account-level mean. This suggests that high ticket activity alone should not automatically be interpreted as support failure. Instead, later churn-focused analysis should examine whether ticket volume becomes more meaningful when combined with escalation frequency, lower satisfaction, or other behavioral signals.

Overall, the support layer reflects a **stable operational environment** with broad account coverage, a balanced priority mix, limited escalation pressure, and generally positive rated satisfaction outcomes. The main analytical takeaway is that support burden exists across the customer base, but it does not appear overwhelmingly concentrated or severely dysfunctional. This makes support behavior a useful but likely **context-dependent** churn signal: it may matter most when combined with other factors such as product friction, weak engagement, or negative commercial signals, rather than as a standalone explanation.

## 11.5 Churn Event Analysis

This step explores the churn-events Understanding its structure is essential before moving into deeper churn-driver analysis or combining churn signals with usage, support, and revenue behavior.This step explores the churn-events layer of the RavenStack dataset in order to understand how churn is represented across the business and which patterns appear most frequently in the observed churn outcomes.


The analysis in this step focuses on the following questions:

- How large is the churn-events layer of the dataset?
- What are the most common churn reasons?
- How common are reactivation-related churn records?
- How often do churn events occur after an upgrade or downgrade?
- What does refund behavior look like across churn events?
- Are churn events broadly distributed across accounts, or concentrated in a smaller subset?



In [61]:
# =========================================================
# 11.5 Churn Event Analysis
# Corrected Full Version
# =========================================================

import pandas as pd
import numpy as np

# Use the final cleaned churn table
churn_df = churn_events_clean.copy()

print("=== 11.5 Churn Event Analysis ===")

# ---------------------------------------------------------
# A) High-level churn summary
# ---------------------------------------------------------
print("\n--- High-Level Churn Summary ---")

# Create numeric-friendly columns for aggregation
churn_df["preceding_upgrade_flag_num"] = pd.to_numeric(
    churn_df["preceding_upgrade_flag"], errors="coerce"
).fillna(0)

churn_df["preceding_downgrade_flag_num"] = pd.to_numeric(
    churn_df["preceding_downgrade_flag"], errors="coerce"
).fillna(0)

churn_df["is_reactivation_num"] = pd.to_numeric(
    churn_df["is_reactivation"], errors="coerce"
).fillna(0)

churn_df["refund_amount_usd_num"] = pd.to_numeric(
    churn_df["refund_amount_usd"], errors="coerce"
)

churn_summary = {
    "Total Churn Events": len(churn_df),
    "Unique Churn Event IDs": churn_df["churn_event_id"].nunique(),
    "Unique Accounts with Churn Events": churn_df["account_id"].nunique(),
    "Total Refund Amount (USD)": churn_df["refund_amount_usd_num"].sum(),
    "Average Refund Amount (USD)": churn_df["refund_amount_usd_num"].mean(),
    "Churn Events with Refund > 0": int((churn_df["refund_amount_usd_num"] > 0).sum()),
    "Reactivation-Related Churn Events": int(churn_df["is_reactivation_num"].sum()),
    "Churn Events After Upgrade": int(churn_df["preceding_upgrade_flag_num"].sum()),
    "Churn Events After Downgrade": int(churn_df["preceding_downgrade_flag_num"].sum()),
}

for metric, value in churn_summary.items():
    print(f"{metric}: {value}")

# ---------------------------------------------------------
# B) Churn reason distribution
# ---------------------------------------------------------
print("\n--- Churn Reason Distribution ---")

print("\nReason Code Distribution:")
display(churn_df["reason_code"].value_counts(dropna=False))

print("\nReason Code Distribution (%):")
display((churn_df["reason_code"].value_counts(normalize=True, dropna=False) * 100).round(2))

# ---------------------------------------------------------
# C) Reactivation and preceding commercial flags
# ---------------------------------------------------------
print("\n--- Reactivation and Preceding Commercial Signals ---")

print("\nReactivation Flag Distribution:")
display(churn_df["is_reactivation"].value_counts(dropna=False))

print("\nPreceding Upgrade Flag Distribution:")
display(churn_df["preceding_upgrade_flag"].value_counts(dropna=False))

print("\nPreceding Downgrade Flag Distribution:")
display(churn_df["preceding_downgrade_flag"].value_counts(dropna=False))

print("\nRates Summary (%):")
print("Reactivation Rate (%):", round(churn_df["is_reactivation_num"].mean() * 100, 2))
print("Preceding Upgrade Rate (%):", round(churn_df["preceding_upgrade_flag_num"].mean() * 100, 2))
print("Preceding Downgrade Rate (%):", round(churn_df["preceding_downgrade_flag_num"].mean() * 100, 2))

# ---------------------------------------------------------
# D) Refund analysis
# ---------------------------------------------------------
print("\n--- Refund Analysis ---")

print("\nRefund Amount Summary:")
display(churn_df["refund_amount_usd_num"].describe())

print("\nRefund > 0 Distribution:")
display((churn_df["refund_amount_usd_num"] > 0).value_counts(dropna=False))

refund_by_reason = (
    churn_df.groupby("reason_code")
    .agg(
        churn_event_count=("churn_event_id", "count"),
        total_refund_amount_usd=("refund_amount_usd_num", "sum"),
        avg_refund_amount_usd=("refund_amount_usd_num", "mean"),
        refund_event_count=("refund_amount_usd_num", lambda x: (x > 0).sum())
    )
    .sort_values("total_refund_amount_usd", ascending=False)
)

print("\nRefund Analysis by Reason Code:")
display(refund_by_reason)

# ---------------------------------------------------------
# E) Churn event concentration by account
# ---------------------------------------------------------
print("\n--- Churn Event Concentration by Account ---")

churn_by_account = (
    churn_df.groupby("account_id")
    .agg(
        churn_event_count=("churn_event_id", "count"),
        total_refund_amount_usd=("refund_amount_usd_num", "sum"),
        any_reactivation=("is_reactivation_num", "max"),
        any_preceding_upgrade=("preceding_upgrade_flag_num", "max"),
        any_preceding_downgrade=("preceding_downgrade_flag_num", "max")
    )
    .reset_index()
)

print("\nChurn Events by Account Summary:")
display(churn_by_account.describe())

print("\nTop 10 Accounts by Number of Churn Events:")
display(
    churn_by_account.sort_values("churn_event_count", ascending=False).head(10)
)

# ---------------------------------------------------------
# F) Refund concentration check
# ---------------------------------------------------------
print("\n--- Refund Concentration Check ---")

total_refund = churn_df["refund_amount_usd_num"].sum()

if total_refund > 0:
    top_10_refund_share = (
        churn_df.sort_values("refund_amount_usd_num", ascending=False)["refund_amount_usd_num"].head(10).sum()
        / total_refund
    ) * 100

    print(f"Top 10 churn events share of total refund amount: {top_10_refund_share:.2f}%")
else:
    print("No positive refund amounts detected.")


=== 11.5 Churn Event Analysis ===

--- High-Level Churn Summary ---
Total Churn Events: 600
Unique Churn Event IDs: 600
Unique Accounts with Churn Events: 352
Total Refund Amount (USD): 8652.25
Average Refund Amount (USD): 14.420416666666666
Churn Events with Refund > 0: 142
Reactivation-Related Churn Events: 61
Churn Events After Upgrade: 123
Churn Events After Downgrade: 53

--- Churn Reason Distribution ---

Reason Code Distribution:


reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64


Reason Code Distribution (%):


reason_code
features      19.00
support       17.33
budget        17.33
unknown       15.83
competitor    15.33
pricing       15.17
Name: proportion, dtype: float64


--- Reactivation and Preceding Commercial Signals ---

Reactivation Flag Distribution:


...


--- Churn Event Concentration by Account ---

Churn Events by Account Summary:


,churn_event_count,total_refund_amount_usd,any_reactivation,any_preceding_upgrade,any_preceding_downgrade
count,352.000000,352.000000,352.000000,352.000000,352.000000
mean,1.704545,24.580256,0.156250,0.306818,0.139205
std,0.846095,53.309174,0.363609,0.461830,0.346653
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000
75%,2.000000,19.797500,0.000000,1.000000,0.000000
max,5.000000,444.100000,1.000000,1.000000,1.000000



Top 10 Accounts by Number of Churn Events:


,account_id,churn_event_count,total_refund_amount_usd,any_reactivation,any_preceding_upgrade,any_preceding_downgrade
40,A-180abf,5,0.00,0,0,1
20,A-0a62f5,5,156.45,0,1,1
202,A-9208f0,4,66.37,1,1,0
120,A-533452,4,147.55,1,0,1
22,A-0baac2,4,99.38,0,1,0
82,A-37f969,4,146.86,0,1,0
236,A-ae052e,4,0.00,0,0,0
142,A-68f37c,4,0.00,1,1,0
24,A-0cc442,4,59.52,1,1,0
347,A-fd7ad3,4,70.03,1,1,0



--- Refund Concentration Check ---
Top 10 churn events share of total refund amount: 25.71%


### 11.5 Interpretation — Churn Event Analysis

The churn-events analysis shows that customer attrition in RavenStack is **meaningful, repeated for some accounts, and commercially layered rather than one-dimensional**. The dataset contains **600 churn events linked to 352 unique accounts**, which immediately indicates that churn is not limited to one event per customer. In practical terms, this means that some accounts appear multiple times in the churn layer, which is consistent with a business environment where accounts may churn, reactivate, and later churn again, or where churn-related events may occur more than once across the customer lifecycle.

The distribution of churn reasons is relatively balanced, but a few themes stand out more clearly than others. **Feature-related churn** is the most common reason (**114 events; 19.0%**), followed by **support** and **budget** (each **104 events; 17.33%**). The remaining churn reasons — **unknown, competitor, and pricing** — are also substantial and closely grouped. This suggests that churn in RavenStack is **not dominated by a single cause**, but instead appears to emerge from a mix of product, service, and commercial pressures. However, the fact that **feature limitations** appear as the top reason is especially important, because it aligns with the broader SaaS context of the project and suggests that product value and capability gaps may be central to retention risk.

The commercial-history flags provide an additional layer of interpretation. Around **20.5% of churn events occurred after an upgrade**, while **8.83% occurred after a downgrade**, and **10.17% are marked as reactivation-related churn events**. These results imply that churn is not always a simple end-state event. In a meaningful subset of cases, customers appear to go through a commercial transition before churning, particularly through plan upgrades. This may reflect situations where customers attempted to expand their usage or value realization, but ultimately still exited. The lower downgrade rate is also notable because it suggests that downgrade is not the dominant pre-churn path in this dataset, even though it remains relevant as a warning signal.

Refund behavior is present, but not universal. Only **142 churn events** involve a positive refund, while the majority of churn events do not result in any refund payment. The refund distribution is also highly skewed: the median refund is **0**, and the top 10 churn events account for **25.71% of total refund value**. This means that financial loss linked to refunds is **concentrated in a relatively small number of churn cases**, rather than being spread evenly across all attrition events. From a business perspective, this suggests that refund exposure is important, but it should be evaluated as a concentrated financial risk rather than as a general characteristic of all churn.

When churn reasons are connected to refunds, the picture becomes more nuanced. **Feature-related churn** generates the highest total refund amount, followed by **unknown** and **pricing**, while **support** produces the highest count of refund-bearing churn events. This means that some churn categories are more expensive in aggregate, while others involve refunds more frequently. Analytically, this distinction matters because it suggests that churn reasons should not be interpreted only by volume. A lower-volume churn category may still be commercially important if it tends to generate larger refunds or more frequent compensation-related exits.

Finally, churn concentration by account confirms that attrition is **spread across the customer base, but not entirely uniform**. The average number of churn events per affected account is **1.70**, and the maximum reaches **5 events** for a single account. Most churned accounts appear only once, but a smaller subset experiences multiple churn events. This strengthens the earlier interpretation that churn in RavenStack includes some **repeat-exit behavior**, which makes the churn table more than just a simple list of lost customers. Some accounts appear to move through a more complex lifecycle involving repeated instability, reactivation, or staged exit behavior.

Overall, this step shows that churn in RavenStack is **multi-causal, commercially meaningful, and behaviorally layered**. The most important descriptive takeaway is that churn should not be treated as a single homogeneous outcome. Instead, it reflects a combination of product-value issues, service-related concerns, commercial dynamics, and in some cases repeated instability across the same customer account. This makes the churn-events table a strong foundation for the next analytical stage, especially when it is later connected with usage patterns, support burden, and revenue structure.

## 11.6 Cross-Domain Analytical Patterns

This step connects the main relational tables in order to examine how customer profile, subscription structure, product usage, support behavior, and churn outcomes interact with one another.

The purpose of this analysis is to move beyond isolated table-level exploration and identify broader business patterns across the customer lifecycle. In a subscription-based SaaS context, churn is often shaped by a combination of factors rather than by a single signal. For that reason, this step evaluates how commercial structure, engagement behavior, support experience, and churn-related outcomes relate across the dataset.

The analysis in this step focuses on the following questions:

- How do account characteristics relate to subscription structure?
- How do revenue and trial behavior differ across churned and non-churned customers?
- Do customers with broader product adoption show different churn behavior?
- Do support burden and support quality vary across churn groups?
- Are churn-related outcomes associated with stronger product friction, weaker engagement, or lower commercial value?

This step is important because it creates the first multi-table business view of the dataset and prepares the project for churn-focused interpretation, segmentation, and later feature engineering.


In [62]:
# =========================================================
# 11.6 Cross-Domain Analytical Patterns
# Fully Corrected Version
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 0) Load working copies
# ---------------------------------------------------------
accounts_df = accounts_clean.copy()
subscriptions_df = subscriptions_clean.copy()
feature_usage_df = feature_usage_clean.copy()
support_df = support_tickets_clean.copy()
churn_df = churn_events_clean.copy()

print("=== 11.6 Cross-Domain Analytical Patterns ===")

# ---------------------------------------------------------
# 1) Helper functions
# ---------------------------------------------------------
def to_binary_num(series):
    """
    Robust conversion for flags that may appear as:
    0/1, True/False, '0'/'1', 'true'/'false', yes/no
    """
    s = series.astype("string").str.strip().str.lower()
    mapped = s.map({
        "1": 1, "0": 0,
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "y": 1, "n": 0
    })
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.fillna(numeric).fillna(0).astype(float)

# ---------------------------------------------------------
# 2) Normalize critical fields
# ---------------------------------------------------------

# Accounts
accounts_df["churn_flag_num"] = to_binary_num(accounts_df["churn_flag"])
accounts_df["is_trial_num"] = to_binary_num(accounts_df["is_trial"])

# Subscriptions
subscriptions_df["is_trial_num"] = to_binary_num(subscriptions_df["is_trial"])
subscriptions_df["churn_flag_num"] = to_binary_num(subscriptions_df["churn_flag"])
subscriptions_df["auto_renew_flag_num"] = to_binary_num(subscriptions_df["auto_renew_flag"])

subscriptions_df["seats_num"] = pd.to_numeric(subscriptions_df["seats"], errors="coerce")
subscriptions_df["mrr_amount_num"] = pd.to_numeric(subscriptions_df["mrr_amount"], errors="coerce")
subscriptions_df["arr_amount_num"] = pd.to_numeric(subscriptions_df["arr_amount"], errors="coerce")

# Feature usage
feature_usage_df["is_beta_feature_num"] = to_binary_num(feature_usage_df["is_beta_feature"])
feature_usage_df["usage_count_num"] = pd.to_numeric(feature_usage_df["usage_count"], errors="coerce")
feature_usage_df["usage_duration_secs_num"] = pd.to_numeric(feature_usage_df["usage_duration_secs"], errors="coerce")
feature_usage_df["error_count_num"] = pd.to_numeric(feature_usage_df["error_count"], errors="coerce")

# Support
support_df["escalation_flag_num"] = to_binary_num(support_df["escalation_flag"])
support_df["resolution_time_hours_num"] = pd.to_numeric(support_df["resolution_time_hours"], errors="coerce")
support_df["first_response_time_minutes_num"] = pd.to_numeric(support_df["first_response_time_minutes"], errors="coerce")
support_df["satisfaction_score_num"] = pd.to_numeric(support_df["satisfaction_score"], errors="coerce")

# If analytical satisfaction column does not exist, create it
if "satisfaction_score_analytical" not in support_df.columns:
    support_df["satisfaction_score_analytical"] = support_df["satisfaction_score_num"].mask(
        support_df["satisfaction_score_num"] == 0, pd.NA
    )

support_df["satisfaction_score_analytical_num"] = pd.to_numeric(
    support_df["satisfaction_score_analytical"], errors="coerce"
)

# Churn
churn_df["preceding_upgrade_flag_num"] = to_binary_num(churn_df["preceding_upgrade_flag"])
churn_df["preceding_downgrade_flag_num"] = to_binary_num(churn_df["preceding_downgrade_flag"])
churn_df["is_reactivation_num"] = to_binary_num(churn_df["is_reactivation"])
churn_df["refund_amount_usd_num"] = pd.to_numeric(churn_df["refund_amount_usd"], errors="coerce")

# ---------------------------------------------------------
# 3) Build account-level analytical table
# ---------------------------------------------------------
print("\n--- Building Account-Level Analytical Table ---")

# A) Subscription summary by account
subscription_by_account = (
    subscriptions_df.groupby("account_id")
    .agg(
        subscription_count=("subscription_id", "count"),
        trial_subscription_count=("is_trial_num", "sum"),
        churn_subscription_count=("churn_flag_num", "sum"),
        total_seats=("seats_num", "sum"),
        avg_subscription_seats=("seats_num", "mean"),
        total_mrr=("mrr_amount_num", "sum"),
        total_arr=("arr_amount_num", "sum"),
        avg_mrr=("mrr_amount_num", "mean"),
        avg_arr=("arr_amount_num", "mean"),
        trial_rate=("is_trial_num", "mean"),
        churn_rate=("churn_flag_num", "mean"),
        auto_renew_rate=("auto_renew_flag_num", "mean"),
    )
    .reset_index()
)

# B) Usage summary by account
usage_with_account = feature_usage_df.merge(
    subscriptions_df[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

usage_by_account = (
    usage_with_account.groupby("account_id")
    .agg(
        usage_event_count=("usage_id", "count"),
        total_usage_count=("usage_count_num", "sum"),
        total_usage_duration_secs=("usage_duration_secs_num", "sum"),
        total_error_count=("error_count_num", "sum"),
        unique_features_used=("feature_name", "nunique"),
        beta_feature_usage_rate=("is_beta_feature_num", "mean"),
        avg_usage_count=("usage_count_num", "mean"),
        avg_usage_duration_secs=("usage_duration_secs_num", "mean"),
        avg_error_count=("error_count_num", "mean"),
    )
    .reset_index()
)

# C) Support summary by account
support_by_account = (
    support_df.groupby("account_id")
    .agg(
        ticket_count=("ticket_id", "count"),
        escalated_ticket_count=("escalation_flag_num", "sum"),
        avg_resolution_time_hours=("resolution_time_hours_num", "mean"),
        avg_first_response_time_minutes=("first_response_time_minutes_num", "mean"),
        avg_satisfaction_score=("satisfaction_score_analytical_num", "mean"),
    )
    .reset_index()
)

# D) Churn summary by account
churn_by_account = (
    churn_df.groupby("account_id")
    .agg(
        churn_event_count=("churn_event_id", "count"),
        total_refund_amount_usd=("refund_amount_usd_num", "sum"),
        reactivation_related_churn_count=("is_reactivation_num", "sum"),
        preceding_upgrade_churn_count=("preceding_upgrade_flag_num", "sum"),
        preceding_downgrade_churn_count=("preceding_downgrade_flag_num", "sum"),
    )
    .reset_index()
)

# E) Merge everything
account_eda = (
    accounts_df
    .merge(subscription_by_account, on="account_id", how="left")
    .merge(usage_by_account, on="account_id", how="left")
    .merge(support_by_account, on="account_id", how="left")
    .merge(churn_by_account, on="account_id", how="left")
)

# Fill numeric missing values
numeric_cols = account_eda.select_dtypes(include=["number"]).columns
account_eda[numeric_cols] = account_eda[numeric_cols].fillna(0)

print("\nAccount-Level Analytical Table Shape:")
print(account_eda.shape)

print("\nAccount-Level Analytical Table Preview:")
display(account_eda.head())

# ---------------------------------------------------------
# 4) Cross-domain summary by account churn flag
# ---------------------------------------------------------
print("\n--- Cross-Domain Comparison by Account Churn Flag ---")

cross_domain_by_churn = (
    account_eda.groupby("churn_flag_num")
    .agg(
        account_count=("account_id", "count"),
        avg_subscription_count=("subscription_count", "mean"),
        avg_total_mrr=("total_mrr", "mean"),
        avg_total_arr=("total_arr", "mean"),
        avg_total_seats=("total_seats", "mean"),
        avg_usage_event_count=("usage_event_count", "mean"),
        avg_total_usage_count=("total_usage_count", "mean"),
        avg_unique_features_used=("unique_features_used", "mean"),
        avg_total_error_count=("total_error_count", "mean"),
        avg_ticket_count=("ticket_count", "mean"),
        avg_escalated_ticket_count=("escalated_ticket_count", "mean"),
        avg_resolution_time_hours=("avg_resolution_time_hours", "mean"),
        avg_first_response_time_minutes=("avg_first_response_time_minutes", "mean"),
        avg_satisfaction_score=("avg_satisfaction_score", "mean"),
        avg_reactivation_related_churn=("reactivation_related_churn_count", "mean"),
        avg_preceding_upgrade_churn=("preceding_upgrade_churn_count", "mean"),
        avg_preceding_downgrade_churn=("preceding_downgrade_churn_count", "mean"),
    )
    .reset_index()
)

display(cross_domain_by_churn)

# ---------------------------------------------------------
# 5) Commercial vs behavioral segmentation
# ---------------------------------------------------------
print("\n--- Commercial vs Behavioral Segmentation ---")

account_eda["usage_breadth_segment"] = pd.qcut(
    account_eda["unique_features_used"].rank(method="first"),
    q=3,
    labels=["Low Breadth", "Medium Breadth", "High Breadth"]
)

account_eda["usage_intensity_segment"] = pd.qcut(
    account_eda["total_usage_count"].rank(method="first"),
    q=3,
    labels=["Low Intensity", "Medium Intensity", "High Intensity"]
)

account_eda["support_burden_segment"] = pd.qcut(
    account_eda["ticket_count"].rank(method="first"),
    q=3,
    labels=["Low Support", "Medium Support", "High Support"]
)

account_eda["revenue_segment"] = pd.qcut(
    account_eda["total_mrr"].rank(method="first"),
    q=3,
    labels=["Low Revenue", "Medium Revenue", "High Revenue"]
)

print("\nChurn Rate by Usage Breadth Segment:")
display(
    account_eda.groupby("usage_breadth_segment")["churn_flag_num"]
    .agg(["mean", "sum", "count"])
)

print("\nChurn Rate by Usage Intensity Segment:")
display(
    account_eda.groupby("usage_intensity_segment")["churn_flag_num"]
    .agg(["mean", "sum", "count"])
)

print("\nChurn Rate by Support Burden Segment:")
display(
    account_eda.groupby("support_burden_segment")["churn_flag_num"]
    .agg(["mean", "sum", "count"])
)

print("\nChurn Rate by Revenue Segment:")
display(
    account_eda.groupby("revenue_segment")["churn_flag_num"]
    .agg(["mean", "sum", "count"])
)

# ---------------------------------------------------------
# 6) Plan tier cross-domain patterns
# ---------------------------------------------------------
print("\n--- Plan Tier Cross-Domain Patterns ---")

plan_cross_domain = (
    account_eda.groupby("plan_tier")
    .agg(
        account_count=("account_id", "count"),
        avg_total_mrr=("total_mrr", "mean"),
        avg_unique_features_used=("unique_features_used", "mean"),
        avg_total_usage_count=("total_usage_count", "mean"),
        avg_total_error_count=("total_error_count", "mean"),
        avg_ticket_count=("ticket_count", "mean"),
        avg_satisfaction_score=("avg_satisfaction_score", "mean"),
        avg_churn_rate=("churn_flag_num", "mean"),
    )
    .sort_values("avg_total_mrr", ascending=False)
)

display(plan_cross_domain)

# ---------------------------------------------------------
# 7) Correlation snapshot
# ---------------------------------------------------------
print("\n--- Correlation Snapshot of Key Analytical Variables ---")

key_numeric_vars = [
    "total_mrr",
    "total_arr",
    "subscription_count",
    "unique_features_used",
    "total_usage_count",
    "total_usage_duration_secs",
    "total_error_count",
    "ticket_count",
    "escalated_ticket_count",
    "avg_resolution_time_hours",
    "avg_first_response_time_minutes",
    "avg_satisfaction_score",
    "churn_event_count",
    "total_refund_amount_usd"
]

existing_numeric_vars = [col for col in key_numeric_vars if col in account_eda.columns]

corr_matrix = account_eda[existing_numeric_vars].corr(numeric_only=True)

print("\nCorrelation Matrix:")
display(corr_matrix)

# ---------------------------------------------------------
# 8) Top accounts by combined exploratory risk score
# ---------------------------------------------------------
print("\n--- Top Accounts by Combined Risk Signals ---")

risk_signal_accounts = account_eda.copy()

# medians for comparison
error_median = risk_signal_accounts["total_error_count"].median()
ticket_median = risk_signal_accounts["ticket_count"].median()
satisfaction_median = risk_signal_accounts["avg_satisfaction_score"].median()
breadth_median = risk_signal_accounts["unique_features_used"].median()

risk_signal_accounts["risk_score"] = (
    risk_signal_accounts["churn_flag_num"] * 2
    + (risk_signal_accounts["total_error_count"] > error_median).astype(int)
    + (risk_signal_accounts["ticket_count"] > ticket_median).astype(int)
    + (risk_signal_accounts["avg_satisfaction_score"] < satisfaction_median).astype(int)
    + (risk_signal_accounts["unique_features_used"] < breadth_median).astype(int)
)

print("\nTop 15 Accounts by Exploratory Risk Score:")
display(
    risk_signal_accounts.sort_values("risk_score", ascending=False)
    .loc[:, [
        "account_id",
        "account_name",
        "risk_score",
        "churn_flag_num",
        "total_mrr",
        "unique_features_used",
        "total_error_count",
        "ticket_count",
        "avg_satisfaction_score",
        "churn_event_count"
    ]]
    .head(15)
)


=== 11.6 Cross-Domain Analytical Patterns ===

--- Building Account-Level Analytical Table ---

Account-Level Analytical Table Shape:
(500, 43)

Account-Level Analytical Table Preview:


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,...,ticket_count,escalated_ticket_count,avg_resolution_time_hours,avg_first_response_time_minutes,avg_satisfaction_score,churn_event_count,total_refund_amount_usd,reactivation_related_churn_count,preceding_upgrade_churn_count,preceding_downgrade_churn_count
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,0,0,...,2.0,0.0,23.000000,91.000000,3.000000,2.0,0.00,0.0,1.0,0.0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,0,1,...,3.0,0.0,38.000000,73.333333,4.000000,0.0,0.00,0.0,0.0,0.0
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,0,0,...,3.0,0.0,43.666667,63.666667,4.666667,2.0,62.66,0.0,0.0,1.0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,1,0,...,2.0,0.0,29.000000,174.000000,0.000000,1.0,0.00,0.0,1.0,0.0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,0,1,...,7.0,1.0,42.285714,107.857143,3.800000,1.0,0.00,0.0,1.0,0.0



--- Cross-Domain Comparison by Account Churn Flag ---


,churn_flag_num,account_count,avg_subscription_count,avg_total_mrr,avg_total_arr,avg_total_seats,avg_usage_event_count,avg_total_usage_count,avg_unique_features_used,avg_total_error_count,avg_ticket_count,avg_escalated_ticket_count,avg_resolution_time_hours,avg_first_response_time_minutes,avg_satisfaction_score,avg_reactivation_related_churn,avg_preceding_upgrade_churn,avg_preceding_downgrade_churn
0,0.0,390,9.917949,23034.620513,276415.446154,299.687179,49.423077,495.130769,27.412821,28.230769,4.020513,0.182051,35.889532,88.203733,3.689835,0.123077,0.248718,0.097436
1,1.0,110,10.290909,21411.318182,256935.818182,294.381818,52.045455,522.036364,28.336364,28.154545,3.927273,0.218182,34.840292,83.382821,3.711494,0.118182,0.236364,0.136364



--- Commercial vs Behavioral Segmentation ---

Churn Rate by Usage Breadth Segment:


...

,account_count,avg_total_mrr,avg_unique_features_used,avg_total_usage_count,avg_total_error_count,avg_ticket_count,avg_satisfaction_score,avg_churn_rate
plan_tier,,,,,,,,
Basic,168,24094.488095,28.238095,528.583333,29.482143,3.750000,3.650836,0.220238
Pro,178,22296.162921,27.415730,484.528090,26.943820,4.106742,3.674157,0.219101
Enterprise,154,21572.441558,27.168831,490.110390,28.298701,4.149351,3.765971,0.220779



--- Correlation Snapshot of Key Analytical Variables ---

Correlation Matrix:


,total_mrr,total_arr,subscription_count,unique_features_used,total_usage_count,total_usage_duration_secs,total_error_count,ticket_count,escalated_ticket_count,avg_resolution_time_hours,avg_first_response_time_minutes,avg_satisfaction_score,churn_event_count,total_refund_amount_usd
total_mrr,1.000000,1.000000,0.403590,0.364156,0.376912,0.369326,0.339129,-0.027154,-0.003376,0.079257,0.039863,0.092446,-0.045620,-0.032535
total_arr,1.000000,1.000000,0.403590,0.364156,0.376912,0.369326,0.339129,-0.027154,-0.003376,0.079257,0.039863,0.092446,-0.045620,-0.032535
subscription_count,0.403590,0.403590,1.000000,0.831971,0.903213,0.878153,0.724668,-0.033010,0.016143,0.069321,0.021209,0.100627,-0.025000,-0.053994
unique_features_used,0.364156,0.364156,0.831971,1.000000,0.901147,0.880569,0.734738,-0.024509,0.041979,0.040197,0.022181,0.085975,0.018284,-0.021552
total_usage_count,0.376912,0.376912,0.903213,0.901147,1.000000,0.973483,0.791414,-0.061127,0.027571,0.050769,0.024814,0.080495,-0.014586,-0.037575
total_usage_duration_secs,0.369326,0.369326,0.878153,0.880569,0.973483,1.000000,0.775149,-0.036928,0.029322,0.047925,0.024095,0.078791,-0.011180,-0.046425
total_error_count,0.339129,0.339129,0.724668,0.734738,0.791414,0.775149,1.000000,-0.010586,0.053426,0.044701,-0.002074,0.116769,0.015340,-0.040927
ticket_count,-0.027154,-0.027154,-0.033010,-0.024509,-0.061127,-0.036928,-0.010586,1.000000,0.174997,0.034049,0.094778,0.314154,-0.011052,-0.021463
escalated_ticket_count,-0.003376,-0.003376,0.016143,0.041979,0.027571,0.029322,0.053426,0.174997,1.000000,0.019282,0.045442,0.063958,0.063788,0.061067
avg_resolution_time_hours,0.079257,0.079257,0.069321,0.040197,0.050769,0.047925,0.044701,0.034049,0.019282,1.000000,0.184656,0.057347,-0.007762,-0.042939



--- Top Accounts by Combined Risk Signals ---

Top 15 Accounts by Exploratory Risk Score:


,account_id,account_name,risk_score,churn_flag_num,total_mrr,unique_features_used,total_error_count,ticket_count,avg_satisfaction_score,churn_event_count
256,A-3ce5b8,Company_256,5.0,1.0,16650,29,29,10.0,3.714286,1.0
126,A-1ac5e0,Company_126,5.0,1.0,18249,31,31,7.0,3.800000,0.0
299,A-7f6b86,Company_299,5.0,1.0,19402,27,14,7.0,3.666667,1.0
135,A-117171,Company_135,5.0,1.0,12015,15,11,6.0,3.666667,3.0
449,A-4c56c9,Company_449,5.0,1.0,30471,34,38,5.0,3.750000,1.0
400,A-0f6450,Company_400,5.0,1.0,23800,31,39,8.0,3.833333,2.0
436,A-34ea2f,Company_436,5.0,1.0,14583,29,30,8.0,3.800000,2.0
222,A-8ed5dd,Company_222,5.0,1.0,11956,29,45,6.0,3.750000,2.0
344,A-b30291,Company_344,5.0,1.0,5375,26,31,1.0,0.000000,1.0
272,A-e4bf56,Company_272,5.0,1.0,18926,25,10,8.0,3.666667,0.0


### 11.6 Interpretation — Cross-Domain Analytical Patterns

The cross-domain analysis provides the first fully integrated view of the RavenStack customer lifecycle by combining account profile, subscription structure, product usage, support behavior, and churn-related outcomes into a single account-level analytical table. The resulting table contains **500 accounts and 43 analytical fields**, which means the project now moves from isolated table-level exploration into a more complete customer-level perspective. This integration is important because it allows churn to be interpreted in context rather than through one variable at a time.

#### 1. Overall churned vs non-churned account comparison

The comparison between churned and non-churned accounts does **not** reveal a simple single-factor explanation for churn. In fact, the churned group does **not** consistently look weaker across the major dimensions of the dataset. Churned accounts have slightly **lower average total MRR and ARR**, but they also show slightly **higher usage volume**, slightly **higher feature breadth**, and nearly identical total error levels. At the same time, churned accounts do **not** appear to have materially worse support conditions: their average ticket count is slightly lower, their average response and resolution times are slightly lower, and their average satisfaction score is actually marginally higher than that of non-churned accounts.

The main implication is that churn in this dataset does **not** appear to be driven by one obvious descriptive weakness such as low revenue, low engagement, severe product friction, or clearly poor support experience. Instead, the churn pattern appears more complex and likely shaped by combinations of factors rather than by a single dominant signal.

---

#### 2. Usage breadth and usage intensity do not behave as expected

One of the most important findings in this step is that the churn rates by **usage breadth** and **usage intensity** move in the opposite direction of the simple expectation. If broader adoption and stronger engagement always protected against churn, the churn rate would be expected to fall as breadth and intensity increase. Instead, the results show a **gradual increase** in churn rate from low to high breadth and from low to high intensity.

This suggests that, in RavenStack, heavier engagement is **not automatically equivalent to lower churn risk**. A possible interpretation is that some highly engaged accounts are also more demanding, more commercially sensitive, or more exposed to feature-related expectations. In other words, stronger product engagement may reflect deeper usage, but it may also create higher standards and stronger reasons to churn if the product no longer meets business needs. This is a very important result because it prevents the project from assuming that “more usage always means lower churn.”

---

#### 3. Support burden is not a strong standalone churn driver

The segmentation by **support burden** shows that the **highest churn rate appears in the low-support segment**, not in the high-support segment. This means that accounts with more support tickets are not automatically more likely to churn. In fact, the result suggests the opposite pattern descriptively: lower support interaction is associated with higher churn than heavier support interaction.

This is a meaningful finding because it supports the earlier interpretation from the support analysis: support does not currently appear to be a strong standalone churn signal in this dataset. A reasonable business reading is that moderate support interaction may reflect active, engaged customers rather than dissatisfied ones. Put differently, customers who interact with support are not necessarily at greater risk; some of them may simply be more operationally active. Therefore, support burden should be treated as a **contextual signal**, not as a direct churn driver on its own.

---

#### 4. Revenue segmentation does not show a strong monotonic churn pattern

The revenue segmentation also does not reveal a strong linear relationship between customer value and churn. The **medium-revenue segment** shows the highest churn rate, while the low- and high-revenue segments are relatively close to one another. This indicates that churn is not concentrated exclusively among the smallest or least valuable customers, nor is it concentrated only among the highest-value accounts.

This is analytically important because it suggests that churn in RavenStack is not merely a low-value customer problem. Revenue level matters descriptively, but the current results do not support the idea that one revenue band clearly dominates churn behavior. As a result, later churn-focused analysis should avoid overly simplistic assumptions such as “low-revenue accounts churn more.”

---

#### 5. Plan tier is not a major differentiator of churn rate

The plan-tier comparison shows that **Basic, Pro, and Enterprise accounts all have very similar average churn rates**, despite meaningful differences in average MRR, usage, and support characteristics. This indicates that plan tier itself is **not a strong standalone separator** of churn behavior at the account level.

From a business perspective, this means that higher-value commercial packaging does not automatically protect against churn, just as lower-value tiers are not automatically more fragile. Plan tier still matters commercially, especially for revenue interpretation, but it is not enough on its own to explain which accounts leave.

---

#### 6. Correlation results confirm weak standalone relationships with churn

The correlation matrix reinforces the same message: most of the cross-domain variables show **weak direct linear relationships** with churn-event frequency and refund-related outcomes. Stronger correlations appear mostly among structurally related variables, such as:

- subscription count with total usage and feature breadth  
- total usage with usage duration  
- subscription count with error count  
- churn-event count with refund amount  

These are analytically useful relationships, but they are mostly structural rather than causal. The important takeaway is that the churn-related variables do **not** exhibit strong simple correlations with the major commercial, usage, or support variables. This again supports the conclusion that churn is likely to be **multi-factor and interaction-driven**, rather than explained well by any one descriptive metric in isolation.

---

#### 7. The risk-score view is useful as an exploratory prioritization tool

The exploratory risk score highlights a subset of accounts that combine several concerning signals at the same time, such as churn presence, elevated error burden, above-median support burden, lower satisfaction, and weaker feature breadth. This does not mean the score is a final predictive model, but it is useful as a descriptive prioritization layer.

Its value lies in showing that risk may emerge most clearly when signals are combined. An account with average revenue alone may not look risky, and an account with high support volume alone may also not appear risky. However, when churn status, product friction, support burden, and weaker experience indicators are viewed together, a more meaningful account-risk profile begins to emerge.

---

### Final Cross-Domain Conclusion

The most important conclusion from this step is that **churn in RavenStack does not behave like a simple one-variable problem**. The cross-domain patterns do not support a strong standalone relationship between churn and any single factor such as revenue, support burden, plan tier, or low usage. Instead, the results suggest that churn should be interpreted as a **multi-dimensional outcome** shaped by the interaction between commercial structure, engagement behavior, support experience, and churn-history characteristics.

For the next analytical stages, this means that the project should avoid relying on isolated descriptive indicators and instead focus on **combined behavioral-commercial profiles**. In practice, the most valuable next step will be to treat churn as the result of interacting signals rather than looking for one dominant descriptive driver.

In [63]:
 account_eda.to_csv("account_eda_final.csv", index=False)
 print("account_eda_final.csv exported successfully.")

account_eda_final.csv exported successfully.


In [64]:

feature_df.to_csv("feature_engineering_final.csv", index=False)
print("feature_engineering_final.csv exported successfully.")


feature_engineering_final.csv exported successfully.


## 12. Feature Engineering for Churn Analysis

Following the exploratory data analysis stage, the next phase of the project focuses on feature engineering. The purpose of this section is to transform the cleaned and integrated RavenStack dataset into a structured analytical feature set that can support churn-focused analysis, statistical testing, segmentation, and later predictive modeling.

At this stage, the project moves from descriptive exploration into variable construction. The objective is not only to preserve the original business signals captured in the relational tables, but also to create higher-value analytical features that better represent customer behavior, commercial structure, product engagement, support experience, and churn-related patterns at the account level.

This section therefore aims to build a unified feature table that converts raw relational data into modeling-ready variables. These engineered features will serve as the foundation for the next stages of the project, including churn interpretation, risk profiling, and predictive analysis.


## 12.1 Define the Modeling Unit and Target

This step establishes the modeling foundation for churn analysis. Before engineering features, the analytical unit of the project must be defined clearly.

In this project, the feature engineering stage will be performed at the **account level**, because churn is ultimately treated as a customer-level business outcome rather than as an isolated support-ticket event, usage record, or subscription transaction.

The objective of this step is to:

- confirm the analytical grain of the feature table
- define the target variable that will later be used in churn-oriented analysis or modeling
- verify that the integrated account-level dataset is ready for further feature engineering

This step creates the formal starting point for transforming the exploratory findings into structured variables suitable for downstream churn analysis.

In [65]:
# =========================================================
# 12.1 Define the Modeling Unit and Target
# =========================================================

# Work from the integrated account-level table created in 11.6
feature_df = account_eda.copy()

print("=== 12.1 Define the Modeling Unit and Target ===")

# ---------------------------------------------------------
# A) Confirm analytical grain
# ---------------------------------------------------------
print("\n--- Analytical Grain Check ---")
print("Rows in feature table:", len(feature_df))
print("Unique account_id:", feature_df["account_id"].nunique())

# ---------------------------------------------------------
# B) Define target variable
# ---------------------------------------------------------
# Target = whether the account is churned at account level
feature_df["target_churn"] = pd.to_numeric(
    feature_df["churn_flag_num"], errors="coerce"
).fillna(0).astype(int)

print("\n--- Target Variable Summary ---")
print("Target field created: target_churn")

print("\nTarget distribution:")
print(feature_df["target_churn"].value_counts(dropna=False))

print("\nTarget distribution (%):")
print((feature_df["target_churn"].value_counts(normalize=True, dropna=False) * 100).round(2))

# ---------------------------------------------------------
# C) Quick structural validation
# ---------------------------------------------------------
print("\n--- Feature Table Structure ---")
print("Shape:", feature_df.shape)

print("\nPreview:")
display(feature_df.head())

=== 12.1 Define the Modeling Unit and Target ===

--- Analytical Grain Check ---
Rows in feature table: 500
Unique account_id: 500

--- Target Variable Summary ---
Target field created: target_churn

Target distribution:
target_churn
0    390
1    110
Name: count, dtype: int64

Target distribution (%):
target_churn
0    78.0
1    22.0
Name: proportion, dtype: float64

--- Feature Table Structure ---
Shape: (500, 48)

Preview:


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,...,churn_event_count,total_refund_amount_usd,reactivation_related_churn_count,preceding_upgrade_churn_count,preceding_downgrade_churn_count,usage_breadth_segment,usage_intensity_segment,support_burden_segment,revenue_segment,target_churn
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,0,0,...,2.0,0.00,0.0,1.0,0.0,Medium Breadth,Medium Intensity,Low Support,Low Revenue,0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,0,1,...,0.0,0.00,0.0,0.0,0.0,Low Breadth,Low Intensity,Low Support,Low Revenue,1
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,0,0,...,2.0,62.66,0.0,0.0,1.0,High Breadth,High Intensity,Low Support,Medium Revenue,0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,1,0,...,1.0,0.00,0.0,1.0,0.0,Low Breadth,Low Intensity,Low Support,Low Revenue,0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,0,1,...,1.0,0.00,0.0,1.0,0.0,High Breadth,Medium Intensity,High Support,High Revenue,1


### 12.1 Interpretation — Define the Modeling Unit and Target

The results confirm that the feature-engineering stage is correctly structured at the **account level**. The modeling table contains **500 rows and 500 unique account IDs**, which means that each row now represents exactly one customer account. This is an important validation step because it confirms that the analytical grain of the project is aligned with the business problem: churn is being treated as a **customer-level outcome**, not as a support-ticket, subscription, or usage-event outcome.

The target variable, `target_churn`, has also been defined successfully. Its distribution shows that **390 accounts (78%)** are labeled as non-churned, while **110 accounts (22%)** are labeled as churned. This indicates that churn is a meaningful but minority outcome in the dataset. In practical terms, the target is not extremely rare, but it is also not balanced, which means that later modeling stages should take class imbalance into account when evaluating model performance.

The final table shape of **500 rows × 48 columns** confirms that the project now has a sufficiently rich account-level feature base. The table already includes customer profile variables, commercial variables, usage-related signals, support-related measures, churn-history indicators, and the engineered segmentation fields created during the cross-domain analysis. This means the dataset is now ready to move into the next stage of feature engineering, where the focus will shift from defining the modeling structure to creating more explicit behavioral, commercial, and risk-oriented features for churn analysis.

## 12.2 Create Core Behavioral, Commercial, and Support Features

This step creates a set of engineered features from the integrated account-level table. The purpose is to move from raw aggregated variables into more informative analytical indicators that better capture customer behavior, commercial value, product engagement, support burden, and churn-history patterns.

Rather than relying only on absolute totals such as total usage, total tickets, or total revenue, this step derives normalized and ratio-based measures that are often more informative in churn analysis. These engineered variables are intended to represent how customers behave relative to their scale, how efficiently they generate revenue, how strongly they engage with the product, and whether their historical churn-related activity suggests elevated business risk.

The objective of this step is to create the first core feature set that can later support segmentation, statistical testing, and predictive modeling.

In [66]:
# =========================================================
# 12.2 Create Core Behavioral, Commercial, and Support Features
# =========================================================

feature_df = feature_df.copy()

print("=== 12.2 Create Core Behavioral, Commercial, and Support Features ===")

# ---------------------------------------------------------
# A) Helper function for safe division
# ---------------------------------------------------------
def safe_divide(numerator, denominator):
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)

# ---------------------------------------------------------
# B) Behavioral / usage features
# ---------------------------------------------------------
feature_df["usage_count_per_subscription"] = safe_divide(
    feature_df["total_usage_count"], feature_df["subscription_count"]
)

feature_df["usage_events_per_subscription"] = safe_divide(
    feature_df["usage_event_count"], feature_df["subscription_count"]
)

feature_df["usage_duration_per_subscription"] = safe_divide(
    feature_df["total_usage_duration_secs"], feature_df["subscription_count"]
)

feature_df["usage_duration_per_event"] = safe_divide(
    feature_df["total_usage_duration_secs"], feature_df["usage_event_count"]
)

feature_df["features_used_per_subscription"] = safe_divide(
    feature_df["unique_features_used"], feature_df["subscription_count"]
)

feature_df["error_rate_per_usage"] = safe_divide(
    feature_df["total_error_count"], feature_df["total_usage_count"]
)

feature_df["error_count_per_subscription"] = safe_divide(
    feature_df["total_error_count"], feature_df["subscription_count"]
)

# ---------------------------------------------------------
# C) Support-related features
# ---------------------------------------------------------
feature_df["tickets_per_subscription"] = safe_divide(
    feature_df["ticket_count"], feature_df["subscription_count"]
)

feature_df["escalation_rate"] = safe_divide(
    feature_df["escalated_ticket_count"], feature_df["ticket_count"]
)

feature_df["resolution_hours_per_ticket"] = safe_divide(
    feature_df["avg_resolution_time_hours"] * feature_df["ticket_count"],
    feature_df["ticket_count"]
)

feature_df["response_minutes_per_ticket"] = safe_divide(
    feature_df["avg_first_response_time_minutes"] * feature_df["ticket_count"],
    feature_df["ticket_count"]
)

feature_df["has_support_tickets"] = (feature_df["ticket_count"] > 0).astype(int)
feature_df["has_escalation"] = (feature_df["escalated_ticket_count"] > 0).astype(int)

# ---------------------------------------------------------
# D) Commercial / revenue features
# ---------------------------------------------------------
feature_df["mrr_per_subscription"] = safe_divide(
    feature_df["total_mrr"], feature_df["subscription_count"]
)

feature_df["arr_per_subscription"] = safe_divide(
    feature_df["total_arr"], feature_df["subscription_count"]
)

feature_df["mrr_per_seat"] = safe_divide(
    feature_df["total_mrr"], feature_df["total_seats"]
)

feature_df["arr_per_seat"] = safe_divide(
    feature_df["total_arr"], feature_df["total_seats"]
)

feature_df["subscriptions_per_seat"] = safe_divide(
    feature_df["subscription_count"], feature_df["total_seats"]
)

feature_df["has_trial_subscription"] = (feature_df["trial_subscription_count"] > 0).astype(int)
feature_df["has_auto_renew_majority"] = (feature_df["auto_renew_rate"] >= 0.5).astype(int)

# ---------------------------------------------------------
# E) Churn-history / risk-related features
# ---------------------------------------------------------
feature_df["has_churn_history"] = (feature_df["churn_event_count"] > 0).astype(int)
feature_df["has_refund_history"] = (feature_df["total_refund_amount_usd"] > 0).astype(int)

feature_df["refund_per_churn_event"] = safe_divide(
    feature_df["total_refund_amount_usd"], feature_df["churn_event_count"]
)

feature_df["upgrade_before_churn_rate"] = safe_divide(
    feature_df["preceding_upgrade_churn_count"], feature_df["churn_event_count"]
)

feature_df["downgrade_before_churn_rate"] = safe_divide(
    feature_df["preceding_downgrade_churn_count"], feature_df["churn_event_count"]
)

feature_df["reactivation_related_churn_rate"] = safe_divide(
    feature_df["reactivation_related_churn_count"], feature_df["churn_event_count"]
)

# ---------------------------------------------------------
# F) Simple composite exploratory signals
# ---------------------------------------------------------
feature_df["engagement_minus_friction"] = (
    feature_df["features_used_per_subscription"].fillna(0)
    + feature_df["usage_count_per_subscription"].fillna(0)
    - feature_df["error_count_per_subscription"].fillna(0)
)

feature_df["support_pressure_score"] = (
    feature_df["ticket_count"].fillna(0)
    + feature_df["escalated_ticket_count"].fillna(0)
)

feature_df["commercial_value_score"] = (
    feature_df["total_mrr"].fillna(0)
    + feature_df["total_arr"].fillna(0) / 12
)

# ---------------------------------------------------------
# G) Quick validation of the new features
# ---------------------------------------------------------
created_features = [
    "usage_count_per_subscription",
    "usage_events_per_subscription",
    "usage_duration_per_subscription",
    "usage_duration_per_event",
    "features_used_per_subscription",
    "error_rate_per_usage",
    "error_count_per_subscription",
    "tickets_per_subscription",
    "escalation_rate",
    "has_support_tickets",
    "has_escalation",
    "mrr_per_subscription",
    "arr_per_subscription",
    "mrr_per_seat",
    "arr_per_seat",
    "subscriptions_per_seat",
    "has_trial_subscription",
    "has_auto_renew_majority",
    "has_churn_history",
    "has_refund_history",
    "refund_per_churn_event",
    "upgrade_before_churn_rate",
    "downgrade_before_churn_rate",
    "reactivation_related_churn_rate",
    "engagement_minus_friction",
    "support_pressure_score",
    "commercial_value_score"
]

print("\n--- Number of Created Features ---")
print(len(created_features))

print("\n--- Created Feature Names ---")
for col in created_features:
    print(col)

print("\n--- Summary Statistics for Created Features ---")
display(feature_df[created_features].describe(include="all"))

print("\n--- Preview of Engineered Feature Table ---")
display(feature_df[["account_id", "target_churn"] + created_features].head())

=== 12.2 Create Core Behavioral, Commercial, and Support Features ===

--- Number of Created Features ---
27

--- Created Feature Names ---
usage_count_per_subscription
usage_events_per_subscription
usage_duration_per_subscription
usage_duration_per_event
features_used_per_subscription
error_rate_per_usage
error_count_per_subscription
tickets_per_subscription
escalation_rate
has_support_tickets
has_escalation
mrr_per_subscription
arr_per_subscription
mrr_per_seat
arr_per_seat
subscriptions_per_seat
has_trial_subscription
has_auto_renew_majority
has_churn_history
has_refund_history
refund_per_churn_event
upgrade_before_churn_rate
downgrade_before_churn_rate
reactivation_related_churn_rate
engagement_minus_friction
support_pressure_score
commercial_value_score

--- Summary Statistics for Created Features ---


,usage_count_per_subscription,usage_events_per_subscription,usage_duration_per_subscription,usage_duration_per_event,features_used_per_subscription,error_rate_per_usage,error_count_per_subscription,tickets_per_subscription,escalation_rate,has_support_tickets,...,has_auto_renew_majority,has_churn_history,has_refund_history,refund_per_churn_event,upgrade_before_churn_rate,downgrade_before_churn_rate,reactivation_related_churn_rate,engagement_minus_friction,support_pressure_score,commercial_value_score
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,492.000000,500.000000,...,500.00000,500.000000,500.000000,352.000000,352.000000,352.000000,352.000000,500.000000,500.000000,500.000000
mean,50.290769,5.016497,15325.644584,3056.326634,2.918462,0.056480,2.833529,0.459182,0.048009,0.984000,...,0.97800,0.704000,0.240000,14.165021,0.201089,0.090625,0.091146,50.375701,4.190000,45354.988000
std,8.193794,0.767472,2787.396523,320.061576,0.615517,0.015935,0.889845,0.330718,0.115516,0.125601,...,0.14683,0.456948,0.427511,33.009451,0.341342,0.248978,0.233929,8.123068,2.008474,34589.704811
min,23.833333,2.500000,7241.500000,1543.266667,1.750000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,23.833333,0.000000,380.000000
25%,45.296875,4.500000,13367.271825,2852.023810,2.500000,0.045566,2.200000,0.250000,0.000000,1.000000,...,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,45.068182,3.000000,23768.500000
50%,49.958333,5.000000,15241.616667,3033.168615,2.833333,0.054621,2.750000,0.400000,0.000000,1.000000,...,1.00000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,50.142857,4.000000,37667.000000
75%,55.606250,5.500000,17116.457143,3235.928744,3.222222,0.065754,3.381250,0.559524,0.000000,1.000000,...,1.00000,1.000000,0.000000,10.965000,0.333333,0.000000,0.000000,55.566667,5.000000,57749.500000
max,82.333333,7.800000,24181.400000,4199.722222,6.000000,0.109756,5.875000,3.000000,1.000000,1.000000,...,1.00000,1.000000,1.000000,277.540000,1.000000,1.000000,1.000000,84.000000,11.000000,276120.000000



--- Preview of Engineered Feature Table ---


,account_id,target_churn,usage_count_per_subscription,usage_events_per_subscription,usage_duration_per_subscription,usage_duration_per_event,features_used_per_subscription,error_rate_per_usage,error_count_per_subscription,tickets_per_subscription,...,has_auto_renew_majority,has_churn_history,has_refund_history,refund_per_churn_event,upgrade_before_churn_rate,downgrade_before_churn_rate,reactivation_related_churn_rate,engagement_minus_friction,support_pressure_score,commercial_value_score
0,A-2e4581,0,53.500000,5.500000,15233.900000,2769.800000,2.700000,0.071028,3.800000,0.200000,...,1,1,0,0.00,0.5,0.0,0.0,52.400000,2.0,25206.0
1,A-43a9e3,1,44.375000,4.375000,12642.000000,2889.600000,2.875000,0.039437,1.750000,0.375000,...,1,0,0,NaN,NaN,NaN,NaN,45.500000,3.0,20008.0
2,A-0a282f,0,54.733333,5.533333,16747.333333,3026.626506,2.266667,0.058465,3.200000,0.200000,...,1,1,1,31.33,0.0,0.5,0.0,53.800000,3.0,36572.0
3,A-1f0ac7,0,54.571429,5.857143,14646.857143,2500.682927,3.714286,0.054974,3.000000,0.285714,...,1,1,0,0.00,1.0,0.0,0.0,55.285714,2.0,18550.0
4,A-ce550d,1,64.333333,6.444444,23975.444444,3720.327586,3.555556,0.053541,3.444444,0.777778,...,1,1,0,0.00,1.0,0.0,0.0,64.444444,8.0,97522.0


### 12.2 Interpretation — Create Core Behavioral, Commercial, and Support Features

This step successfully transformed the integrated account-level dataset into a richer analytical feature set by creating **27 engineered variables** that capture customer behavior, commercial structure, support exposure, and churn-history context in a more informative way than raw totals alone. This is an important transition in the project because the data is no longer limited to descriptive aggregates such as total usage, total tickets, or total revenue. Instead, it now includes normalized and ratio-based indicators that better reflect how customers behave relative to their scale and business footprint.

From a behavioral perspective, the engineered usage variables show that customer engagement is generally active but not uniform. On average, each account records approximately **50 usage actions per subscription**, just over **5 usage events per subscription**, and about **2.9 distinct features used per subscription**. This indicates that the typical account is not using the product in a shallow or one-dimensional way; instead, many accounts engage with multiple product capabilities and generate repeated interaction across subscriptions. At the same time, the spread of these variables is meaningful, which confirms that customer engagement varies across the portfolio and supports their usefulness as churn-oriented behavioral signals.

The product-friction features also add analytical value. The average **error rate per usage** is approximately **5.6%**, while the average **error count per subscription** is about **2.83**. These measures are more informative than raw error totals because they express friction relative to usage activity and account scale. This is particularly important in churn analysis, since the business question is not simply whether errors exist, but whether specific customers appear to experience disproportionately higher friction relative to their overall engagement.

The support-related features provide a similarly improved view of operational burden. The average account records approximately **0.46 tickets per subscription**, while the average **escalation rate** remains low at around **4.8%** among accounts with support activity. Nearly all accounts have support-ticket presence, but escalation is much less common. This suggests that support exposure is widespread, yet severe support escalation remains relatively limited. As engineered variables, these features allow the next stages of the analysis to distinguish between customers who merely interact with support and those whose support experience may indicate higher friction or greater operational risk.

From a commercial perspective, the engineered revenue variables make the dataset significantly more useful for later churn-oriented analysis. Measures such as **MRR per subscription**, **ARR per subscription**, **MRR per seat**, and **ARR per seat** help normalize customer value relative to account size and subscription structure. This is more analytically useful than relying only on total revenue, because it allows the project to distinguish between customers who generate high revenue because they are large and customers who generate revenue more efficiently relative to their usage footprint or seat base.

The churn-history features are especially important because they convert raw historical churn activity into interpretable risk indicators. Variables such as **has_churn_history**, **has_refund_history**, **refund_per_churn_event**, **upgrade_before_churn_rate**, **downgrade_before_churn_rate**, and **reactivation_related_churn_rate** provide a more nuanced view of prior customer instability. These variables are valuable because churn risk is often better understood through historical patterns than through current status alone. In particular, they allow the project to represent whether a customer has prior evidence of commercial instability, refund-related exits, or repeated churn behavior.

The composite exploratory features add one more useful layer. Indicators such as **engagement_minus_friction**, **support_pressure_score**, and **commercial_value_score** combine multiple signals into simplified summaries that may later help with segmentation, early risk screening, or exploratory model interpretation. These features should still be treated as analytical constructs rather than formal business KPIs, but they are useful for reducing complexity and highlighting broader patterns.

Overall, the results of this step confirm that the dataset is now moving from a descriptive business table toward a **feature-engineered churn analysis table**. The engineered variables are diverse, interpretable, and structurally aligned with the project’s objective of explaining churn through a combination of engagement, support, commercial value, and historical customer behavior. This means the project is now ready for the next stage, where these engineered features should be reviewed for missingness, validity, redundancy, and analytical usefulness before they are used in statistical testing or predictive modeling.

## 12.3 Feature Quality Review / Missing and Validity Check

After creating the engineered feature set, the next step is to evaluate its overall quality before moving into statistical testing or predictive modeling.

The purpose of this step is to verify that the engineered variables are usable, internally consistent, and analytically reliable. In particular, this review checks for:

- missing values in the newly created features
- infinite values or division-related artifacts
- invalid or suspicious ranges
- constant or near-constant features that may carry little analytical value
- duplicated columns or redundant outputs that could affect downstream analysis

This step is important because feature engineering can introduce technical issues even when the raw data is clean. Ensuring feature-table quality at this stage helps prevent misleading statistical results and reduces the risk of introducing weak or broken variables into later churn analysis and modeling.

In [67]:
# =========================================================
# 12.3 Feature Quality Review / Missing & Validity Check
# =========================================================

import pandas as pd
import numpy as np

feature_df = feature_df.copy()

print("=== 12.3 Feature Quality Review / Missing & Validity Check ===")

# ---------------------------------------------------------
# A) List of engineered features from 12.2
# ---------------------------------------------------------
engineered_features = [
    "usage_count_per_subscription",
    "usage_events_per_subscription",
    "usage_duration_per_subscription",
    "usage_duration_per_event",
    "features_used_per_subscription",
    "error_rate_per_usage",
    "error_count_per_subscription",
    "tickets_per_subscription",
    "escalation_rate",
    "has_support_tickets",
    "has_escalation",
    "mrr_per_subscription",
    "arr_per_subscription",
    "mrr_per_seat",
    "arr_per_seat",
    "subscriptions_per_seat",
    "has_trial_subscription",
    "has_auto_renew_majority",
    "has_churn_history",
    "has_refund_history",
    "refund_per_churn_event",
    "upgrade_before_churn_rate",
    "downgrade_before_churn_rate",
    "reactivation_related_churn_rate",
    "engagement_minus_friction",
    "support_pressure_score",
    "commercial_value_score"
]

# keep only columns that actually exist
engineered_features = [col for col in engineered_features if col in feature_df.columns]

print("\nNumber of engineered features found:", len(engineered_features))

# ---------------------------------------------------------
# B) Missing values check
# ---------------------------------------------------------
print("\n--- Missing Values in Engineered Features ---")
missing_summary = feature_df[engineered_features].isna().sum().sort_values(ascending=False)
missing_nonzero = missing_summary[missing_summary > 0]

if len(missing_nonzero) > 0:
    display(missing_nonzero)
else:
    print("No missing values found.")

# ---------------------------------------------------------
# C) Infinite values check
# ---------------------------------------------------------
print("\n--- Infinite Values Check ---")
infinite_summary = pd.Series({
    col: np.isinf(pd.to_numeric(feature_df[col], errors="coerce")).sum()
    for col in engineered_features
})
infinite_nonzero = infinite_summary[infinite_summary > 0]

if len(infinite_nonzero) > 0:
    display(infinite_nonzero)
else:
    print("No infinite values found.")

# ---------------------------------------------------------
# D) Descriptive range review
# ---------------------------------------------------------
print("\n--- Descriptive Range Review ---")
display(feature_df[engineered_features].describe(include="all").T)

# ---------------------------------------------------------
# E) Constant / near-constant feature detection
# ---------------------------------------------------------
print("\n--- Constant / Near-Constant Features ---")

nunique_summary = feature_df[engineered_features].nunique(dropna=False).sort_values()
display(nunique_summary)

constant_features = nunique_summary[nunique_summary == 1].index.tolist()
near_constant_features = nunique_summary[nunique_summary <= 2].index.tolist()

print("\nConstant features:")
print(constant_features if constant_features else "None")

print("\nNear-constant features (2 or fewer unique values):")
print(near_constant_features if near_constant_features else "None")

# ---------------------------------------------------------
# F) Duplicate column name check
# ---------------------------------------------------------
print("\n--- Duplicate Column Name Check ---")
duplicate_columns = feature_df.columns[feature_df.columns.duplicated()].tolist()
print(duplicate_columns if duplicate_columns else "No duplicated column names found.")

# ---------------------------------------------------------
# G) Simple validity checks for key bounded features
# ---------------------------------------------------------
print("\n--- Bounded Feature Validity Checks ---")

bounded_checks = {}

if "error_rate_per_usage" in feature_df.columns:
    bounded_checks["error_rate_per_usage < 0"] = (feature_df["error_rate_per_usage"] < 0).sum()
    bounded_checks["error_rate_per_usage > 1"] = (feature_df["error_rate_per_usage"] > 1).sum()

if "escalation_rate" in feature_df.columns:
    bounded_checks["escalation_rate < 0"] = (feature_df["escalation_rate"] < 0).sum(skipna=True)
    bounded_checks["escalation_rate > 1"] = (feature_df["escalation_rate"] > 1).sum(skipna=True)

if "upgrade_before_churn_rate" in feature_df.columns:
    bounded_checks["upgrade_before_churn_rate < 0"] = (feature_df["upgrade_before_churn_rate"] < 0).sum(skipna=True)
    bounded_checks["upgrade_before_churn_rate > 1"] = (feature_df["upgrade_before_churn_rate"] > 1).sum(skipna=True)

if "downgrade_before_churn_rate" in feature_df.columns:
    bounded_checks["downgrade_before_churn_rate < 0"] = (feature_df["downgrade_before_churn_rate"] < 0).sum(skipna=True)
    bounded_checks["downgrade_before_churn_rate > 1"] = (feature_df["downgrade_before_churn_rate"] > 1).sum(skipna=True)

if "reactivation_related_churn_rate" in feature_df.columns:
    bounded_checks["reactivation_related_churn_rate < 0"] = (feature_df["reactivation_related_churn_rate"] < 0).sum(skipna=True)
    bounded_checks["reactivation_related_churn_rate > 1"] = (feature_df["reactivation_related_churn_rate"] > 1).sum(skipna=True)

bounded_checks_series = pd.Series(bounded_checks)
display(bounded_checks_series)

# ---------------------------------------------------------
# H) Final quick review table
# ---------------------------------------------------------
print("\n--- Final Feature Quality Snapshot ---")

quality_snapshot = pd.DataFrame({
    "missing_count": feature_df[engineered_features].isna().sum(),
    "nunique": feature_df[engineered_features].nunique(dropna=False),
    "min": feature_df[engineered_features].min(numeric_only=False),
    "max": feature_df[engineered_features].max(numeric_only=False),
})

display(quality_snapshot.sort_values("missing_count", ascending=False))

=== 12.3 Feature Quality Review / Missing & Validity Check ===

Number of engineered features found: 27

--- Missing Values in Engineered Features ---


reactivation_related_churn_rate    148
downgrade_before_churn_rate        148
upgrade_before_churn_rate          148
refund_per_churn_event             148
escalation_rate                      8
dtype: int64


--- Infinite Values Check ---
No infinite values found.

--- Descriptive Range Review ---


,count,mean,std,min,25%,50%,75%,max
usage_count_per_subscription,500.0,50.290769,8.193794,23.833333,45.296875,49.958333,55.606250,82.333333
usage_events_per_subscription,500.0,5.016497,0.767472,2.500000,4.500000,5.000000,5.500000,7.800000
usage_duration_per_subscription,500.0,15325.644584,2787.396523,7241.500000,13367.271825,15241.616667,17116.457143,24181.400000
usage_duration_per_event,500.0,3056.326634,320.061576,1543.266667,2852.023810,3033.168615,3235.928744,4199.722222
features_used_per_subscription,500.0,2.918462,0.615517,1.750000,2.500000,2.833333,3.222222,6.000000
error_rate_per_usage,500.0,0.056480,0.015935,0.000000,0.045566,0.054621,0.065754,0.109756
error_count_per_subscription,500.0,2.833529,0.889845,0.000000,2.200000,2.750000,3.381250,5.875000
tickets_per_subscription,500.0,0.459182,0.330718,0.000000,0.250000,0.400000,0.559524,3.000000
escalation_rate,492.0,0.048009,0.115516,0.000000,0.000000,0.000000,0.000000,1.000000
has_support_tickets,500.0,0.984000,0.125601,0.000000,1.000000,1.000000,1.000000,1.000000



--- Constant / Near-Constant Features ---


has_support_tickets                  2
has_auto_renew_majority              2
has_trial_subscription               2
has_escalation                       2
has_churn_history                    2
has_refund_history                   2
reactivation_related_churn_rate      7
downgrade_before_churn_rate          8
upgrade_before_churn_rate            8
escalation_rate                     12
support_pressure_score              12
tickets_per_subscription            80
features_used_per_subscription     111
refund_per_churn_event             122
usage_events_per_subscription      186
error_count_per_subscription       194
subscriptions_per_seat             413
usage_count_per_subscription       415
engagement_minus_friction          446
error_rate_per_usage               470
commercial_value_score             496
arr_per_subscription               497
mrr_per_subscription               497
arr_per_seat                       497
mrr_per_seat                       497
usage_duration_per_event 


Constant features:
None

Near-constant features (2 or fewer unique values):
['has_support_tickets', 'has_auto_renew_majority', 'has_trial_subscription', 'has_escalation', 'has_churn_history', 'has_refund_history']

--- Duplicate Column Name Check ---
No duplicated column names found.

--- Bounded Feature Validity Checks ---


error_rate_per_usage < 0               0
error_rate_per_usage > 1               0
escalation_rate < 0                    0
escalation_rate > 1                    0
upgrade_before_churn_rate < 0          0
upgrade_before_churn_rate > 1          0
downgrade_before_churn_rate < 0        0
downgrade_before_churn_rate > 1        0
reactivation_related_churn_rate < 0    0
reactivation_related_churn_rate > 1    0
dtype: int64


--- Final Feature Quality Snapshot ---


,missing_count,nunique,min,max
reactivation_related_churn_rate,148,7,0.000000,1.000000
downgrade_before_churn_rate,148,8,0.000000,1.000000
upgrade_before_churn_rate,148,8,0.000000,1.000000
refund_per_churn_event,148,122,0.000000,277.540000
escalation_rate,8,12,0.000000,1.000000
usage_count_per_subscription,0,415,23.833333,82.333333
arr_per_seat,0,497,50.666667,2156.059701
support_pressure_score,0,12,0.000000,11.000000
engagement_minus_friction,0,446,23.833333,84.000000
has_refund_history,0,2,0.000000,1.000000


### 12.3 Interpretation — Feature Quality Review / Missing and Validity Check

The feature-quality review confirms that the engineered feature set is **generally clean, usable, and analytically reliable**, with no major technical defects that would block downstream churn analysis or modeling. All **27 engineered features** were found successfully in the feature table, which confirms that the feature-engineering step completed as intended.

A key strength of the feature set is that **no infinite values were detected**, which means the ratio-based calculations were handled correctly and did not produce broken outputs from division-related issues. In addition, the bounded-feature checks show that all rate-based variables — including `error_rate_per_usage`, `escalation_rate`, `upgrade_before_churn_rate`, `downgrade_before_churn_rate`, and `reactivation_related_churn_rate` — remain within valid logical ranges between 0 and 1. This is an important validation result because it confirms that the engineered ratios behave consistently and are safe for further analysis.

The missing-value review shows that missingness is **limited and explainable rather than problematic**. The churn-history ratio variables — `refund_per_churn_event`, `upgrade_before_churn_rate`, `downgrade_before_churn_rate`, and `reactivation_related_churn_rate` — each contain **148 missing values**, which is analytically expected because these features are only meaningful for accounts that actually have churn history. In other words, accounts with no churn events cannot logically have refund-per-churn or upgrade-before-churn rates, so these missing values reflect business structure rather than feature failure. Similarly, `escalation_rate` contains **8 missing values**, which is also understandable because accounts with no support tickets do not have a defined escalation rate.

The near-constant feature review shows that several binary indicators have only two unique values, such as `has_support_tickets`, `has_auto_renew_majority`, `has_trial_subscription`, `has_escalation`, `has_churn_history`, and `has_refund_history`. This is not an error, but it does mean these features are structurally simple and may contribute less information than the richer continuous variables unless they interact meaningfully with other features. Importantly, there are **no fully constant features**, so there is no variable that would be completely uninformative.

The descriptive range review also suggests that the engineered variables are well behaved and broadly realistic. Usage, support, revenue, and churn-history features all show meaningful variation across accounts, which is a positive signal for later statistical testing and predictive modeling. In particular, variables such as `usage_count_per_subscription`, `error_rate_per_usage`, `tickets_per_subscription`, `mrr_per_seat`, and `commercial_value_score` show enough spread to be analytically useful rather than flat or overly compressed.

Overall, this step confirms that the engineered feature table is in **good analytical condition**. The remaining missing values are business-expected, the rate features are valid, the table contains no duplicated column names, and the feature set shows sufficient variability for downstream use. As a result, the project is now ready to move into the next stage, where the engineered variables can be examined more directly against the churn target through statistical testing, segmentation, and feature-importance style analysis.

## 12.4 Statistical Testing of Engineered Features vs Target

After constructing the engineered feature set and reviewing its quality, the next step is to test whether the newly created variables differ meaningfully across the churn target.

The purpose of this step is to move beyond descriptive inspection and begin evaluating which engineered features show statistically meaningful differences between churned and non-churned accounts. This helps identify which variables may carry stronger analytical value for downstream churn interpretation and later predictive modeling.

The analysis in this step focuses on two groups of engineered variables:

- continuous and ratio-based features, which will be compared across churn groups using non-parametric statistical testing
- binary indicator features, which will be evaluated through contingency-based significance testing

This step is important because it provides the first structured evidence on which engineered variables appear more strongly associated with the churn target at the account level.


In [68]:
# =========================================================
# 12.4 Statistical Testing of Engineered Features vs Target
# =========================================================

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, chi2_contingency

feature_df = feature_df.copy()

print("=== 12.4 Statistical Testing of Engineered Features vs Target ===")

# ---------------------------------------------------------
# A) Define feature groups
# ---------------------------------------------------------
continuous_features = [
    "usage_count_per_subscription",
    "usage_events_per_subscription",
    "usage_duration_per_subscription",
    "usage_duration_per_event",
    "features_used_per_subscription",
    "error_rate_per_usage",
    "error_count_per_subscription",
    "tickets_per_subscription",
    "escalation_rate",
    "mrr_per_subscription",
    "arr_per_subscription",
    "mrr_per_seat",
    "arr_per_seat",
    "subscriptions_per_seat",
    "refund_per_churn_event",
    "upgrade_before_churn_rate",
    "downgrade_before_churn_rate",
    "reactivation_related_churn_rate",
    "engagement_minus_friction",
    "support_pressure_score",
    "commercial_value_score"
]

binary_features = [
    "has_support_tickets",
    "has_escalation",
    "has_trial_subscription",
    "has_auto_renew_majority",
    "has_churn_history",
    "has_refund_history"
]

# keep only columns that actually exist
continuous_features = [col for col in continuous_features if col in feature_df.columns]
binary_features = [col for col in binary_features if col in feature_df.columns]

# ---------------------------------------------------------
# B) Continuous features: Mann-Whitney U test
# ---------------------------------------------------------
print("\n--- Continuous Feature Testing (Mann-Whitney U) ---")

continuous_results = []

for col in continuous_features:
    group_0 = feature_df.loc[feature_df["target_churn"] == 0, col].dropna()
    group_1 = feature_df.loc[feature_df["target_churn"] == 1, col].dropna()

    if len(group_0) > 0 and len(group_1) > 0:
        try:
            stat, p_value = mannwhitneyu(group_0, group_1, alternative="two-sided")
        except ValueError:
            stat, p_value = np.nan, np.nan
    else:
        stat, p_value = np.nan, np.nan

    continuous_results.append({
        "feature": col,
        "non_churn_mean": group_0.mean() if len(group_0) > 0 else np.nan,
        "churn_mean": group_1.mean() if len(group_1) > 0 else np.nan,
        "non_churn_median": group_0.median() if len(group_0) > 0 else np.nan,
        "churn_median": group_1.median() if len(group_1) > 0 else np.nan,
        "mean_difference": (group_1.mean() - group_0.mean()) if len(group_0) > 0 and len(group_1) > 0 else np.nan,
        "mannwhitney_u_stat": stat,
        "p_value": p_value
    })

continuous_results_df = pd.DataFrame(continuous_results).sort_values("p_value", ascending=True)

print("\nTop continuous features by statistical significance:")
display(continuous_results_df.head(15))

# ---------------------------------------------------------
# C) Binary features: Chi-square test
# ---------------------------------------------------------
print("\n--- Binary Feature Testing (Chi-Square) ---")

binary_results = []

for col in binary_features:
    contingency = pd.crosstab(feature_df[col], feature_df["target_churn"])

    if contingency.shape[0] >= 2 and contingency.shape[1] >= 2:
        chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
    else:
        chi2_stat, p_value, dof = np.nan, np.nan, np.nan

    churn_rate_if_0 = feature_df.loc[feature_df[col] == 0, "target_churn"].mean() if (feature_df[col] == 0).sum() > 0 else np.nan
    churn_rate_if_1 = feature_df.loc[feature_df[col] == 1, "target_churn"].mean() if (feature_df[col] == 1).sum() > 0 else np.nan

    binary_results.append({
        "feature": col,
        "churn_rate_if_0": churn_rate_if_0,
        "churn_rate_if_1": churn_rate_if_1,
        "chi2_stat": chi2_stat,
        "p_value": p_value,
        "degrees_of_freedom": dof
    })

binary_results_df = pd.DataFrame(binary_results).sort_values("p_value", ascending=True)

print("\nBinary features ranked by statistical significance:")
display(binary_results_df)

# ---------------------------------------------------------
# D) Significant features snapshot
# ---------------------------------------------------------
print("\n--- Significant Feature Snapshot (p < 0.05) ---")

significant_continuous = continuous_results_df[continuous_results_df["p_value"] < 0.05]
significant_binary = binary_results_df[binary_results_df["p_value"] < 0.05]

print("\nSignificant continuous features:")
display(significant_continuous if len(significant_continuous) > 0 else pd.DataFrame())

print("\nSignificant binary features:")
display(significant_binary if len(significant_binary) > 0 else pd.DataFrame())

# ---------------------------------------------------------
# E) Quick interpretation helper
# ---------------------------------------------------------
print("\n--- Quick Directional Summary ---")

directional_summary = continuous_results_df[[
    "feature", "non_churn_mean", "churn_mean", "mean_difference", "p_value"
]].copy()

directional_summary["direction"] = np.where(
    directional_summary["mean_difference"] > 0,
    "Higher in churned accounts",
    "Higher in non-churned accounts"
)

display(directional_summary.head(15))

=== 12.4 Statistical Testing of Engineered Features vs Target ===

--- Continuous Feature Testing (Mann-Whitney U) ---

Top continuous features by statistical significance:


,feature,non_churn_mean,churn_mean,non_churn_median,churn_median,mean_difference,mannwhitney_u_stat,p_value
5,error_rate_per_usage,0.057219,0.053858,0.054981,0.053699,-0.003361,23558.5,0.115228
10,arr_per_subscription,27783.599437,24995.066185,23669.818182,21772.650000,-2788.533252,23356.5,0.154393
9,mrr_per_subscription,2315.299953,2082.922182,1972.484848,1814.387500,-232.377771,23356.5,0.154393
16,downgrade_before_churn_rate,0.079543,0.131556,0.000000,0.000000,0.052013,9734.0,0.164956
14,refund_per_churn_event,12.667629,19.695389,0.000000,0.000000,7.027760,9510.0,0.184200
7,tickets_per_subscription,0.468448,0.426327,0.400000,0.379808,-0.042121,23077.5,0.223793
6,error_count_per_subscription,2.858827,2.743836,2.794737,2.732143,-0.114991,23066.5,0.227184
12,arr_per_seat,923.236636,877.960610,907.003040,852.196833,-45.276026,22991.0,0.249700
11,mrr_per_seat,76.936386,73.163384,75.583587,71.016403,-3.773002,22991.0,0.249700
18,engagement_minus_friction,50.235661,50.872205,49.881944,51.000000,0.636543,19999.5,0.278604



--- Binary Feature Testing (Chi-Square) ---

Binary features ranked by statistical significance:


,feature,churn_rate_if_0,churn_rate_if_1,chi2_stat,p_value,degrees_of_freedom
2,has_trial_subscription,0.175258,0.230769,1.099103,0.294463,1
5,has_refund_history,0.207895,0.258333,1.074127,0.300015,1
1,has_escalation,0.212714,0.252747,0.481495,0.487746,1
4,has_churn_history,0.236486,0.213068,0.210500,0.646376,1
3,has_auto_renew_majority,0.272727,0.218814,0.003467,0.953048,1
0,has_support_tickets,0.250000,0.219512,0.000000,1.000000,1



--- Significant Feature Snapshot (p < 0.05) ---

Significant continuous features:


""



Significant binary features:


""



--- Quick Directional Summary ---


,feature,non_churn_mean,churn_mean,mean_difference,p_value,direction
5,error_rate_per_usage,0.057219,0.053858,-0.003361,0.115228,Higher in non-churned accounts
10,arr_per_subscription,27783.599437,24995.066185,-2788.533252,0.154393,Higher in non-churned accounts
9,mrr_per_subscription,2315.299953,2082.922182,-232.377771,0.154393,Higher in non-churned accounts
16,downgrade_before_churn_rate,0.079543,0.131556,0.052013,0.164956,Higher in churned accounts
14,refund_per_churn_event,12.667629,19.695389,7.027760,0.184200,Higher in churned accounts
7,tickets_per_subscription,0.468448,0.426327,-0.042121,0.223793,Higher in non-churned accounts
6,error_count_per_subscription,2.858827,2.743836,-0.114991,0.227184,Higher in non-churned accounts
12,arr_per_seat,923.236636,877.960610,-45.276026,0.249700,Higher in non-churned accounts
11,mrr_per_seat,76.936386,73.163384,-3.773002,0.249700,Higher in non-churned accounts
18,engagement_minus_friction,50.235661,50.872205,0.636543,0.278604,Higher in churned accounts


## 12.5 Prepare the Feature Set for Modeling / Feature Selection

After creating and reviewing the engineered feature set, the next step is to prepare the final modeling dataset.

The purpose of this step is to define which variables should be retained for downstream churn modeling and which variables should be excluded. This includes removing identifiers, obvious leakage-related fields, duplicated target-like fields, and variables that may not be suitable for direct modeling in their current form.

This step also helps separate the dataset into:

- target variable
- candidate numerical features
- candidate categorical features
- excluded columns

The objective is to produce a clean and well-defined feature matrix that can later be used for baseline modeling, feature selection, and model evaluation.

In [69]:
# =========================================================
# 12.5 Prepare the Feature Set for Modeling / Feature Selection
# =========================================================

import pandas as pd
import numpy as np

modeling_df = feature_df.copy()

print("=== 12.5 Prepare the Feature Set for Modeling / Feature Selection ===")

# ---------------------------------------------------------
# A) Define target
# ---------------------------------------------------------
target_col = "target_churn"

# ---------------------------------------------------------
# B) Define columns to exclude
# ---------------------------------------------------------
# IDs, names, direct leakage-like fields, and duplicated target-like columns
exclude_cols = [
    "account_id",
    "account_name",
    "signup_date",
    "churn_flag",
    "churn_flag_num",
    "target_churn",
]

# Optional: exclude segmentation labels from 11.6 if you don't want to model with them directly
segment_cols = [
    "usage_breadth_segment",
    "usage_intensity_segment",
    "support_burden_segment",
    "revenue_segment"
]

# Keep only columns that actually exist
exclude_cols = [col for col in exclude_cols if col in modeling_df.columns]
segment_cols = [col for col in segment_cols if col in modeling_df.columns]

# ---------------------------------------------------------
# C) Separate numeric and categorical candidates
# ---------------------------------------------------------
numeric_cols = modeling_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = modeling_df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

# Remove excluded columns from candidate lists
numeric_features = [col for col in numeric_cols if col not in exclude_cols]
categorical_features = [col for col in categorical_cols if col not in exclude_cols and col not in segment_cols]

# ---------------------------------------------------------
# D) Remove near-constant numeric features
# ---------------------------------------------------------
near_constant_numeric = [
    col for col in numeric_features
    if modeling_df[col].nunique(dropna=False) <= 2
]

# We will not drop all binary fields automatically, but we want to inspect them
print("\n--- Near-Constant Numeric/Binary Features ---")
print(near_constant_numeric if near_constant_numeric else "None")

# ---------------------------------------------------------
# E) Missing value profile for modeling candidates
# ---------------------------------------------------------
print("\n--- Missing Values in Candidate Features ---")

candidate_features = numeric_features + categorical_features
missing_candidates = modeling_df[candidate_features].isna().sum().sort_values(ascending=False)
display(missing_candidates[missing_candidates > 0] if missing_candidates.sum() > 0 else pd.Series(dtype="int64"))

# ---------------------------------------------------------
# F) Final selected feature groups
# ---------------------------------------------------------
# For baseline modeling, start with numeric features only
baseline_numeric_features = [col for col in numeric_features if col != target_col]

# Optional: keep selected categorical variables for later encoding
baseline_categorical_features = categorical_features.copy()

print("\n--- Final Feature Group Summary ---")
print("Target column:", target_col)
print("Number of numeric candidate features:", len(numeric_features))
print("Number of categorical candidate features:", len(categorical_features))
print("Number of baseline numeric features:", len(baseline_numeric_features))
print("Number of baseline categorical features:", len(baseline_categorical_features))

# ---------------------------------------------------------
# G) Preview feature lists
# ---------------------------------------------------------
print("\n--- Baseline Numeric Features ---")
for col in baseline_numeric_features:
    print(col)

print("\n--- Baseline Categorical Features ---")
for col in baseline_categorical_features:
    print(col)

print("\n--- Excluded Columns ---")
for col in exclude_cols + segment_cols:
    print(col)

# ---------------------------------------------------------
# H) Create baseline X / y objects
# ---------------------------------------------------------
X_numeric = modeling_df[baseline_numeric_features].copy()
y = modeling_df[target_col].copy()

print("\n--- Baseline Modeling Shapes ---")
print("X_numeric shape:", X_numeric.shape)
print("y shape:", y.shape)

print("\nPreview of X_numeric:")
display(X_numeric.head())

print("\nPreview of y:")
display(y.head())

=== 12.5 Prepare the Feature Set for Modeling / Feature Selection ===

--- Near-Constant Numeric/Binary Features ---
['is_trial', 'is_trial_num', 'has_support_tickets', 'has_escalation', 'has_trial_subscription', 'has_auto_renew_majority', 'has_churn_history', 'has_refund_history']

--- Missing Values in Candidate Features ---


downgrade_before_churn_rate        148
upgrade_before_churn_rate          148
refund_per_churn_event             148
reactivation_related_churn_rate    148
response_minutes_per_ticket          8
resolution_hours_per_ticket          8
escalation_rate                      8
dtype: int64


--- Final Feature Group Summary ---
Target column: target_churn
Number of numeric candidate features: 63
Number of categorical candidate features: 4
Number of baseline numeric features: 63
Number of baseline categorical features: 4

--- Baseline Numeric Features ---
seats
is_trial
is_trial_num
subscription_count
trial_subscription_count
churn_subscription_count
total_seats
avg_subscription_seats
total_mrr
total_arr
avg_mrr
avg_arr
trial_rate
churn_rate
auto_renew_rate
usage_event_count
total_usage_count
total_usage_duration_secs
total_error_count
unique_features_used
beta_feature_usage_rate
avg_usage_count
avg_usage_duration_secs
avg_error_count
ticket_count
escalated_ticket_count
avg_resolution_time_hours
avg_first_response_time_minutes
avg_satisfaction_score
churn_event_count
total_refund_amount_usd
reactivation_related_churn_count
preceding_upgrade_churn_count
preceding_downgrade_churn_count
usage_count_per_subscription
usage_events_per_subscription
usage_duration_per_subscription


,seats,is_trial,is_trial_num,subscription_count,trial_subscription_count,churn_subscription_count,total_seats,avg_subscription_seats,total_mrr,total_arr,...,has_auto_renew_majority,has_churn_history,has_refund_history,refund_per_churn_event,upgrade_before_churn_rate,downgrade_before_churn_rate,reactivation_related_churn_rate,engagement_minus_friction,support_pressure_score,commercial_value_score
0,9,0,0.0,10,2.0,0.0,312,31.200000,12603,151236,...,1,1,0,0.00,0.5,0.0,0.0,52.400000,2.0,25206.0
1,18,0,0.0,8,0.0,0.0,176,22.000000,10004,120048,...,1,0,0,NaN,NaN,NaN,NaN,45.500000,3.0,20008.0
2,1,0,0.0,15,1.0,3.0,282,18.800000,18286,219432,...,1,1,1,31.33,0.0,0.5,0.0,53.800000,3.0,36572.0
3,24,1,1.0,7,1.0,0.0,209,29.857143,9275,111300,...,1,1,0,0.00,1.0,0.0,0.0,55.285714,2.0,18550.0
4,35,0,0.0,9,1.0,2.0,424,47.111111,48761,585132,...,1,1,0,0.00,1.0,0.0,0.0,64.444444,8.0,97522.0



Preview of y:


0    0
1    1
2    0
3    0
4    1
Name: target_churn, dtype: int32

### 12.5 Interpretation — Prepare the Feature Set for Modeling / Feature Selection

This step confirms that the churn feature table is now **structurally ready for modeling**, with a clearly defined target variable, a broad set of candidate features, and a documented exclusion logic.

#### 1. Modeling structure is now clearly defined  
The final modeling setup uses:

- **`target_churn`** as the target variable  
- **63 numeric candidate features**
- **4 categorical candidate features**
- a feature matrix of shape **(500, 63)** for the baseline numerical modeling stage

This means the project now has a sufficiently rich set of variables to move from descriptive analysis into predictive churn modeling.

---

#### 2. Exclusion logic is appropriate  
Several columns were correctly excluded from direct modeling, including:

- identifiers such as `account_id` and `account_name`
- temporal registration information such as `signup_date`
- duplicated target-like columns such as `churn_flag` and `churn_flag_num`
- segmentation labels created for exploratory analysis only, such as:
  - `usage_breadth_segment`
  - `usage_intensity_segment`
  - `support_burden_segment`
  - `revenue_segment`

This is an important modeling decision because these fields either do not carry predictive value in their raw form, may introduce leakage, or were designed for descriptive segmentation rather than direct model input.

---

#### 3. The feature space is broad and multi-dimensional  
The baseline numerical feature set includes variables from all major analytical domains:

- **commercial features** such as revenue, ARR, MRR, seat-related measures, and subscription structure
- **behavioral features** such as usage volume, feature breadth, and error-related measures
- **support features** such as ticket burden, escalation behavior, and support timing
- **churn-history features** such as refund history, reactivation-related churn, and upgrade/downgrade-before-churn signals

This is a strong result because it means the modeling table is not biased toward one business dimension only. Instead, it reflects the full customer lifecycle and preserves the multi-factor view that emerged from the EDA.

---

#### 4. Missing values remain limited and interpretable  
The remaining missing values are concentrated in a small set of churn-history and support-rate variables:

- `refund_per_churn_event`
- `upgrade_before_churn_rate`
- `downgrade_before_churn_rate`
- `reactivation_related_churn_rate`
- `response_minutes_per_ticket`
- `resolution_hours_per_ticket`
- `escalation_rate`

These missing values are still **business-expected rather than problematic**. For example:

- churn-history ratios are undefined for accounts without churn history
- escalation and support-time rates may be undefined for accounts without support events

This means the modeling table is usable, but these variables will likely require **imputation or explicit handling** before fitting a formal model.

---

#### 5. Several binary features are near-constant  
The feature review also shows a small set of binary indicators with only two unique values, including:

- `has_support_tickets`
- `has_escalation`
- `has_trial_subscription`
- `has_auto_renew_majority`
- `has_churn_history`
- `has_refund_history`

These features are not invalid, but they are structurally simple and may contribute less information on their own compared with richer continuous variables. Their value may become clearer later when combined with other features rather than interpreted in isolation.

---

### Final Modeling Preparation Conclusion

Overall, this step successfully converts the engineered feature table into a **model-ready analytical structure**. The target is clearly defined, the baseline feature groups are separated properly, exclusion logic is documented, and the final numeric feature matrix is large enough and diverse enough to support baseline churn modeling.

The most important takeaway is that the project is now ready to move into the next analytical stage: **building the first baseline churn model** and testing how these features behave together in a predictive setting.

## 12.6 Build the First Baseline Logistic Regression Model

This step builds the first baseline churn model using Logistic Regression. The purpose is to test whether the engineered account-level feature set contains enough predictive signal to distinguish between churned and non-churned accounts.

A baseline model is important because it provides the first practical assessment of modeling readiness. It helps answer whether the engineered variables work better together than they did individually in descriptive or statistical testing.

In this step, the modeling workflow will:

- use the baseline numeric feature set created in the previous step
- handle missing values through median imputation
- standardize the numeric features
- split the data into training and testing subsets
- fit a Logistic Regression classifier
- evaluate baseline performance using classification metrics and ROC-AUC
- inspect model coefficients to identify the strongest directional signals

This step is not intended to produce the final churn model. Instead, it establishes the first predictive benchmark and helps determine whether the project is ready for more advanced modeling and feature refinement.

In [70]:
# =========================================================
# 12.6 Build the First Baseline Logistic Regression Model
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=== 12.6 Build the First Baseline Logistic Regression Model ===")

# ---------------------------------------------------------
# A) Prepare X and y
# ---------------------------------------------------------
X = X_numeric.copy()
y_model = y.copy()

print("\n--- Baseline Input Shapes ---")
print("X shape:", X.shape)
print("y shape:", y_model.shape)

print("\nTarget distribution:")
print(y_model.value_counts(dropna=False))

# ---------------------------------------------------------
# B) Train-test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

print("\n--- Train/Test Split ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# ---------------------------------------------------------
# C) Build baseline pipeline
# ---------------------------------------------------------
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# ---------------------------------------------------------
# D) Fit model
# ---------------------------------------------------------
baseline_model.fit(X_train, y_train)

# ---------------------------------------------------------
# E) Predictions
# ---------------------------------------------------------
y_pred = baseline_model.predict(X_test)
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# F) Evaluation metrics
# ---------------------------------------------------------
print("\n--- Baseline Classification Metrics ---")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_summary)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
display(cm_df)

# ---------------------------------------------------------
# G) Coefficient importance
# ---------------------------------------------------------
print("\n--- Logistic Regression Coefficients ---")

log_reg_model = baseline_model.named_steps["log_reg"]

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": log_reg_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 15 Features by Absolute Coefficient Strength:")
display(coef_df.head(15))

print("\nTop Positive Coefficients (associated with higher churn probability):")
display(coef_df.sort_values("coefficient", ascending=False).head(10))

print("\nTop Negative Coefficients (associated with lower churn probability):")
display(coef_df.sort_values("coefficient", ascending=True).head(10))

=== 12.6 Build the First Baseline Logistic Regression Model ===

--- Baseline Input Shapes ---
X shape: (500, 63)
y shape: (500,)

Target distribution:
target_churn
0    390
1    110
Name: count, dtype: int64

--- Train/Test Split ---
X_train shape: (400, 63)
X_test shape: (100, 63)
y_train distribution:
target_churn
0    312
1     88
Name: count, dtype: int64
y_test distribution:
target_churn
0    78
1    22
Name: count, dtype: int64

--- Baseline Classification Metrics ---


,Metric,Value
0,Accuracy,0.540000
1,Precision,0.260000
2,Recall,0.590909
3,F1 Score,0.361111
4,ROC-AUC,0.529138



Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.53      0.64        78
           1       0.26      0.59      0.36        22

    accuracy                           0.54       100
   macro avg       0.54      0.56      0.50       100
weighted avg       0.70      0.54      0.58       100


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,41,37
Actual 1,9,13



--- Logistic Regression Coefficients ---

Top 15 Features by Absolute Coefficient Strength:


,feature,coefficient,abs_coefficient
13,churn_rate,0.733051,0.733051
40,error_count_per_subscription,0.683388,0.683388
39,error_rate_per_usage,-0.614176,0.614176
5,churn_subscription_count,-0.602572,0.602572
25,escalated_ticket_count,-0.596629,0.596629
56,refund_per_churn_event,0.525917,0.525917
7,avg_subscription_seats,-0.441966,0.441966
46,has_escalation,0.396378,0.396378
12,trial_rate,-0.388349,0.388349
4,trial_subscription_count,0.349189,0.349189



Top Positive Coefficients (associated with higher churn probability):


,feature,coefficient,abs_coefficient
13,churn_rate,0.733051,0.733051
40,error_count_per_subscription,0.683388,0.683388
56,refund_per_churn_event,0.525917,0.525917
46,has_escalation,0.396378,0.396378
4,trial_subscription_count,0.349189,0.349189
15,usage_event_count,0.347343,0.347343
38,features_used_per_subscription,0.340865,0.340865
29,churn_event_count,0.320613,0.320613
42,escalation_rate,0.258411,0.258411
58,downgrade_before_churn_rate,0.243205,0.243205



Top Negative Coefficients (associated with lower churn probability):


,feature,coefficient,abs_coefficient
39,error_rate_per_usage,-0.614176,0.614176
5,churn_subscription_count,-0.602572,0.602572
25,escalated_ticket_count,-0.596629,0.596629
7,avg_subscription_seats,-0.441966,0.441966
12,trial_rate,-0.388349,0.388349
30,total_refund_amount_usd,-0.334103,0.334103
59,reactivation_related_churn_rate,-0.316564,0.316564
35,usage_events_per_subscription,-0.296149,0.296149
21,avg_usage_count,-0.261870,0.261870
54,has_churn_history,-0.253185,0.253185


### 12.6 Interpretation — Baseline Logistic Regression Model

The first baseline Logistic Regression model provides an important early benchmark for churn prediction, but its overall performance indicates that the current feature set does **not yet produce a strong predictive model** in its present form.

#### 1. Overall model performance is weak
The model achieved:

- **Accuracy = 0.54**
- **Precision = 0.26**
- **Recall = 0.59**
- **F1 Score = 0.36**
- **ROC-AUC = 0.53**

These results suggest that the model is only performing **slightly above random discrimination**. In particular, the **ROC-AUC of 0.53** indicates that the model currently has very limited ability to separate churned from non-churned accounts in a reliable way.

---

#### 2. The model is better at catching churn than confirming it
The model’s **recall for the churn class is 0.59**, which means it successfully identifies a little more than half of the true churned accounts in the test set. However, its **precision is only 0.26**, which means that many of the accounts predicted as churned are actually non-churned.

This is also visible in the confusion matrix:

- **13 true positives**
- **9 false negatives**
- **37 false positives**
- **41 true negatives**

In practical terms, the model is reasonably sensitive to churn, but it produces **too many false alarms**. This kind of behavior may still be useful in early risk screening, but it is not strong enough yet for reliable operational deployment.

---

#### 3. The coefficient results should be interpreted carefully
The coefficient table highlights several features with relatively large weights, including:

- `churn_rate`
- `error_count_per_subscription`
- `error_rate_per_usage`
- `churn_subscription_count`
- `escalated_ticket_count`
- `refund_per_churn_event`
- `has_escalation`
- `trial_subscription_count`
- `usage_event_count`
- `features_used_per_subscription`

These coefficients indicate which variables the model is using most strongly in its current classification boundary. Positive coefficients are associated with a higher predicted churn probability, while negative coefficients are associated with lower predicted churn probability.

However, these results should **not yet be interpreted as final business drivers**, because several of the strongest variables appear to be **very close to the churn outcome itself**. For example, variables such as:

- `churn_rate`
- `churn_subscription_count`
- `churn_event_count`
- `refund_per_churn_event`
- `has_churn_history`

are not purely pre-churn behavioral predictors. Instead, they reflect outcomes or churn-history conditions that are likely too close to the target. This means the current model is useful as a **technical baseline**, but not yet as a clean predictive churn model for business deployment.

---

#### 4. Evidence of target leakage should be addressed before trusting the model
A particularly important conclusion from this baseline is that the model should be **re-run after removing leakage-prone variables**. Any feature that directly encodes churn status, churn history, or post-churn financial consequences may artificially distort coefficient importance and reduce the analytical credibility of the model.

This is critical because the goal of churn modeling is to predict churn **before it happens**, not to explain churn using variables that are already part of the churn outcome itself. Therefore, the current baseline is useful as a first diagnostic step, but it should be treated as a preliminary benchmark rather than a final predictive result.

---

### Baseline Modeling Conclusion

The baseline Logistic Regression model confirms that the project now has a functioning predictive pipeline, but the current model performance remains weak and should not yet be considered satisfactory. The low ROC-AUC and modest class-separation power suggest that the feature set still requires refinement, while the coefficient structure indicates that some high-impact variables may introduce leakage.

The correct next step is therefore to:

- remove target-adjacent and leakage-prone features
- rebuild the baseline model using only valid pre-churn signals
- compare performance after feature refinement
- and then consider more advanced models if needed

In summary, the baseline model is valuable because it establishes the first modeling benchmark, but it also shows clearly that **further feature filtering and refinement are necessary before meaningful churn prediction can be achieved.**

## 12.7 Remove Leakage Features and Rebuild the Baseline Model

The previous baseline model showed weak overall performance and suggested that some high-impact variables may be too close to the churn target. This creates a risk of target leakage, where the model benefits from information that would not be available in a true pre-churn prediction setting.

The objective of this step is therefore to rebuild the baseline churn model after removing leakage-prone and target-adjacent variables. This allows the project to test whether the remaining behavioral, support, and commercial features still contain meaningful predictive signal when the model is restricted to more realistic pre-churn inputs.

This step is important because a churn model should rely on signals that could be observed before churn occurs, rather than variables that directly encode churn outcomes or post-churn history.

In [71]:
# =========================================================
# 12.7 Remove Leakage Features and Rebuild the Baseline Model
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=== 12.7 Remove Leakage Features and Rebuild the Baseline Model ===")

# ---------------------------------------------------------
# A) Start from the same numeric feature matrix
# ---------------------------------------------------------
X_model = X_numeric.copy()
y_model = y.copy()

# ---------------------------------------------------------
# B) Define leakage-prone / target-adjacent features to remove
# ---------------------------------------------------------
leakage_features = [
    # direct churn-like variables
    "churn_subscription_count",
    "churn_rate",
    "churn_event_count",
    "has_churn_history",

    # refund / post-churn outcome variables
    "total_refund_amount_usd",
    "has_refund_history",
    "refund_per_churn_event",

    # churn-history transition variables
    "reactivation_related_churn_count",
    "preceding_upgrade_churn_count",
    "preceding_downgrade_churn_count",
    "upgrade_before_churn_rate",
    "downgrade_before_churn_rate",
    "reactivation_related_churn_rate",
]

# keep only columns that actually exist
leakage_features = [col for col in leakage_features if col in X_model.columns]

# remove them
X_model_reduced = X_model.drop(columns=leakage_features, errors="ignore")

print("\n--- Leakage Features Removed ---")
for col in leakage_features:
    print(col)

print("\nOriginal X shape:", X_model.shape)
print("Reduced X shape:", X_model_reduced.shape)

# ---------------------------------------------------------
# C) Train-test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_model_reduced,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

print("\n--- Train/Test Split ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# ---------------------------------------------------------
# D) Build clean baseline pipeline
# ---------------------------------------------------------
clean_baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# ---------------------------------------------------------
# E) Fit model
# ---------------------------------------------------------
clean_baseline_model.fit(X_train, y_train)

# ---------------------------------------------------------
# F) Predictions
# ---------------------------------------------------------
y_pred = clean_baseline_model.predict(X_test)
y_pred_proba = clean_baseline_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# G) Evaluation
# ---------------------------------------------------------
print("\n--- Rebuilt Baseline Classification Metrics ---")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_summary)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
display(cm_df)

# ---------------------------------------------------------
# H) Coefficient review
# ---------------------------------------------------------
print("\n--- Logistic Regression Coefficients After Leakage Removal ---")

log_reg_model = clean_baseline_model.named_steps["log_reg"]

coef_df = pd.DataFrame({
    "feature": X_model_reduced.columns,
    "coefficient": log_reg_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 15 Features by Absolute Coefficient Strength:")
display(coef_df.head(15))

print("\nTop Positive Coefficients (associated with higher churn probability):")
display(coef_df.sort_values("coefficient", ascending=False).head(10))

print("\nTop Negative Coefficients (associated with lower churn probability):")
display(coef_df.sort_values("coefficient", ascending=True).head(10))

=== 12.7 Remove Leakage Features and Rebuild the Baseline Model ===

--- Leakage Features Removed ---
churn_subscription_count
churn_rate
churn_event_count
has_churn_history
total_refund_amount_usd
has_refund_history
refund_per_churn_event
reactivation_related_churn_count
preceding_upgrade_churn_count
preceding_downgrade_churn_count
upgrade_before_churn_rate
downgrade_before_churn_rate
reactivation_related_churn_rate

Original X shape: (500, 63)
Reduced X shape: (500, 50)

--- Train/Test Split ---
X_train shape: (400, 50)
X_test shape: (100, 50)
y_train distribution:
target_churn
0    312
1     88
Name: count, dtype: int64
y_test distribution:
target_churn
0    78
1    22
Name: count, dtype: int64

--- Rebuilt Baseline Classification Metrics ---


,Metric,Value
0,Accuracy,0.470000
1,Precision,0.218182
2,Recall,0.545455
3,F1 Score,0.311688
4,ROC-AUC,0.490676



Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.45      0.57        78
           1       0.22      0.55      0.31        22

    accuracy                           0.47       100
   macro avg       0.50      0.50      0.44       100
weighted avg       0.65      0.47      0.51       100


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,35,43
Actual 1,10,12



--- Logistic Regression Coefficients After Leakage Removal ---

Top 15 Features by Absolute Coefficient Strength:


,feature,coefficient,abs_coefficient
32,error_rate_per_usage,-0.654688,0.654688
33,error_count_per_subscription,0.635882,0.635882
23,escalated_ticket_count,-0.608710,0.608710
6,avg_subscription_seats,-0.494696,0.494696
39,has_escalation,0.422924,0.422924
3,subscription_count,-0.382837,0.382837
28,usage_events_per_subscription,-0.330362,0.330362
13,usage_event_count,0.318706,0.318706
17,unique_features_used,0.276824,0.276824
15,total_usage_duration_secs,-0.274028,0.274028



Top Positive Coefficients (associated with higher churn probability):


,feature,coefficient,abs_coefficient
33,error_count_per_subscription,0.635882,0.635882
39,has_escalation,0.422924,0.422924
13,usage_event_count,0.318706,0.318706
17,unique_features_used,0.276824,0.276824
4,trial_subscription_count,0.239899,0.239899
35,escalation_rate,0.223221,0.223221
14,total_usage_count,0.212820,0.212820
45,has_trial_subscription,0.202611,0.202611
0,seats,0.175025,0.175025
5,total_seats,0.131420,0.131420



Top Negative Coefficients (associated with lower churn probability):


,feature,coefficient,abs_coefficient
32,error_rate_per_usage,-0.654688,0.654688
23,escalated_ticket_count,-0.608710,0.608710
6,avg_subscription_seats,-0.494696,0.494696
3,subscription_count,-0.382837,0.382837
28,usage_events_per_subscription,-0.330362,0.330362
15,total_usage_duration_secs,-0.274028,0.274028
11,trial_rate,-0.257909,0.257909
19,avg_usage_count,-0.237124,0.237124
18,beta_feature_usage_rate,-0.203480,0.203480
21,avg_error_count,-0.144553,0.144553


### 12.7 Interpretation — Remove Leakage Features and Rebuild the Baseline Model

After removing the leakage-prone and target-adjacent variables, the rebuilt baseline Logistic Regression model becomes **more analytically realistic**, but its performance declines further. This is a useful result rather than a failure, because it shows that the earlier baseline model was likely benefiting from information that was too close to the churn outcome itself.

#### 1. Model performance became weaker after leakage removal
The rebuilt model produced the following results:

- **Accuracy = 0.47**
- **Precision = 0.22**
- **Recall = 0.55**
- **F1 Score = 0.31**
- **ROC-AUC = 0.49**

These values indicate that once the leakage-prone variables were removed, the model lost much of its already limited predictive power. In practical terms, the model is now performing **at or slightly below random discrimination**, particularly based on the **ROC-AUC of 0.49**. This suggests that the remaining valid pre-churn features, in their current form, do not provide strong enough signal for accurate churn prediction.

---

#### 2. The rebuilt model still detects some churned accounts, but with many false positives
The confusion matrix shows:

- **12 true positives**
- **10 false negatives**
- **43 false positives**
- **35 true negatives**

This means the model is still moderately sensitive to churn (**recall = 0.55**), but it remains weak in precision. In other words, it continues to identify some churned accounts, but at the cost of flagging many non-churned accounts incorrectly.

From a business perspective, this kind of model may still have limited usefulness for broad early-warning screening, but it is not reliable enough for precise churn targeting or intervention planning.

---

#### 3. Leakage removal changed the meaning of the top coefficients
Once the target-adjacent variables were removed, the strongest coefficients shifted toward more realistic pre-churn signals, including:

##### Features associated with **higher churn probability**
- `error_count_per_subscription`
- `has_escalation`
- `usage_event_count`
- `unique_features_used`
- `trial_subscription_count`
- `escalation_rate`
- `total_usage_count`
- `has_trial_subscription`

##### Features associated with **lower churn probability**
- `error_rate_per_usage`
- `escalated_ticket_count`
- `avg_subscription_seats`
- `subscription_count`
- `usage_events_per_subscription`
- `total_usage_duration_secs`
- `trial_rate`
- `avg_usage_count`

This is analytically more meaningful than the earlier coefficient ranking, because the model is now relying on actual behavioral, support, and commercial characteristics rather than on direct churn-history variables. However, the directions are still somewhat mixed and not always intuitively clean. For example, some usage-related variables appear on both positive and negative sides in different forms, which suggests that the churn dynamics are still complex and not well captured by a simple linear model.

---

#### 4. The results reinforce the idea that churn is not easily predictable from the current features alone
This rebuilt model provides an important conclusion:

> **Once leakage is removed, the remaining account-level features do not yet produce a strong churn classifier.**

This supports the broader conclusion reached earlier in the EDA and statistical testing stages:

- churn does not appear to be driven by one obvious standalone factor
- many of the valid pre-churn features are weak when used independently
- even when combined inside a baseline model, the current feature set still does not separate churned and non-churned accounts effectively

In other words, the modeling result confirms what the descriptive work had already suggested: **churn in RavenStack behaves like a multi-factor and potentially nonlinear problem.**

---

### Final Conclusion for Step 12.7

This step was essential because it removed misleading target-adjacent features and revealed the true predictive strength of the remaining pre-churn variables. The resulting performance is weak, but that weakness is analytically honest. It shows that the project now has a more trustworthy baseline, even if that baseline is not yet effective.

The most important takeaway is:

- the previous model was partly inflated by leakage-prone variables
- the cleaned feature set is more realistic
- but additional work is still needed to improve predictive quality

That additional work may include:

- refining the feature set further
- removing redundant variables
- engineering more targeted behavioral features
- introducing categorical encoding and interaction-aware modeling
- testing nonlinear models such as tree-based classifiers

At this point, the project has successfully established a **clean baseline benchmark**, and the next stage should focus on **improving predictive signal rather than simply adding more variables**.

## 12.8 Refine the Feature Set Before Trying Another Model

After removing leakage-prone variables, the next step is to refine the remaining feature set before trying another model.

The purpose of this step is to reduce redundancy and improve the analytical quality of the modeling inputs. In practice, this means removing features that are too simple, too repetitive, or too strongly correlated with other variables. Keeping highly redundant variables can make model interpretation unstable and may reduce the usefulness of a baseline linear model such as Logistic Regression.

This refinement step focuses on:

- removing near-constant features with very limited variability
- identifying highly correlated numeric features
- dropping one feature from each highly correlated pair
- producing a cleaner reduced feature matrix for the next modeling attempt

The objective is to preserve the strongest and most interpretable pre-churn signals while reducing feature overlap and unnecessary complexity.
``

In [72]:
# =========================================================
# 12.8 Refine the Feature Set Before Trying Another Model
# =========================================================

import pandas as pd
import numpy as np

print("=== 12.8 Refine the Feature Set Before Trying Another Model ===")

# ---------------------------------------------------------
# A) Start from the leakage-clean feature matrix
# ---------------------------------------------------------
X_refine = X_model_reduced.copy()

print("\n--- Initial Shape ---")
print("Initial feature matrix shape:", X_refine.shape)

# ---------------------------------------------------------
# B) Remove near-constant features
# ---------------------------------------------------------
near_constant_threshold = 2  # 2 or fewer unique values

near_constant_features_refined = [
    col for col in X_refine.columns
    if X_refine[col].nunique(dropna=False) <= near_constant_threshold
]

X_refine = X_refine.drop(columns=near_constant_features_refined, errors="ignore")

print("\n--- Near-Constant Features Removed ---")
print(near_constant_features_refined if near_constant_features_refined else "None")

print("\nShape after near-constant removal:", X_refine.shape)

# ---------------------------------------------------------
# C) Correlation-based redundancy check
# ---------------------------------------------------------
# Use numeric-only columns
numeric_cols_refined = X_refine.select_dtypes(include=["number"]).columns.tolist()

corr_matrix_refined = X_refine[numeric_cols_refined].corr(numeric_only=True).abs()

# keep only upper triangle
upper_triangle = corr_matrix_refined.where(
    np.triu(np.ones(corr_matrix_refined.shape), k=1).astype(bool)
)

# threshold for high correlation
corr_threshold = 0.90

high_corr_pairs = []
features_to_drop_corr = set()

for col in upper_triangle.columns:
    high_corr_cols = upper_triangle.index[upper_triangle[col] > corr_threshold].tolist()
    for row_feature in high_corr_cols:
        high_corr_pairs.append((row_feature, col, upper_triangle.loc[row_feature, col]))
        features_to_drop_corr.add(col)  # drop the later column by convention

features_to_drop_corr = sorted(list(features_to_drop_corr))

# Drop highly correlated features
X_refined_final = X_refine.drop(columns=features_to_drop_corr, errors="ignore")

print("\n--- Highly Correlated Feature Pairs (>|0.90|) ---")
if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs, columns=["feature_1", "feature_2", "abs_correlation"])
    display(high_corr_df.sort_values("abs_correlation", ascending=False))
else:
    print("No highly correlated pairs found above threshold.")

print("\n--- Features Removed Due to High Correlation ---")
print(features_to_drop_corr if features_to_drop_corr else "None")

print("\nShape after correlation-based reduction:", X_refined_final.shape)

# ---------------------------------------------------------
# D) Missing value profile after refinement
# ---------------------------------------------------------
print("\n--- Missing Values After Refinement ---")
missing_after_refinement = X_refined_final.isna().sum().sort_values(ascending=False)
missing_after_nonzero = missing_after_refinement[missing_after_refinement > 0]

if len(missing_after_nonzero) > 0:
    display(missing_after_nonzero)
else:
    print("No missing values found.")

# ---------------------------------------------------------
# E) Final refined feature list
# ---------------------------------------------------------
refined_feature_list = X_refined_final.columns.tolist()

print("\n--- Final Refined Feature Count ---")
print(len(refined_feature_list))

print("\n--- Final Refined Feature Names ---")
for col in refined_feature_list:
    print(col)

# ---------------------------------------------------------
# F) Save refined objects for next modeling step
# ---------------------------------------------------------
X_refined = X_refined_final.copy()

print("\n--- Refined Modeling Shapes ---")
print("X_refined shape:", X_refined.shape)
print("y shape:", y.shape)

print("\nPreview of X_refined:")
display(X_refined.head())

=== 12.8 Refine the Feature Set Before Trying Another Model ===

--- Initial Shape ---
Initial feature matrix shape: (500, 50)

--- Near-Constant Features Removed ---
['is_trial', 'is_trial_num', 'has_support_tickets', 'has_escalation', 'has_trial_subscription', 'has_auto_renew_majority']

Shape after near-constant removal: (500, 44)

--- Highly Correlated Feature Pairs (>|0.90|) ---


,feature_1,feature_2,abs_correlation
13,avg_resolution_time_hours,resolution_hours_per_ticket,1.000000
11,avg_usage_duration_secs,usage_duration_per_event,1.000000
24,total_mrr,commercial_value_score,1.000000
20,mrr_per_seat,arr_per_seat,1.000000
19,mrr_per_subscription,arr_per_subscription,1.000000
18,avg_arr,arr_per_subscription,1.000000
17,avg_mrr,arr_per_subscription,1.000000
16,avg_arr,mrr_per_subscription,1.000000
15,avg_mrr,mrr_per_subscription,1.000000
14,avg_first_response_time_minutes,response_minutes_per_ticket,1.000000



--- Features Removed Due to High Correlation ---
['arr_per_seat', 'arr_per_subscription', 'avg_arr', 'avg_subscription_seats', 'commercial_value_score', 'engagement_minus_friction', 'error_rate_per_usage', 'mrr_per_subscription', 'resolution_hours_per_ticket', 'response_minutes_per_ticket', 'support_pressure_score', 'total_arr', 'total_usage_count', 'total_usage_duration_secs', 'unique_features_used', 'usage_duration_per_event', 'usage_event_count', 'usage_events_per_subscription']

Shape after correlation-based reduction: (500, 26)

--- Missing Values After Refinement ---


escalation_rate    8
dtype: int64


--- Final Refined Feature Count ---
26

--- Final Refined Feature Names ---
seats
subscription_count
trial_subscription_count
total_seats
total_mrr
avg_mrr
trial_rate
auto_renew_rate
total_error_count
beta_feature_usage_rate
avg_usage_count
avg_usage_duration_secs
avg_error_count
ticket_count
escalated_ticket_count
avg_resolution_time_hours
avg_first_response_time_minutes
avg_satisfaction_score
usage_count_per_subscription
usage_duration_per_subscription
features_used_per_subscription
error_count_per_subscription
tickets_per_subscription
escalation_rate
mrr_per_seat
subscriptions_per_seat

--- Refined Modeling Shapes ---
X_refined shape: (500, 26)
y shape: (500,)

Preview of X_refined:


,seats,subscription_count,trial_subscription_count,total_seats,total_mrr,avg_mrr,trial_rate,auto_renew_rate,total_error_count,beta_feature_usage_rate,...,avg_first_response_time_minutes,avg_satisfaction_score,usage_count_per_subscription,usage_duration_per_subscription,features_used_per_subscription,error_count_per_subscription,tickets_per_subscription,escalation_rate,mrr_per_seat,subscriptions_per_seat
0,9,10,2.0,312,12603,1260.300000,0.200000,0.800000,38,0.072727,...,91.000000,3.000000,53.500000,15233.900000,2.700000,3.800000,0.200000,0.000000,40.394231,0.032051
1,18,8,0.0,176,10004,1250.500000,0.000000,0.750000,14,0.057143,...,73.333333,4.000000,44.375000,12642.000000,2.875000,1.750000,0.375000,0.000000,56.840909,0.045455
2,1,15,1.0,282,18286,1219.066667,0.066667,1.000000,48,0.060241,...,63.666667,4.666667,54.733333,16747.333333,2.266667,3.200000,0.200000,0.000000,64.843972,0.053191
3,24,7,1.0,209,9275,1325.000000,0.142857,0.714286,21,0.121951,...,174.000000,0.000000,54.571429,14646.857143,3.714286,3.000000,0.285714,0.000000,44.377990,0.033493
4,35,9,1.0,424,48761,5417.888889,0.111111,0.666667,31,0.068966,...,107.857143,3.800000,64.333333,23975.444444,3.555556,3.444444,0.777778,0.142857,115.002358,0.021226


### 12.8 Interpretation — Refine the Feature Set Before Trying Another Model

This step successfully reduced the modeling feature space from **50 features to 26 features**, which is a substantial improvement in terms of simplicity, redundancy reduction, and interpretability. The refinement process removed both **near-constant variables** and **highly correlated variables**, producing a cleaner and more compact feature matrix for the next modeling attempt.

#### 1. Near-constant features were removed appropriately
The removed near-constant features were mostly simple binary indicators such as:

- `is_trial`
- `is_trial_num`
- `has_support_tickets`
- `has_escalation`
- `has_trial_subscription`
- `has_auto_renew_majority`

These variables are not invalid, but they carry limited variability across accounts and are therefore less useful in a baseline linear model. Removing them helps reduce noise and allows the model to focus more on richer continuous signals.

---

#### 2. High-correlation reduction significantly improved the feature set
A large number of highly correlated variable pairs were identified, with many correlations extremely close to **1.00**. This shows that the original modeling table contained strong redundancy across several engineered variables, especially in the following areas:

- revenue features (`total_mrr`, `total_arr`, `commercial_value_score`, `mrr_per_subscription`, `arr_per_subscription`, `mrr_per_seat`, `arr_per_seat`)
- support timing features (`avg_resolution_time_hours`, `resolution_hours_per_ticket`, `avg_first_response_time_minutes`, `response_minutes_per_ticket`)
- usage activity features (`usage_event_count`, `total_usage_count`, `total_usage_duration_secs`, `usage_events_per_subscription`)
- composite features such as `engagement_minus_friction` and `support_pressure_score`

Removing one variable from each highly correlated pair is a strong modeling decision because it reduces multicollinearity and makes later coefficient interpretation more stable, especially for Logistic Regression.

---

#### 3. The refined feature set is still analytically broad
Even after reduction, the final set of **26 refined features** still preserves a balanced multi-domain structure, including:

- **commercial features** such as `total_mrr`, `avg_mrr`, `mrr_per_seat`, and `subscriptions_per_seat`
- **behavioral features** such as `avg_usage_count`, `avg_usage_duration_secs`, `usage_count_per_subscription`, and `features_used_per_subscription`
- **product-friction features** such as `total_error_count`, `avg_error_count`, and `error_count_per_subscription`
- **support features** such as `ticket_count`, `escalated_ticket_count`, `avg_resolution_time_hours`, `avg_first_response_time_minutes`, `avg_satisfaction_score`, and `escalation_rate`

This means the refinement process removed redundancy without collapsing the feature table into a single-domain view. The resulting matrix still captures the main dimensions of customer behavior and risk.

---

#### 4. Remaining missing values are minimal
After refinement, only **one feature still contains missing values**:

- `escalation_rate` with **8 missing values**

This is a very manageable issue and is consistent with business logic, since escalation rate may be undefined for accounts that do not have support tickets or escalation activity. This confirms that the refined feature set is structurally clean and requires only minimal imputation before the next modeling run.

---

### Final Conclusion for Step 12.8

This step significantly improved the modeling table by removing weak binary indicators and highly redundant numeric variables while preserving the strongest and most interpretable pre-churn signals. The final refined matrix of **500 rows × 26 columns** is much cleaner than the earlier version and is now better suited for a second baseline modeling attempt.

The most important takeaway is:

> **The refined feature set is now leaner, less redundant, easier to interpret, and more appropriate for a clean Logistic Regression retry.**

The next logical step is to rebuild the churn model using this refined feature matrix and compare its performance against the earlier baseline.


## 12.9 Rebuild the Logistic Regression Model on the Refined Feature Set

After removing leakage-prone variables and reducing redundancy in the feature set, the next step is to rebuild the baseline Logistic Regression model using the refined feature matrix.

The purpose of this step is to evaluate whether a cleaner and less redundant set of pre-churn variables improves model behavior and interpretability. By reducing unnecessary overlap among features, the model may become more stable and its coefficients may better reflect meaningful pre-churn signals.

This step follows the same baseline modeling workflow used earlier, including:

- train/test split
- missing-value imputation
- feature scaling
- Logistic Regression fitting
- classification evaluation
- coefficient inspection

The objective is to compare the refined-model results with the earlier baseline and determine whether feature refinement improved predictive quality.


In [73]:
# =========================================================
# 12.9 Rebuild the Logistic Regression Model on the Refined Feature Set
# Reviewed Version
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=== 12.9 Rebuild the Logistic Regression Model on the Refined Feature Set ===")

# ---------------------------------------------------------
# A) Prepare refined X and y
# ---------------------------------------------------------
X_final = X_refined.copy()
y_final = y.copy()

# Safety checks
X_final = X_final.loc[:, ~X_final.columns.duplicated()].copy()
X_final = X_final.apply(pd.to_numeric, errors="coerce")
y_final = pd.to_numeric(y_final, errors="coerce").fillna(0).astype(int)

print("\n--- Refined Input Shapes ---")
print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)

print("\nTarget distribution:")
print(y_final.value_counts(dropna=False))

# ---------------------------------------------------------
# B) Train-test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    random_state=42,
    stratify=y_final
)

print("\n--- Train/Test Split ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# ---------------------------------------------------------
# C) Build refined baseline pipeline
# ---------------------------------------------------------
refined_baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# ---------------------------------------------------------
# D) Fit model
# ---------------------------------------------------------
refined_baseline_model.fit(X_train, y_train)

# ---------------------------------------------------------
# E) Predictions
# ---------------------------------------------------------
y_pred = refined_baseline_model.predict(X_test)
y_pred_proba = refined_baseline_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# F) Evaluation metrics
# ---------------------------------------------------------
print("\n--- Refined Baseline Classification Metrics ---")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_summary)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
display(cm_df)

# ---------------------------------------------------------
# G) Coefficient importance
# ---------------------------------------------------------
print("\n--- Logistic Regression Coefficients After Feature Refinement ---")

log_reg_model = refined_baseline_model.named_steps["log_reg"]

coef_df = pd.DataFrame({
    "feature": X_final.columns,
    "coefficient": log_reg_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 15 Features by Absolute Coefficient Strength:")
display(coef_df.head(15))

print("\nTop Positive Coefficients (associated with higher churn probability):")
display(coef_df.sort_values("coefficient", ascending=False).head(10))

print("\nTop Negative Coefficients (associated with lower churn probability):")
display(coef_df.sort_values("coefficient", ascending=True).head(10))


=== 12.9 Rebuild the Logistic Regression Model on the Refined Feature Set ===

--- Refined Input Shapes ---
X_final shape: (500, 26)
y_final shape: (500,)

Target distribution:
target_churn
0    390
1    110
Name: count, dtype: int64

--- Train/Test Split ---
X_train shape: (400, 26)
X_test shape: (100, 26)
y_train distribution:
target_churn
0    312
1     88
Name: count, dtype: int64
y_test distribution:
target_churn
0    78
1    22
Name: count, dtype: int64

--- Refined Baseline Classification Metrics ---


,Metric,Value
0,Accuracy,0.480000
1,Precision,0.232143
2,Recall,0.590909
3,F1 Score,0.333333
4,ROC-AUC,0.526807



Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.45      0.57        78
           1       0.23      0.59      0.33        22

    accuracy                           0.48       100
   macro avg       0.51      0.52      0.45       100
weighted avg       0.67      0.48      0.52       100


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,35,43
Actual 1,9,13



--- Logistic Regression Coefficients After Feature Refinement ---

Top 15 Features by Absolute Coefficient Strength:


,feature,coefficient,abs_coefficient
12,avg_error_count,-0.625124,0.625124
21,error_count_per_subscription,0.425514,0.425514
22,tickets_per_subscription,-0.342995,0.342995
23,escalation_rate,0.332498,0.332498
14,escalated_ticket_count,-0.303183,0.303183
13,ticket_count,0.275946,0.275946
20,features_used_per_subscription,0.222165,0.222165
2,trial_subscription_count,0.216322,0.216322
3,total_seats,-0.170746,0.170746
9,beta_feature_usage_rate,-0.166937,0.166937



Top Positive Coefficients (associated with higher churn probability):


,feature,coefficient,abs_coefficient
21,error_count_per_subscription,0.425514,0.425514
23,escalation_rate,0.332498,0.332498
13,ticket_count,0.275946,0.275946
20,features_used_per_subscription,0.222165,0.222165
2,trial_subscription_count,0.216322,0.216322
1,subscription_count,0.145128,0.145128
0,seats,0.114832,0.114832
11,avg_usage_duration_secs,0.057732,0.057732
7,auto_renew_rate,0.048869,0.048869
17,avg_satisfaction_score,0.046789,0.046789



Top Negative Coefficients (associated with lower churn probability):


,feature,coefficient,abs_coefficient
12,avg_error_count,-0.625124,0.625124
22,tickets_per_subscription,-0.342995,0.342995
14,escalated_ticket_count,-0.303183,0.303183
3,total_seats,-0.170746,0.170746
9,beta_feature_usage_rate,-0.166937,0.166937
16,avg_first_response_time_minutes,-0.141558,0.141558
5,avg_mrr,-0.141370,0.141370
6,trial_rate,-0.128691,0.128691
10,avg_usage_count,-0.115935,0.115935
15,avg_resolution_time_hours,-0.074340,0.074340


### 12.9 Interpretation — Rebuild the Logistic Regression Model on the Refined Feature Set

The refined Logistic Regression model shows **only a very limited improvement over the previous leakage-clean baseline** and still remains **weak as a predictive churn model**.

#### 1. Overall predictive performance remains low
The rebuilt model achieved:

- **Accuracy = 0.48**
- **Precision = 0.23**
- **Recall = 0.59**
- **F1 Score = 0.33**
- **ROC-AUC = 0.53**

These results indicate that the model is still performing **close to random discrimination**. The **ROC-AUC of 0.53** suggests that the refined feature set contains only weak predictive signal for account-level churn in its current form.

---

#### 2. The model still catches some churn cases, but with too many false positives
The confusion matrix shows:

- **13 true positives**
- **9 false negatives**
- **43 false positives**
- **35 true negatives**

This means the model is able to identify more than half of the churned accounts (**recall = 0.59**), but it does so at the cost of many incorrect churn alerts. The **precision of 0.23** remains low, which means most accounts predicted as churned are actually non-churned.

From a business perspective, this kind of model may still be useful for **broad risk screening**, but it is not strong enough for precise intervention targeting.

---

#### 3. Feature refinement improved interpretability more than predictive power
Although the refined model did not materially improve performance, it did improve the **clarity of the coefficient structure**.

The strongest positive coefficients — associated with higher predicted churn probability — include:

- `error_count_per_subscription`
- `escalation_rate`
- `ticket_count`
- `features_used_per_subscription`
- `trial_subscription_count`

The strongest negative coefficients — associated with lower predicted churn probability — include:

- `avg_error_count`
- `tickets_per_subscription`
- `escalated_ticket_count`
- `total_seats`
- `beta_feature_usage_rate`

This coefficient structure is now more realistic than the earlier leakage-affected model, because it is driven by **pre-churn behavioral, support, and commercial signals** rather than by direct churn-history variables.

---

#### 4. The remaining signals are still mixed and not strong enough
Even after feature refinement, the coefficient directions remain somewhat mixed. Some support- and error-related features point toward higher churn risk, while others point in the opposite direction. Some usage features are positively associated with churn, while others are negative.

This reinforces the earlier conclusion of the project:

> **Churn in RavenStack does not appear to be explained by one clear linear pattern.**

Instead, the results suggest that churn is likely influenced by:

- combinations of signals
- interactions between features
- nonlinear relationships
- or customer subtypes that are not captured well by a simple Logistic Regression model

---

### Final Conclusion for Step 12.9

The refined Logistic Regression model confirms that the feature set is now cleaner and more interpretable, but it still does **not** provide strong predictive performance. The modeling pipeline is technically sound, and the refinement process improved feature quality, yet the account-level churn target remains difficult to predict with a simple linear classifier.

The most important takeaway is:

- **feature refinement improved model quality conceptually**
- **but not enough to produce a strong classifier**
- therefore, the next logical step is to move beyond baseline linear modeling and test a more flexible model that can capture nonlinear patterns and feature interactions

### Recommended next step
The project should now proceed to:

**12.10 Build a Tree-Based Baseline Model (e.g., Random Forest)**

This will help determine whether churn signal exists in the data but is being missed by Logistic Regression because of its linear assumptions.
``

## 12.10 Build a Tree-Based Baseline Model (Random Forest)

After testing the refined Logistic Regression baseline, the next step is to evaluate a more flexible tree-based model using Random Forest.

The purpose of this step is to determine whether the churn signal in the dataset is nonlinear or interaction-driven, which may not be captured effectively by a linear classifier. Random Forest is well suited for this purpose because it can model complex decision boundaries and automatically capture interactions between behavioral, commercial, and support-related features.

In this step, the workflow will:

- use the refined feature matrix created in the previous step
- handle missing values through median imputation
- split the data into training and testing subsets
- fit a Random Forest classifier
- evaluate model performance using standard classification metrics and ROC-AUC
- inspect feature importance to identify the strongest predictive signals

This step provides a stronger baseline for comparison and helps determine whether the churn problem benefits from nonlinear modeling.


In [74]:
# =========================================================
# 12.10 Build a Tree-Based Baseline Model (Random Forest)
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=== 12.10 Build a Tree-Based Baseline Model (Random Forest) ===")

# ---------------------------------------------------------
# A) Prepare X and y
# ---------------------------------------------------------
X_rf = X_refined.copy()
y_rf = y.copy()

# Safety checks
X_rf = X_rf.loc[:, ~X_rf.columns.duplicated()].copy()
X_rf = X_rf.apply(pd.to_numeric, errors="coerce")
y_rf = pd.to_numeric(y_rf, errors="coerce").fillna(0).astype(int)

print("\n--- Random Forest Input Shapes ---")
print("X_rf shape:", X_rf.shape)
print("y_rf shape:", y_rf.shape)

print("\nTarget distribution:")
print(y_rf.value_counts(dropna=False))

# ---------------------------------------------------------
# B) Train-test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_rf,
    y_rf,
    test_size=0.20,
    random_state=42,
    stratify=y_rf
)

print("\n--- Train/Test Split ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# ---------------------------------------------------------
# C) Build Random Forest pipeline
# ---------------------------------------------------------
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# ---------------------------------------------------------
# D) Fit model
# ---------------------------------------------------------
rf_model.fit(X_train, y_train)

# ---------------------------------------------------------
# E) Predictions
# ---------------------------------------------------------
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# F) Evaluation metrics
# ---------------------------------------------------------
print("\n--- Random Forest Classification Metrics ---")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_summary)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
display(cm_df)

# ---------------------------------------------------------
# G) Feature importance
# ---------------------------------------------------------
print("\n--- Random Forest Feature Importance ---")

rf_fitted = rf_model.named_steps["rf"]

importance_df = pd.DataFrame({
    "feature": X_rf.columns,
    "importance": rf_fitted.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 Features by Importance:")
display(importance_df.head(15))

# ---------------------------------------------------------
# H) Compare with majority-class baseline
# ---------------------------------------------------------
print("\n--- Majority-Class Baseline Reference ---")

majority_class_accuracy = (y_test == y_test.mode()[0]).mean()

baseline_compare = pd.DataFrame({
    "Model": ["Majority Class Baseline", "Random Forest"],
    "Accuracy": [majority_class_accuracy, accuracy],
    "ROC-AUC": [0.50, roc_auc]
})

display(baseline_compare)

=== 12.10 Build a Tree-Based Baseline Model (Random Forest) ===

--- Random Forest Input Shapes ---
X_rf shape: (500, 26)
y_rf shape: (500,)

Target distribution:
target_churn
0    390
1    110
Name: count, dtype: int64

--- Train/Test Split ---
X_train shape: (400, 26)
X_test shape: (100, 26)
y_train distribution:
target_churn
0    312
1     88
Name: count, dtype: int64
y_test distribution:
target_churn
0    78
1    22
Name: count, dtype: int64

--- Random Forest Classification Metrics ---


,Metric,Value
0,Accuracy,0.750000
1,Precision,0.200000
2,Recall,0.045455
3,F1 Score,0.074074
4,ROC-AUC,0.509324



Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.95      0.86        78
           1       0.20      0.05      0.07        22

    accuracy                           0.75       100
   macro avg       0.49      0.50      0.46       100
weighted avg       0.65      0.75      0.68       100


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,74,4
Actual 1,21,1



--- Random Forest Feature Importance ---

Top 15 Features by Importance:


,feature,importance
12,avg_error_count,0.090318
5,avg_mrr,0.060992
11,avg_usage_duration_secs,0.059576
24,mrr_per_seat,0.059432
21,error_count_per_subscription,0.057394
16,avg_first_response_time_minutes,0.052067
15,avg_resolution_time_hours,0.046548
19,usage_duration_per_subscription,0.046116
9,beta_feature_usage_rate,0.044829
18,usage_count_per_subscription,0.043561



--- Majority-Class Baseline Reference ---


,Model,Accuracy,ROC-AUC
0,Majority Class Baseline,0.78,0.500000
1,Random Forest,0.75,0.509324


### 12.10 Interpretation — Random Forest Baseline Model

The Random Forest model does **not** provide a meaningful improvement over the earlier Logistic Regression baselines. Although the model achieves an apparently high **accuracy of 0.75**, this result is misleading because the dataset is imbalanced and the model is predicting the majority class far more often than the churn class.

#### 1. Accuracy is not the right success signal here
The majority-class baseline already achieves **0.78 accuracy**, while the Random Forest achieves only **0.75**. This means the model is actually performing **worse than simply predicting every account as non-churned** in terms of raw accuracy.

This is confirmed by the **ROC-AUC of 0.51**, which is only slightly above random performance. As a result, the model currently shows **very weak discriminatory power** for the churn problem.

---

#### 2. The model performs poorly on the churn class
The most important weakness is visible in the churn-class metrics:

- **Precision = 0.20**
- **Recall = 0.05**
- **F1 Score = 0.07**

This means the model correctly identified only **1 churned account out of 22** in the test set. In other words, the Random Forest is strongly biased toward predicting the non-churn class and fails to capture the minority churn cases effectively.

The confusion matrix confirms this:

- **74 true negatives**
- **4 false positives**
- **21 false negatives**
- **1 true positive**

So while the model classifies non-churned accounts well, it is almost unusable as a churn-detection model in its current form.

---

#### 3. Feature importance is more useful descriptively than predictively
Even though predictive performance is weak, the feature importance ranking still gives useful directional insight into which variables the model pays the most attention to. The top features include:

- `avg_error_count`
- `avg_mrr`
- `avg_usage_duration_secs`
- `mrr_per_seat`
- `error_count_per_subscription`
- `avg_first_response_time_minutes`
- `avg_resolution_time_hours`
- `usage_duration_per_subscription`
- `beta_feature_usage_rate`
- `usage_count_per_subscription`

This suggests that the model sees value in a mixture of:

- **error / friction signals**
- **commercial value signals**
- **support-effort signals**
- **usage-depth signals**

So while the model does not predict churn well, it still reinforces the broader project conclusion that churn is likely influenced by a **combination of usage, support, and commercial behavior**, rather than by one isolated feature.

---

### Final Conclusion for Step 12.10

The Random Forest baseline confirms that moving to a nonlinear model does **not** solve the problem automatically. In its current form, the model performs poorly on the churn class and does not outperform even a trivial majority-class baseline in a meaningful way.

The most important takeaway is:

> **The current feature set does not yet provide strong predictive power for account-level churn, even when a more flexible tree-based model is used.**

This suggests that the next improvement should focus on:

- better feature design
- stronger class-imbalance handling
- threshold tuning
- and possibly reframing the problem at a different analytical level (for example subscription-level instead of account-level, if business logic allows)

At this stage, the project has successfully shown that:
- the modeling pipeline works
- the feature set is technically usable
- but the current account-level churn signal remains weak for reliable prediction
``

In [75]:
# =========================================================
# 12.11 Improve the Feature Set
# =========================================================

import pandas as pd
import numpy as np

print("=== 12.11 Improve the Feature Set ===")

# ---------------------------------------------------------
# A) Start from the refined feature matrix
# ---------------------------------------------------------
X_improved = X_refined.copy()

print("\n--- Starting Shape ---")
print("X_refined shape:", X_refined.shape)

# ---------------------------------------------------------
# B) Helper function
# ---------------------------------------------------------
def safe_divide(numerator, denominator):
    """
    Safe division that avoids division-by-zero and inf values.
    """
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    result = num / den.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)

# ---------------------------------------------------------
# C) Add interaction / ratio features
# ---------------------------------------------------------
created_improved_features = []

# 1) Support relative to account size
if {"ticket_count", "total_seats"}.issubset(X_improved.columns):
    X_improved["tickets_per_seat"] = safe_divide(X_improved["ticket_count"], X_improved["total_seats"])
    created_improved_features.append("tickets_per_seat")

if {"escalated_ticket_count", "ticket_count"}.issubset(X_improved.columns):
    X_improved["escalations_per_ticket"] = safe_divide(X_improved["escalated_ticket_count"], X_improved["ticket_count"])
    created_improved_features.append("escalations_per_ticket")

# 2) Friction relative to support
if {"total_error_count", "ticket_count"}.issubset(X_improved.columns):
    X_improved["errors_per_ticket"] = safe_divide(X_improved["total_error_count"], X_improved["ticket_count"])
    created_improved_features.append("errors_per_ticket")

# 3) Usage relative to size
if {"usage_count_per_subscription", "seats"}.issubset(X_improved.columns):
    X_improved["usage_per_seat"] = safe_divide(X_improved["usage_count_per_subscription"], X_improved["seats"])
    created_improved_features.append("usage_per_seat")

if {"features_used_per_subscription", "seats"}.issubset(X_improved.columns):
    X_improved["feature_breadth_per_seat"] = safe_divide(X_improved["features_used_per_subscription"], X_improved["seats"])
    created_improved_features.append("feature_breadth_per_seat")

# 4) Revenue relative to behavior
if {"avg_mrr", "usage_count_per_subscription"}.issubset(X_improved.columns):
    X_improved["mrr_per_usage_unit"] = safe_divide(X_improved["avg_mrr"], X_improved["usage_count_per_subscription"])
    created_improved_features.append("mrr_per_usage_unit")

# 5) Interaction signals
if {"error_count_per_subscription", "tickets_per_subscription"}.issubset(X_improved.columns):
    X_improved["error_support_interaction"] = (
        pd.to_numeric(X_improved["error_count_per_subscription"], errors="coerce").fillna(0)
        * pd.to_numeric(X_improved["tickets_per_subscription"], errors="coerce").fillna(0)
    )
    created_improved_features.append("error_support_interaction")

if {"escalation_rate", "error_count_per_subscription"}.issubset(X_improved.columns):
    X_improved["escalation_error_interaction"] = (
        pd.to_numeric(X_improved["escalation_rate"], errors="coerce").fillna(0)
        * pd.to_numeric(X_improved["error_count_per_subscription"], errors="coerce").fillna(0)
    )
    created_improved_features.append("escalation_error_interaction")

if {"trial_rate", "mrr_per_seat"}.issubset(X_improved.columns):
    X_improved["trial_low_value_interaction"] = (
        pd.to_numeric(X_improved["trial_rate"], errors="coerce").fillna(0)
        * (1 / (1 + pd.to_numeric(X_improved["mrr_per_seat"], errors="coerce").fillna(0)))
    )
    created_improved_features.append("trial_low_value_interaction")

# ---------------------------------------------------------
# D) Identify skewed numeric features
# ---------------------------------------------------------
numeric_cols = X_improved.select_dtypes(include=["number"]).columns.tolist()

skew_before = X_improved[numeric_cols].skew(numeric_only=True).sort_values(ascending=False)

# Use a practical threshold for positive skew
skew_threshold = 1.0

skewed_features = [
    col for col in numeric_cols
    if col in skew_before.index and skew_before[col] > skew_threshold and (X_improved[col] >= 0).all()
]

# ---------------------------------------------------------
# E) Create log-transformed versions of skewed positive features
# ---------------------------------------------------------
log_transformed_features = []

for col in skewed_features:
    new_col = f"log1p_{col}"
    X_improved[new_col] = np.log1p(pd.to_numeric(X_improved[col], errors="coerce"))
    log_transformed_features.append(new_col)

# ---------------------------------------------------------
# F) Review skewness after transformation
# ---------------------------------------------------------
log_original_map = {new_col: new_col.replace("log1p_", "") for new_col in log_transformed_features}

skew_comparison_rows = []

for new_col, old_col in log_original_map.items():
    skew_comparison_rows.append({
        "original_feature": old_col,
        "original_skew": skew_before.get(old_col, np.nan),
        "transformed_feature": new_col,
        "transformed_skew": X_improved[new_col].skew()
    })

skew_comparison_df = pd.DataFrame(skew_comparison_rows)

# ---------------------------------------------------------
# G) Final checks
# ---------------------------------------------------------
print("\n--- Interaction / Ratio Features Added ---")
print(created_improved_features if created_improved_features else "None")

print("\n--- Positively Skewed Features Selected for log1p Transformation ---")
print(skewed_features if skewed_features else "None")

print("\n--- Log-Transformed Features Created ---")
print(log_transformed_features if log_transformed_features else "None")

print("\n--- Skewness Comparison (Before vs After) ---")
display(skew_comparison_df if len(skew_comparison_df) > 0 else pd.DataFrame())

print("\n--- Missing Values in Improved Feature Set ---")
missing_improved = X_improved.isna().sum().sort_values(ascending=False)
display(missing_improved[missing_improved > 0] if missing_improved.sum() > 0 else pd.Series(dtype="int64"))

print("\n--- Final Improved Feature Set Shape ---")
print("X_improved shape:", X_improved.shape)

print("\n--- Preview of Improved Feature Set ---")
display(X_improved.head())

=== 12.11 Improve the Feature Set ===

--- Starting Shape ---
X_refined shape: (500, 26)

--- Interaction / Ratio Features Added ---
['tickets_per_seat', 'escalations_per_ticket', 'errors_per_ticket', 'usage_per_seat', 'feature_breadth_per_seat', 'mrr_per_usage_unit', 'error_support_interaction', 'escalation_error_interaction', 'trial_low_value_interaction']

--- Positively Skewed Features Selected for log1p Transformation ---
['seats', 'total_seats', 'total_mrr', 'avg_mrr', 'escalated_ticket_count', 'features_used_per_subscription', 'tickets_per_subscription', 'subscriptions_per_seat', 'tickets_per_seat', 'usage_per_seat', 'feature_breadth_per_seat', 'mrr_per_usage_unit', 'error_support_interaction', 'escalation_error_interaction', 'trial_low_value_interaction']

--- Log-Transformed Features Created ---
['log1p_seats', 'log1p_total_seats', 'log1p_total_mrr', 'log1p_avg_mrr', 'log1p_escalated_ticket_count', 'log1p_features_used_per_subscription', 'log1p_tickets_per_subscription', 'log1

,original_feature,original_skew,transformed_feature,transformed_skew
0,seats,2.325967,log1p_seats,-0.248111
1,total_seats,2.499399,log1p_total_seats,-0.261989
2,total_mrr,2.507210,log1p_total_mrr,-1.086569
3,avg_mrr,2.523401,log1p_avg_mrr,-0.561966
4,escalated_ticket_count,2.157900,log1p_escalated_ticket_count,1.772571
5,features_used_per_subscription,1.096353,log1p_features_used_per_subscription,0.527980
6,tickets_per_subscription,2.653368,log1p_tickets_per_subscription,1.253577
7,subscriptions_per_seat,2.849829,log1p_subscriptions_per_seat,2.526153
8,tickets_per_seat,7.085710,log1p_tickets_per_seat,6.296042
9,usage_per_seat,2.548451,log1p_usage_per_seat,0.874509



--- Missing Values in Improved Feature Set ---


escalations_per_ticket    8
escalation_rate           8
errors_per_ticket         8
dtype: int64


--- Final Improved Feature Set Shape ---
X_improved shape: (500, 50)

--- Preview of Improved Feature Set ---


,seats,subscription_count,trial_subscription_count,total_seats,total_mrr,avg_mrr,trial_rate,auto_renew_rate,total_error_count,beta_feature_usage_rate,...,log1p_features_used_per_subscription,log1p_tickets_per_subscription,log1p_subscriptions_per_seat,log1p_tickets_per_seat,log1p_usage_per_seat,log1p_feature_breadth_per_seat,log1p_mrr_per_usage_unit,log1p_error_support_interaction,log1p_escalation_error_interaction,log1p_trial_low_value_interaction
0,9,10,2.0,312,12603,1260.300000,0.200000,0.800000,38,0.072727,...,1.308333,0.182322,0.031548,0.006390,1.937942,0.262364,3.200997,0.565314,0.00000,0.004820
1,18,8,0.0,176,10004,1250.500000,0.000000,0.750000,14,0.057143,...,1.354546,0.318454,0.044452,0.016902,1.242793,0.148181,3.373493,0.504556,0.00000,0.000000
2,1,15,1.0,282,18286,1219.066667,0.066667,1.000000,48,0.060241,...,1.183770,0.182322,0.051825,0.010582,4.020578,1.183770,3.147287,0.494696,0.00000,0.001012
3,24,7,1.0,209,9275,1325.000000,0.142857,0.714286,21,0.121951,...,1.550597,0.251314,0.032944,0.009524,1.185954,0.143894,3.230018,0.619039,0.00000,0.003143
4,35,9,1.0,424,48761,5417.888889,0.111111,0.666667,31,0.068966,...,1.516347,0.575364,0.021004,0.016375,1.043133,0.096752,4.445188,1.302644,0.40016,0.000957


### 12.11 Interpretation — Improve the Feature Set

This step improved the account-level feature set by adding more informative interaction and ratio-based variables and by applying logarithmic transformation to several positively skewed numeric features. As a result, the modeling table became richer and more analytically expressive.

At the same time, the feature review shows that not all skew-related issues were fully resolved. A small subset of features still remains strongly skewed even after transformation, particularly those that reflect sparse support burden, seat-normalized behavior, or rare interaction effects. This suggests that additional treatment is needed before expecting better predictive performance from the current account-level model.

For that reason, the next stage of the workflow will proceed in three practical steps:

1. **Adjust the most heavily skewed features** through clipping and selective flag conversion
2. **Handle class imbalance explicitly** using RandomOverSampler, SMOTE, and threshold tuning
3. **Rebuild the churn models** using the improved and rebalanced feature set, beginning with Logistic Regression and then comparing it with a tree-based model

The objective of this sequence is to strengthen the usable predictive signal before moving to a different analytical unit or more advanced modeling choices.

## 12.11.1 Skewed Feature Adjustment

This step applies targeted treatment to the most heavily skewed engineered features that remained unstable after the earlier transformation stage.

The goal is not to remove these variables immediately, but to make them more modeling-friendly by:

- clipping extreme outliers in selected ratio features
- converting sparse high-risk interaction features into binary indicator flags
- removing the original unstable versions after creating more stable replacements

This step is intended to preserve business meaning while reducing distortion caused by extreme skewness.

In [76]:
# =========================================================
# 12.11.1 Skewed Feature Adjustment
# =========================================================

import pandas as pd
import numpy as np

print("=== 12.11.1 Skewed Feature Adjustment ===")

# Start from the improved feature set from 12.11
X_adjusted = X_improved.copy()

# ---------------------------------------------------------
# A) Review target skew-sensitive features
# ---------------------------------------------------------
skew_sensitive_features = [
    "tickets_per_seat",
    "feature_breadth_per_seat",
    "escalation_error_interaction",
    "trial_low_value_interaction",
]

existing_skew_sensitive = [col for col in skew_sensitive_features if col in X_adjusted.columns]

print("\n--- Skew-Sensitive Features Found ---")
print(existing_skew_sensitive if existing_skew_sensitive else "None")

# ---------------------------------------------------------
# B) Clip extreme ratio features
# ---------------------------------------------------------
clipped_features_created = []

for col in ["tickets_per_seat", "feature_breadth_per_seat"]:
    if col in X_adjusted.columns:
        upper_cap = X_adjusted[col].quantile(0.99)
        new_col = f"{col}_clipped"
        X_adjusted[new_col] = X_adjusted[col].clip(upper=upper_cap)
        clipped_features_created.append(new_col)

# ---------------------------------------------------------
# C) Convert sparse interaction features to binary risk flags
# ---------------------------------------------------------
flag_features_created = []

if "escalation_error_interaction" in X_adjusted.columns:
    threshold_val = X_adjusted["escalation_error_interaction"].quantile(0.90)
    X_adjusted["has_high_escalation_error_interaction"] = (
        X_adjusted["escalation_error_interaction"] > threshold_val
    ).astype(int)
    flag_features_created.append("has_high_escalation_error_interaction")

if "trial_low_value_interaction" in X_adjusted.columns:
    threshold_val = X_adjusted["trial_low_value_interaction"].quantile(0.90)
    X_adjusted["has_trial_low_value_risk"] = (
        X_adjusted["trial_low_value_interaction"] > threshold_val
    ).astype(int)
    flag_features_created.append("has_trial_low_value_risk")

# ---------------------------------------------------------
# D) Drop original unstable skew-sensitive features
# Also drop their log-transformed versions if they exist
# ---------------------------------------------------------
cols_to_drop_after_adjustment = [
    "tickets_per_seat",
    "feature_breadth_per_seat",
    "escalation_error_interaction",
    "trial_low_value_interaction",
    "log1p_tickets_per_seat",
    "log1p_feature_breadth_per_seat",
    "log1p_escalation_error_interaction",
    "log1p_trial_low_value_interaction"
]

cols_to_drop_after_adjustment = [
    col for col in cols_to_drop_after_adjustment if col in X_adjusted.columns
]

X_adjusted = X_adjusted.drop(columns=cols_to_drop_after_adjustment, errors="ignore")

# ---------------------------------------------------------
# E) Quick review
# ---------------------------------------------------------
print("\n--- Clipped Features Created ---")
print(clipped_features_created if clipped_features_created else "None")

print("\n--- Flag Features Created ---")
print(flag_features_created if flag_features_created else "None")

print("\n--- Original Skew-Sensitive Features Removed ---")
print(cols_to_drop_after_adjustment if cols_to_drop_after_adjustment else "None")

print("\n--- Missing Values After Adjustment ---")
missing_after_adjustment = X_adjusted.isna().sum().sort_values(ascending=False)
print(missing_after_adjustment[missing_after_adjustment > 0] if missing_after_adjustment.sum() > 0 else "No missing values found.")

print("\n--- Shape After Adjustment ---")
print("X_adjusted shape:", X_adjusted.shape)

print("\n--- Preview of Adjusted Feature Set ---")
display(X_adjusted.head())

=== 12.11.1 Skewed Feature Adjustment ===

--- Skew-Sensitive Features Found ---
['tickets_per_seat', 'feature_breadth_per_seat', 'escalation_error_interaction', 'trial_low_value_interaction']

--- Clipped Features Created ---
['tickets_per_seat_clipped', 'feature_breadth_per_seat_clipped']

--- Flag Features Created ---
['has_high_escalation_error_interaction', 'has_trial_low_value_risk']

--- Original Skew-Sensitive Features Removed ---
['tickets_per_seat', 'feature_breadth_per_seat', 'escalation_error_interaction', 'trial_low_value_interaction', 'log1p_tickets_per_seat', 'log1p_feature_breadth_per_seat', 'log1p_escalation_error_interaction', 'log1p_trial_low_value_interaction']

--- Missing Values After Adjustment ---
escalation_rate           8
escalations_per_ticket    8
errors_per_ticket         8
dtype: int64

--- Shape After Adjustment ---
X_adjusted shape: (500, 46)

--- Preview of Adjusted Feature Set ---


,seats,subscription_count,trial_subscription_count,total_seats,total_mrr,avg_mrr,trial_rate,auto_renew_rate,total_error_count,beta_feature_usage_rate,...,log1p_features_used_per_subscription,log1p_tickets_per_subscription,log1p_subscriptions_per_seat,log1p_usage_per_seat,log1p_mrr_per_usage_unit,log1p_error_support_interaction,tickets_per_seat_clipped,feature_breadth_per_seat_clipped,has_high_escalation_error_interaction,has_trial_low_value_risk
0,9,10,2.0,312,12603,1260.300000,0.200000,0.800000,38,0.072727,...,1.308333,0.182322,0.031548,1.937942,3.200997,0.565314,0.006410,0.300000,0,0
1,18,8,0.0,176,10004,1250.500000,0.000000,0.750000,14,0.057143,...,1.354546,0.318454,0.044452,1.242793,3.373493,0.504556,0.017045,0.159722,0,0
2,1,15,1.0,282,18286,1219.066667,0.066667,1.000000,48,0.060241,...,1.183770,0.182322,0.051825,4.020578,3.147287,0.494696,0.010638,2.266667,0,0
3,24,7,1.0,209,9275,1325.000000,0.142857,0.714286,21,0.121951,...,1.550597,0.251314,0.032944,1.185954,3.230018,0.619039,0.009569,0.154762,0,0
4,35,9,1.0,424,48761,5417.888889,0.111111,0.666667,31,0.068966,...,1.516347,0.575364,0.021004,1.043133,4.445188,1.302644,0.016509,0.101587,0,0


### 12.11.1 Interpretation — Skewed Feature Adjustment

This step applied targeted treatment to the most unstable skew-sensitive engineered features that remained problematic after the earlier transformation stage. The review identified four features with especially strong skewness:

- `tickets_per_seat`
- `feature_breadth_per_seat`
- `escalation_error_interaction`
- `trial_low_value_interaction`

Rather than removing them without review, the adjustment strategy applied two different treatments based on feature behavior.

First, the ratio-based seat-normalized variables (`tickets_per_seat` and `feature_breadth_per_seat`) were **clipped** at the upper tail to reduce the impact of extreme outliers while preserving their business meaning. This is appropriate because these variables can inflate sharply when the denominator is very small.

Second, the sparse interaction variables (`escalation_error_interaction` and `trial_low_value_interaction`) were converted into **binary risk flags**. This treatment is more suitable because these variables behave like rare-event indicators rather than stable continuous signals. Their transformed versions:

- `has_high_escalation_error_interaction`
- `has_trial_low_value_risk`

retain the high-risk logic while reducing distortion caused by extreme skewness.

After creating the clipped and binary replacements, the original unstable skew-sensitive variables — along with their log-transformed versions — were removed from the feature matrix. This produced a cleaner adjusted table with **46 features**, while preserving the most meaningful information in a more modeling-friendly form.

The remaining missing values are still minimal and limited to support-related rate variables (`escalation_rate`, `escalations_per_ticket`, and `errors_per_ticket`), each with only 8 missing observations. These remain business-expected and manageable through later imputation.

Overall, this step improved the quality of the feature set by replacing unstable raw skewed signals with more robust analytical representations. The adjusted table is now better prepared for the next stage, where class imbalance will be handled before the models are rebuilt.

## 12.12 Handle Class Imbalance

The churn target remains imbalanced, with non-churned accounts clearly outnumbering churned accounts. This can bias classification models toward the majority class and reduce their ability to identify actual churn cases.

This step prepares three imbalance-handling strategies:

- **baseline training set** with no resampling
- **RandomOverSampler** to duplicate minority-class observations
- **SMOTE** to generate synthetic minority-class observations

In addition, an exploratory threshold grid is defined so that model probabilities can later be converted into class predictions using more flexible cutoffs than the default 0.50 threshold.

This step is important because weak recall or poor churn detection may be caused not only by weak features, but also by class imbalance during training.

In [80]:
# =========================================================
# 12.12 Handle Class Imbalance
# Robust Version:
# - Tries to import imblearn
# - If unavailable, tries installation in current kernel env
# - If still unavailable, falls back to Manual ROS only
# - Handles NaN before SMOTE
# =========================================================

import sys
import subprocess
import importlib
import warnings

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.utils import resample   # <-- IMPORTANT FIX

print("=== 12.12 Handle Class Imbalance ===")

# ---------------------------------------------------------
# A) Try to load imblearn safely
# ---------------------------------------------------------
IMBLEARN_AVAILABLE = False

try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    IMBLEARN_AVAILABLE = True
    print("\nimbalanced-learn is already available.")
except ModuleNotFoundError:
    print("\nimblearn not found. Attempting installation in the current kernel environment...")

    try:
        install_cmd = [sys.executable, "-m", "pip", "install", "imbalanced-learn"]
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True
        )

        print("\n--- pip install stdout ---")
        print(result.stdout[-1500:] if result.stdout else "No stdout output.")

        print("\n--- pip install stderr ---")
        print(result.stderr[-1500:] if result.stderr else "No stderr output.")

        importlib.invalidate_caches()

        try:
            from imblearn.over_sampling import RandomOverSampler, SMOTE
            IMBLEARN_AVAILABLE = True
            print("\nimbalanced-learn installed and imported successfully.")
        except ModuleNotFoundError:
            print(
                "\nWARNING: imbalanced-learn still cannot be imported in this kernel.\n"
                "The code will continue with Manual Random Oversampling only.\n"
                "If you want SMOTE, restart the kernel and rerun this cell."
            )

    except Exception as e:
        print(
            "\nWARNING: Automatic installation failed.\n"
            "The code will continue with Manual Random Oversampling only.\n"
            f"Installation error details: {e}"
        )

# ---------------------------------------------------------
# B) Prepare X and y
# ---------------------------------------------------------
X_bal = X_adjusted.copy()
y_bal = y.copy()

# Safety conversion
X_bal = X_bal.loc[:, ~X_bal.columns.duplicated()].copy()
X_bal = X_bal.apply(pd.to_numeric, errors="coerce")
y_bal = pd.to_numeric(y_bal, errors="coerce").fillna(0).astype(int)

print("\n--- Original Class Distribution ---")
print(y_bal.value_counts(dropna=False))

# ---------------------------------------------------------
# C) Train-test split
# Keep test set untouched
# ---------------------------------------------------------
X_train_base, X_test_holdout, y_train_base, y_test_holdout = train_test_split(
    X_bal,
    y_bal,
    test_size=0.20,
    random_state=42,
    stratify=y_bal
)

print("\n--- Train/Test Split ---")
print("X_train_base shape:", X_train_base.shape)
print("X_test_holdout shape:", X_test_holdout.shape)
print("y_train_base distribution:")
print(y_train_base.value_counts(dropna=False))
print("y_test_holdout distribution:")
print(y_test_holdout.value_counts(dropna=False))

# ---------------------------------------------------------
# D) Manual Random Oversampling (always available)
# ---------------------------------------------------------
train_base = X_train_base.copy()
train_base["_target_churn"] = y_train_base.values

majority_df = train_base[train_base["_target_churn"] == 0]
minority_df = train_base[train_base["_target_churn"] == 1]

minority_upsampled = resample(
    minority_df,
    replace=True,
    n_samples=len(majority_df),
    random_state=42
)

train_ros = pd.concat([majority_df, minority_upsampled], axis=0).sample(frac=1, random_state=42)

X_train_ros = train_ros.drop(columns=["_target_churn"])
y_train_ros = train_ros["_target_churn"]

print("\n--- Manual Random Oversampling Output ---")
print("X_train_ros shape:", X_train_ros.shape)
print("y_train_ros distribution:")
print(y_train_ros.value_counts(dropna=False))

# ---------------------------------------------------------
# E) SMOTE (only if imblearn is available)
# IMPORTANT: SMOTE cannot handle NaN
# So we impute training data first
# ---------------------------------------------------------
X_train_smote = None
y_train_smote = None

if IMBLEARN_AVAILABLE:
    print("\nPreparing NaN-safe training matrix for SMOTE...")

    smote_imputer = SimpleImputer(strategy="median")
    X_train_base_imputed = pd.DataFrame(
        smote_imputer.fit_transform(X_train_base),
        columns=X_train_base.columns,
        index=X_train_base.index
    )

    smote = SMOTE(random_state=42)
    X_train_smote_arr, y_train_smote_arr = smote.fit_resample(X_train_base_imputed, y_train_base)

    X_train_smote = pd.DataFrame(X_train_smote_arr, columns=X_train_base.columns)
    y_train_smote = pd.Series(y_train_smote_arr, name="target_churn")

    print("\n--- SMOTE Output ---")
    print("X_train_smote shape:", X_train_smote.shape)
    print("y_train_smote distribution:")
    print(y_train_smote.value_counts(dropna=False))
else:
    print("\n--- SMOTE Output ---")
    print("Skipped because imbalanced-learn is not available in the current kernel.")

# ---------------------------------------------------------
# F) Threshold candidates for later tuning
# ---------------------------------------------------------
threshold_grid = np.arange(0.20, 0.81, 0.05)

print("\n--- Threshold Grid for Later Tuning ---")
print(threshold_grid)

# ---------------------------------------------------------
# G) Final availability summary
# ---------------------------------------------------------
print("\n--- Availability Summary ---")
print("Manual ROS ready:", X_train_ros is not None)
print("SMOTE ready:", X_train_smote is not None)

=== 12.12 Handle Class Imbalance ===

imblearn not found. Attempting installation in the current kernel environment...

--- pip install stdout ---
 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- -

### 12.12 Interpretation — Handle Class Imbalance

This step successfully prepared the dataset for imbalance-aware modeling. The original churn target remains moderately imbalanced, with:

- **390 non-churned accounts**
- **110 churned accounts**

This confirms that the majority class still dominates the dataset and that imbalance handling is necessary to improve churn detection.

The account-level data was then split into a training set and a holdout test set, while preserving class proportions through stratified sampling. The holdout test set remains untouched, which is important because it provides a fair evaluation environment for the later models.

Two resampling strategies were then prepared on the training data:

1. **Manual Random Oversampling (ROS)**  
   This successfully balanced the training set to:
   - **312 non-churned**
   - **312 churned**

2. **SMOTE**  
   After successful installation of `imbalanced-learn` and NaN-safe preprocessing, SMOTE also produced a balanced training set of:
   - **312 non-churned**
   - **312 churned**

This is a strong result because it gives the project two valid imbalance-handling options for comparison:

- oversampling by duplication
- oversampling by synthetic generation

In addition, a probability threshold grid was created, ranging from **0.20 to 0.80**, to support later threshold tuning. This is important because the default 0.50 threshold may not be optimal in an imbalanced churn setting, especially when recall and precision need to be balanced more carefully.

Overall, this step completed successfully. The project now has:

- a clean base training set
- a balanced ROS training set
- a balanced SMOTE training set
- and a threshold grid for later tuning

This means the next modeling step can directly compare whether weak churn detection was caused by class imbalance, and whether oversampling improves the model’s ability to identify churned accounts.

## 12.13 Rebuild the Model After Imbalance Handling

This step rebuilds the churn models using the adjusted feature set and the class-imbalance strategies prepared in the previous step.

The objective is to compare multiple training strategies using the same holdout test set:

- Logistic Regression on the original imbalanced training set
- Logistic Regression after Random Oversampling
- Logistic Regression after SMOTE
- Random Forest after Random Oversampling
- Random Forest after SMOTE

This step also includes exploratory threshold tuning for the best Logistic Regression variant in order to test whether a probability cutoff other than 0.50 improves the balance between precision and recall.

The purpose is to evaluate whether weak churn detection is caused mainly by class imbalance, and whether rebalancing improves the model’s ability to identify churned accounts.

In [81]:
# =========================================================
# 12.13 Rebuild the Model After Imbalance Handling
# =========================================================

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("=== 12.13 Rebuild the Model After Imbalance Handling ===")

# ---------------------------------------------------------
# A) Helper functions
# ---------------------------------------------------------
def evaluate_classifier(model_name, fitted_model, X_test, y_test, threshold=0.50):
    y_proba = fitted_model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    return {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "y_pred": y_pred,
        "y_proba": y_proba
    }

def print_model_report(title, y_true, y_pred):
    print(f"\n{title} - Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print(f"\n{title} - Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))

# ---------------------------------------------------------
# B) Define models
# ---------------------------------------------------------
log_reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# ---------------------------------------------------------
# C) Fit models
# ---------------------------------------------------------
fitted_models = {}

# Logistic Regression - base
fitted_models["LogReg_Base"] = log_reg_pipeline.fit(X_train_base, y_train_base)

# Logistic Regression - ROS
fitted_models["LogReg_ROS"] = log_reg_pipeline.fit(X_train_ros, y_train_ros)

# Logistic Regression - SMOTE
fitted_models["LogReg_SMOTE"] = log_reg_pipeline.fit(X_train_smote, y_train_smote)

# Random Forest - ROS
fitted_models["RF_ROS"] = rf_pipeline.fit(X_train_ros, y_train_ros)

# Random Forest - SMOTE
fitted_models["RF_SMOTE"] = rf_pipeline.fit(X_train_smote, y_train_smote)

# ---------------------------------------------------------
# D) Evaluate all models at default threshold 0.50
# ---------------------------------------------------------
results_default = []

for model_name, model_obj in fitted_models.items():
    results_default.append(
        evaluate_classifier(model_name, model_obj, X_test_holdout, y_test_holdout, threshold=0.50)
    )

results_default_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["y_pred", "y_proba"]}
    for r in results_default
]).sort_values("roc_auc", ascending=False)

print("\n--- Default Threshold Model Comparison ---")
display(results_default_df)

# ---------------------------------------------------------
# E) Threshold tuning for the best Logistic Regression model
# ---------------------------------------------------------
logreg_candidates = [r for r in results_default if r["model"].startswith("LogReg")]
best_logreg_name = sorted(logreg_candidates, key=lambda x: x["roc_auc"], reverse=True)[0]["model"]
best_logreg_model = fitted_models[best_logreg_name]

threshold_results = []

for th in threshold_grid:
    eval_result = evaluate_classifier(
        best_logreg_name,
        best_logreg_model,
        X_test_holdout,
        y_test_holdout,
        threshold=th
    )
    threshold_results.append({
        "threshold": th,
        "accuracy": eval_result["accuracy"],
        "precision": eval_result["precision"],
        "recall": eval_result["recall"],
        "f1": eval_result["f1"],
        "roc_auc": eval_result["roc_auc"]
    })

threshold_results_df = pd.DataFrame(threshold_results).sort_values("f1", ascending=False)

print(f"\n--- Exploratory Threshold Tuning for {best_logreg_name} ---")
display(threshold_results_df)

best_threshold = threshold_results_df.iloc[0]["threshold"]
best_threshold_eval = evaluate_classifier(
    best_logreg_name,
    best_logreg_model,
    X_test_holdout,
    y_test_holdout,
    threshold=best_threshold
)

print(f"\nBest exploratory threshold by F1: {best_threshold:.2f}")

# ---------------------------------------------------------
# F) Detailed reports
# ---------------------------------------------------------
best_default_model_name = results_default_df.iloc[0]["model"]
best_default_result = [r for r in results_default if r["model"] == best_default_model_name][0]

print_model_report(
    f"{best_default_model_name} @ 0.50",
    y_test_holdout,
    best_default_result["y_pred"]
)

print_model_report(
    f"{best_logreg_name} @ tuned threshold {best_threshold:.2f}",
    y_test_holdout,
    best_threshold_eval["y_pred"]
)

# ---------------------------------------------------------
# G) Explainability
# ---------------------------------------------------------
print("\n--- Model Explainability Snapshot ---")

# Best logistic coefficients
best_logreg_coef = best_logreg_model.named_steps["log_reg"].coef_[0]

logreg_coef_df = pd.DataFrame({
    "feature": X_train_base.columns,
    "coefficient": best_logreg_coef
})
logreg_coef_df["abs_coefficient"] = logreg_coef_df["coefficient"].abs()
logreg_coef_df = logreg_coef_df.sort_values("abs_coefficient", ascending=False)

print(f"\nTop 15 Logistic Regression Coefficients for {best_logreg_name}:")
display(logreg_coef_df.head(15))

# RF ROS importance
rf_ros_importance_df = pd.DataFrame({
    "feature": X_train_base.columns,
    "importance": fitted_models["RF_ROS"].named_steps["rf"].feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 Random Forest Feature Importances (ROS):")
display(rf_ros_importance_df.head(15))

# RF SMOTE importance
rf_smote_importance_df = pd.DataFrame({
    "feature": X_train_base.columns,
    "importance": fitted_models["RF_SMOTE"].named_steps["rf"].feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 Random Forest Feature Importances (SMOTE):")
display(rf_smote_importance_df.head(15))

=== 12.13 Rebuild the Model After Imbalance Handling ===

--- Default Threshold Model Comparison ---


,model,threshold,accuracy,precision,recall,f1,roc_auc
3,RF_ROS,0.5,0.70,0.3,0.272727,0.285714,0.526807
4,RF_SMOTE,0.5,0.70,0.3,0.272727,0.285714,0.526807
0,LogReg_Base,0.5,0.45,0.2,0.500000,0.285714,0.490676
1,LogReg_ROS,0.5,0.45,0.2,0.500000,0.285714,0.490676
2,LogReg_SMOTE,0.5,0.45,0.2,0.500000,0.285714,0.490676



--- Exploratory Threshold Tuning for LogReg_Base ---


,threshold,accuracy,precision,recall,f1,roc_auc
0,0.20,0.28,0.234043,1.000000,0.379310,0.490676
1,0.25,0.29,0.224719,0.909091,0.360360,0.490676
3,0.35,0.33,0.222222,0.818182,0.349515,0.490676
4,0.40,0.38,0.222222,0.727273,0.340426,0.490676
2,0.30,0.30,0.214286,0.818182,0.339623,0.490676
5,0.45,0.41,0.223881,0.681818,0.337079,0.490676
7,0.55,0.55,0.232558,0.454545,0.307692,0.490676
6,0.50,0.45,0.200000,0.500000,0.285714,0.490676
9,0.65,0.70,0.250000,0.181818,0.210526,0.490676
8,0.60,0.59,0.172414,0.227273,0.196078,0.490676



Best exploratory threshold by F1: 0.20

RF_ROS @ 0.50 - Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.82      0.81        78
           1       0.30      0.27      0.29        22

    accuracy                           0.70       100
   macro avg       0.55      0.55      0.55       100
weighted avg       0.69      0.70      0.69       100


RF_ROS @ 0.50 - Confusion Matrix:


...

,feature,coefficient,abs_coefficient
33,log1p_total_mrr,1.267987,1.267987
24,mrr_per_seat,-1.010027,1.010027
32,log1p_total_seats,-0.877120,0.877120
14,escalated_ticket_count,-0.704746,0.704746
12,avg_error_count,-0.700974,0.700974
13,ticket_count,0.529306,0.529306
31,log1p_seats,-0.516007,0.516007
36,log1p_features_used_per_subscription,0.508804,0.508804
44,has_high_escalation_error_interaction,-0.467842,0.467842
21,error_count_per_subscription,0.458350,0.458350



Top 15 Random Forest Feature Importances (ROS):


,feature,importance
12,avg_error_count,0.063538
2,trial_subscription_count,0.046436
21,error_count_per_subscription,0.044473
24,mrr_per_seat,0.043037
0,seats,0.033148
6,trial_rate,0.032487
11,avg_usage_duration_secs,0.027793
31,log1p_seats,0.025563
28,usage_per_seat,0.025169
34,log1p_avg_mrr,0.024558



Top 15 Random Forest Feature Importances (SMOTE):


,feature,importance
12,avg_error_count,0.063538
2,trial_subscription_count,0.046436
21,error_count_per_subscription,0.044473
24,mrr_per_seat,0.043037
0,seats,0.033148
6,trial_rate,0.032487
11,avg_usage_duration_secs,0.027793
31,log1p_seats,0.025563
28,usage_per_seat,0.025169
34,log1p_avg_mrr,0.024558


### 12.13 Interpretation — Rebuild the Model After Imbalance Handling### 12.13 Interpretation — Rebuild the Model After key issue is that the model still misses many churned accounts:

- **6 true positives**
- **16 false negatives**

So while the Random Forest is more balanced than Logistic Regression, it is still **not strong enough** for confident churn targeting.

---

#### 3. Threshold tuning improved recall, but at a very large cost
Threshold tuning on the best Logistic Regression variant showed that the **best F1** occurred at a threshold of **0.20**.

At that threshold:

- **Recall = 1.00**
- **Precision = 0.23**
- **Accuracy = 0.28**
- **F1 = 0.38**

This means the tuned Logistic model was able to identify **all churned accounts**, but only by predicting churn for almost everyone. In the confusion matrix:

- it found **22 / 22 churned accounts**
- but also labeled **72 non-churned accounts incorrectly as churn**

This is a classic high-recall / low-precision tradeoff. It may be useful if the business wants an extremely broad **early-warning screen**, but it is not efficient for focused retention action because it creates too many false alarms.

---

#### 4. The feature rankings remain directionally useful
Even though predictive performance is still weak, the importance rankings provide useful analytical insight.

##### Logistic Regression highlights signals such as:
- `log1p_total_mrr`
- `mrr_per_seat`
- `log1p_total_seats`
- `ticket_count`
- `avg_error_count`
- `error_count_per_subscription`
- `escalations_per_ticket`
- `escalation_rate`

##### Random Forest importance also emphasizes:
- `avg_error_count`
- `trial_subscription_count`
- `error_count_per_subscription`
- `mrr_per_seat`
- `trial_rate`
- `avg_usage_duration_secs`
- `avg_first_response_time_minutes`
- `avg_resolution_time_hours`
- `usage_per_seat`

This is an important consistency signal. Even if the models are not predicting churn well, they repeatedly focus on the same broad domains:

- **error / friction**
- **trial exposure**
- **support burden**
- **commercial value**
- **usage depth**

That means these are still the most analytically meaningful axes in the problem.

---

### Final Conclusion for Step 12.13

The main conclusion is:

> **Class imbalance was not the only reason for weak churn prediction.**

Handling imbalance through **Random Oversampling**, **SMOTE**, and **threshold tuning** did not unlock strong predictive performance. This suggests that the current **account-level representation** still does not contain a sufficiently strong and separable churn signal for reliable prediction.

At this point, the findings strongly support the next logical move:

## **12.14 Build a Subscription-Level Dataset and Repeat the Pipeline**

This is likely the most valuable next step because churn may be expressed **more clearly at the subscription level** than at the aggregated account level. The account-level view may be smoothing away important variation across subscriptions, especially in a business where one account can hold multiple subscriptions with different revenue, trial, billing, and churn characteristics.

---

### Practical takeaway
- **Random Forest is currently the strongest model among those tested**
- **Threshold tuning can increase recall dramatically**, but with too many false positives
- **The feature importance results are still useful analytically**
- **The next best move is not another account-level model**, but a **subscription-level rebuild of the pipeline**

This step tested whether the weak churn-detection performance observed earlier was mainly caused by **class imbalance**. The results show that **rebalancing helped only marginally**, and the overall predictive signal at the current **account level** remains weak.

---

#### 1. Oversampling did not materially improve discrimination
At the default threshold of **0.50**, the best-performing models by ROC-AUC were:

- **RF_ROS** and **RF_SMOTE** with **ROC-AUC = 0.527**
- All three Logistic Regression variants remained at **ROC-AUC = 0.491**

This means that neither Random Oversampling nor SMOTE produced a meaningful improvement in class separation. The models are still operating **close to random discrimination**, which suggests that the main limitation is **not imbalance alone**, but also the underlying strength of the available signal in the current feature set.

---

#### 2. Random Forest is better than Logistic Regression, but still weak for churn detection
Among the default-threshold models, **Random Forest** clearly outperformed Logistic Regression in practical classification quality:

- **Accuracy = 0.70**
- **Precision = 0.30**
- **Recall = 0.27**
- **F1 = 0.29**



## 12.14.1 Build the Subscription-Level Analytical Table

This step creates the subscription-level analytical dataset for the alternative churn modeling path.

Unlike the previous account-level pipeline, this dataset is defined at the **subscription level**, meaning that each row represents one subscription rather than one customer account. This analytical shift is important because churn may appear more clearly at the subscription level, especially in cases where a single account holds multiple subscriptions with different plans, usage patterns, trial exposure, revenue values, or churn outcomes.

The purpose of this step is to construct a unified subscription-level table by combining:

- subscription-level commercial data
- account-level customer context
- product usage summary at the subscription level
- support context from the parent account
- churn-event context from the parent account

This step creates the structural foundation for repeating the churn-analysis pipeline at a more granular level.

In [82]:
# =========================================================
# 12.14.1 Build the Subscription-Level Analytical Table
# =========================================================

import pandas as pd
import numpy as np

print("=== 12.14.1 Build the Subscription-Level Analytical Table ===")

# ---------------------------------------------------------
# A) Load clean working tables
# ---------------------------------------------------------
accounts_df = accounts_clean.copy()
subscriptions_df = subscriptions_clean.copy()
feature_usage_df = feature_usage_clean.copy()
support_df = support_tickets_clean.copy()
churn_df = churn_events_clean.copy()

# ---------------------------------------------------------
# B) Helper function for robust binary conversion
# ---------------------------------------------------------
def to_binary_num(series):
    s = series.astype("string").str.strip().str.lower()
    mapped = s.map({
        "1": 1, "0": 0,
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "y": 1, "n": 0
    })
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.fillna(numeric).fillna(0).astype(float)

# ---------------------------------------------------------
# C) Normalize subscription fields
# ---------------------------------------------------------
subscriptions_df["is_trial_num"] = to_binary_num(subscriptions_df["is_trial"])
subscriptions_df["churn_flag_num"] = to_binary_num(subscriptions_df["churn_flag"])
subscriptions_df["upgrade_flag_num"] = to_binary_num(subscriptions_df["upgrade_flag"])
subscriptions_df["downgrade_flag_num"] = to_binary_num(subscriptions_df["downgrade_flag"])
subscriptions_df["auto_renew_flag_num"] = to_binary_num(subscriptions_df["auto_renew_flag"])

subscriptions_df["seats_num"] = pd.to_numeric(subscriptions_df["seats"], errors="coerce")
subscriptions_df["mrr_amount_num"] = pd.to_numeric(subscriptions_df["mrr_amount"], errors="coerce")
subscriptions_df["arr_amount_num"] = pd.to_numeric(subscriptions_df["arr_amount"], errors="coerce")

# ---------------------------------------------------------
# D) Usage summary by subscription
# ---------------------------------------------------------
feature_usage_df["is_beta_feature_num"] = to_binary_num(feature_usage_df["is_beta_feature"])
feature_usage_df["usage_count_num"] = pd.to_numeric(feature_usage_df["usage_count"], errors="coerce")
feature_usage_df["usage_duration_secs_num"] = pd.to_numeric(feature_usage_df["usage_duration_secs"], errors="coerce")
feature_usage_df["error_count_num"] = pd.to_numeric(feature_usage_df["error_count"], errors="coerce")

usage_by_subscription = (
    feature_usage_df.groupby("subscription_id")
    .agg(
        usage_event_count=("usage_id", "count"),
        total_usage_count=("usage_count_num", "sum"),
        total_usage_duration_secs=("usage_duration_secs_num", "sum"),
        total_error_count=("error_count_num", "sum"),
        unique_features_used=("feature_name", "nunique"),
        beta_feature_usage_rate=("is_beta_feature_num", "mean"),
        avg_usage_count=("usage_count_num", "mean"),
        avg_usage_duration_secs=("usage_duration_secs_num", "mean"),
        avg_error_count=("error_count_num", "mean"),
    )
    .reset_index()
)

# ---------------------------------------------------------
# E) Support summary by account
# (contextual features attached to each subscription through account_id)
# ---------------------------------------------------------
support_df["escalation_flag_num"] = to_binary_num(support_df["escalation_flag"])
support_df["resolution_time_hours_num"] = pd.to_numeric(support_df["resolution_time_hours"], errors="coerce")
support_df["first_response_time_minutes_num"] = pd.to_numeric(support_df["first_response_time_minutes"], errors="coerce")
support_df["satisfaction_score_num"] = pd.to_numeric(support_df["satisfaction_score"], errors="coerce")

if "satisfaction_score_analytical" not in support_df.columns:
    support_df["satisfaction_score_analytical"] = support_df["satisfaction_score_num"].mask(
        support_df["satisfaction_score_num"] == 0, pd.NA
    )

support_df["satisfaction_score_analytical_num"] = pd.to_numeric(
    support_df["satisfaction_score_analytical"], errors="coerce"
)

support_by_account = (
    support_df.groupby("account_id")
    .agg(
        account_ticket_count=("ticket_id", "count"),
        account_escalated_ticket_count=("escalation_flag_num", "sum"),
        account_avg_resolution_time_hours=("resolution_time_hours_num", "mean"),
        account_avg_first_response_time_minutes=("first_response_time_minutes_num", "mean"),
        account_avg_satisfaction_score=("satisfaction_score_analytical_num", "mean"),
    )
    .reset_index()
)

# ---------------------------------------------------------
# F) Churn-event context by account
# (context only; later we can remove leakage-like features if needed)
# ---------------------------------------------------------
churn_df["preceding_upgrade_flag_num"] = to_binary_num(churn_df["preceding_upgrade_flag"])
churn_df["preceding_downgrade_flag_num"] = to_binary_num(churn_df["preceding_downgrade_flag"])
churn_df["is_reactivation_num"] = to_binary_num(churn_df["is_reactivation"])
churn_df["refund_amount_usd_num"] = pd.to_numeric(churn_df["refund_amount_usd"], errors="coerce")

churn_by_account = (
    churn_df.groupby("account_id")
    .agg(
        account_churn_event_count=("churn_event_id", "count"),
        account_total_refund_amount_usd=("refund_amount_usd_num", "sum"),
        account_reactivation_related_churn_count=("is_reactivation_num", "sum"),
        account_preceding_upgrade_churn_count=("preceding_upgrade_flag_num", "sum"),
        account_preceding_downgrade_churn_count=("preceding_downgrade_flag_num", "sum"),
    )
    .reset_index()
)

# ---------------------------------------------------------
# G) Account context columns
# ---------------------------------------------------------
account_context = accounts_df[[
    "account_id",
    "industry",
    "country",
    "referral_source",
    "signup_date"
]].copy()

# ---------------------------------------------------------
# H) Build the subscription-level analytical table
# ---------------------------------------------------------
subscription_eda = (
    subscriptions_df
    .merge(account_context, on="account_id", how="left")
    .merge(usage_by_subscription, on="subscription_id", how="left")
    .merge(support_by_account, on="account_id", how="left")
    .merge(churn_by_account, on="account_id", how="left")
)

# Fill numeric missing values
numeric_cols = subscription_eda.select_dtypes(include=["number"]).columns
subscription_eda[numeric_cols] = subscription_eda[numeric_cols].fillna(0)

# ---------------------------------------------------------
# I) Define subscription-level target
# ---------------------------------------------------------
subscription_eda["target_churn_subscription"] = pd.to_numeric(
    subscription_eda["churn_flag_num"], errors="coerce"
).fillna(0).astype(int)

# ---------------------------------------------------------
# J) Quick structural checks
# ---------------------------------------------------------
print("\n--- Subscription-Level Grain Check ---")
print("Rows in subscription table:", len(subscription_eda))
print("Unique subscription_id:", subscription_eda["subscription_id"].nunique())

print("\n--- Target Distribution ---")
print(subscription_eda["target_churn_subscription"].value_counts(dropna=False))

print("\n--- Target Distribution (%) ---")
print((subscription_eda["target_churn_subscription"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\n--- Final Subscription-Level Table Shape ---")
print(subscription_eda.shape)

print("\n--- Preview of subscription_eda ---")
display(subscription_eda.head())


=== 12.14.1 Build the Subscription-Level Analytical Table ===

--- Subscription-Level Grain Check ---
Rows in subscription table: 5000
Unique subscription_id: 5000

--- Target Distribution ---
target_churn_subscription
0    4514
1     486
Name: count, dtype: int64

--- Target Distribution (%) ---
target_churn_subscription
0    90.28
1     9.72
Name: proportion, dtype: float64

--- Final Subscription-Level Table Shape ---
(5000, 46)

--- Preview of subscription_eda ---


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,...,account_escalated_ticket_count,account_avg_resolution_time_hours,account_avg_first_response_time_minutes,account_avg_satisfaction_score,account_churn_event_count,account_total_refund_amount_usd,account_reactivation_related_churn_count,account_preceding_upgrade_churn_count,account_preceding_downgrade_churn_count,target_churn_subscription
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,0,0,...,0.0,33.25,56.50,3.500000,0.0,0.00,0.0,0.0,0.0,1
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,0,0,...,0.0,33.50,156.50,4.000000,2.0,27.92,0.0,0.0,0.0,0
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,1,1,...,0.0,8.00,123.75,4.333333,2.0,27.75,0.0,0.0,0.0,0
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,0,0,...,0.0,45.00,37.00,4.000000,3.0,6.20,1.0,1.0,0.0,1
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,0,0,...,0.0,31.50,71.00,4.000000,1.0,0.00,0.0,0.0,0.0,0


In [83]:
subscription_eda.to_csv("subscription_level_analytical_table.csv", index=False)
print("subscription_level_analytical_table.csv exported successfully.")

subscription_level_analytical_table.csv exported successfully.


### 12.14.1 Interpretation — Build the Subscription-Level Analytical Table

This step successfully established the **subscription-level analytical foundation** for the second churn-modeling path. The final table contains **5,000 rows and 5,000 unique subscription IDs**, which confirms that the analytical grain is now correctly aligned at the **subscription level**, with each row representing one subscription rather than one account.

This is an important structural shift from the earlier account-level pipeline. At the subscription level, churn can be studied in a more granular way, which is especially valuable in a business environment where a single account may hold multiple subscriptions with different commercial characteristics, trial exposure, product usage patterns, and churn outcomes.

The target variable, `target_churn_subscription`, is also clearly defined and shows that:

- **4,514 subscriptions (90.28%)** are non-churned
- **486 subscriptions (9.72%)** are churned

This confirms that subscription-level churn is still an imbalanced outcome, but it is now represented at a more specific behavioral and commercial level than before. That is analytically useful because it increases the chance of detecting patterns that may have been diluted when everything was aggregated to the account level.

The final table shape of **5,000 rows × 46 columns** indicates that the subscription-level dataset is already rich and multi-dimensional. It contains:

- direct subscription-level commercial variables
- customer/account context
- usage behavior summarized at the subscription level
- support context from the parent account
- churn-event context from the parent account

Overall, this step confirms that the project now has a structurally sound and analytically richer alternative dataset for churn modeling. The subscription-level table is ready for the next stage, where feature engineering will be repeated at this more granular level to test whether churn signal becomes stronger than it was in the account-level pipeline.
``

## 12.14.2 Subscription-Level Feature Engineering

This step creates the first engineered feature set for the subscription-level churn pipeline.

The purpose of this stage is to transform the raw subscription-level analytical table into a richer set of variables that better describe:

- subscription behavior
- commercial structure
- product engagement
- product friction
- support exposure
- churn-related account context

Unlike the earlier account-level feature engineering stage, the current step focuses on one subscription at a time. This is important because subscription-level churn may be driven by more specific behavioral and commercial patterns that were partially diluted when data was aggregated to the account level.

The objective is to create a subscription-level feature table that can later support feature-quality review, imbalance handling, and churn modeling.

In [86]:
# =========================================================
# 12.14.2 Subscription-Level Feature Engineering
# Corrected Version
# =========================================================

import pandas as pd
import numpy as np

print("=== 12.14.2 Subscription-Level Feature Engineering ===")

# Start from the subscription-level analytical table
subscription_feature_df = subscription_eda.copy()

# ---------------------------------------------------------
# A) Helper function
# ---------------------------------------------------------
def safe_divide(numerator, denominator):
    """
    Safe division that works with either:
    - pandas Series
    - scalar values
    """
    num = pd.to_numeric(numerator, errors="coerce")

    # if denominator is a scalar (e.g. 1)
    if np.isscalar(denominator):
        den = denominator
        if den == 0:
            return pd.Series(np.nan, index=num.index)
        result = num / den
        return result.replace([np.inf, -np.inf], np.nan)

    # if denominator is a Series
    den = pd.to_numeric(denominator, errors="coerce")
    result = num / den.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan)

# ---------------------------------------------------------
# B) Core behavioral / usage features
# ---------------------------------------------------------
subscription_feature_df["usage_count_per_event"] = safe_divide(
    subscription_feature_df["total_usage_count"],
    subscription_feature_df["usage_event_count"]
)

subscription_feature_df["usage_duration_per_event"] = safe_divide(
    subscription_feature_df["total_usage_duration_secs"],
    subscription_feature_df["usage_event_count"]
)

subscription_feature_df["error_rate_per_usage"] = safe_divide(
    subscription_feature_df["total_error_count"],
    subscription_feature_df["total_usage_count"]
)

subscription_feature_df["feature_breadth_ratio"] = safe_divide(
    subscription_feature_df["unique_features_used"],
    subscription_feature_df["usage_event_count"]
)

# ---------------------------------------------------------
# C) Commercial features
# ---------------------------------------------------------
subscription_feature_df["mrr_per_seat"] = safe_divide(
    subscription_feature_df["mrr_amount_num"],
    subscription_feature_df["seats_num"]
)

subscription_feature_df["arr_per_seat"] = safe_divide(
    subscription_feature_df["arr_amount_num"],
    subscription_feature_df["seats_num"]
)

subscription_feature_df["has_zero_revenue"] = (
    (pd.to_numeric(subscription_feature_df["mrr_amount_num"], errors="coerce").fillna(0) == 0) |
    (pd.to_numeric(subscription_feature_df["arr_amount_num"], errors="coerce").fillna(0) == 0)
).astype(int)

subscription_feature_df["is_paid_subscription"] = (
    1 - pd.to_numeric(subscription_feature_df["is_trial_num"], errors="coerce").fillna(0)
).astype(int)

# ---------------------------------------------------------
# D) Support-context features
# ---------------------------------------------------------
subscription_feature_df["account_ticket_pressure_per_subscription"] = safe_divide(
    subscription_feature_df["account_ticket_count"],
    1
)

subscription_feature_df["account_escalation_rate"] = safe_divide(
    subscription_feature_df["account_escalated_ticket_count"],
    subscription_feature_df["account_ticket_count"]
)

subscription_feature_df["account_ticket_per_seat"] = safe_divide(
    subscription_feature_df["account_ticket_count"],
    subscription_feature_df["seats_num"]
)

# ---------------------------------------------------------
# E) Churn-context / transition features
# ---------------------------------------------------------
subscription_feature_df["account_refund_per_churn_event"] = safe_divide(
    subscription_feature_df["account_total_refund_amount_usd"],
    subscription_feature_df["account_churn_event_count"]
)

subscription_feature_df["account_upgrade_before_churn_rate"] = safe_divide(
    subscription_feature_df["account_preceding_upgrade_churn_count"],
    subscription_feature_df["account_churn_event_count"]
)

subscription_feature_df["account_downgrade_before_churn_rate"] = safe_divide(
    subscription_feature_df["account_preceding_downgrade_churn_count"],
    subscription_feature_df["account_churn_event_count"]
)

subscription_feature_df["account_reactivation_related_churn_rate"] = safe_divide(
    subscription_feature_df["account_reactivation_related_churn_count"],
    subscription_feature_df["account_churn_event_count"]
)

subscription_feature_df["has_account_churn_history"] = (
    pd.to_numeric(subscription_feature_df["account_churn_event_count"], errors="coerce").fillna(0) > 0
).astype(int)

# ---------------------------------------------------------
# F) Interaction / composite features
# ---------------------------------------------------------
subscription_feature_df["usage_minus_errors"] = (
    pd.to_numeric(subscription_feature_df["total_usage_count"], errors="coerce").fillna(0)
    - pd.to_numeric(subscription_feature_df["total_error_count"], errors="coerce").fillna(0)
)

subscription_feature_df["support_friction_interaction"] = (
    pd.to_numeric(subscription_feature_df["account_ticket_count"], errors="coerce").fillna(0)
    * pd.to_numeric(subscription_feature_df["total_error_count"], errors="coerce").fillna(0)
)

subscription_feature_df["trial_low_value_interaction"] = (
    pd.to_numeric(subscription_feature_df["is_trial_num"], errors="coerce").fillna(0)
    * (1 / (1 + pd.to_numeric(subscription_feature_df["mrr_per_seat"], errors="coerce").fillna(0)))
)

# ---------------------------------------------------------
# G) Collect created feature names
# ---------------------------------------------------------
subscription_created_features = [
    "usage_count_per_event",
    "usage_duration_per_event",
    "error_rate_per_usage",
    "feature_breadth_ratio",
    "mrr_per_seat",
    "arr_per_seat",
    "has_zero_revenue",
    "is_paid_subscription",
    "account_ticket_pressure_per_subscription",
    "account_escalation_rate",
    "account_ticket_per_seat",
    "account_refund_per_churn_event",
    "account_upgrade_before_churn_rate",
    "account_downgrade_before_churn_rate",
    "account_reactivation_related_churn_rate",
    "has_account_churn_history",
    "usage_minus_errors",
    "support_friction_interaction",
    "trial_low_value_interaction"
]

subscription_created_features = [
    col for col in subscription_created_features if col in subscription_feature_df.columns
]

# ---------------------------------------------------------
# H) Quick review
# ---------------------------------------------------------
print("\n--- Number of Created Subscription-Level Features ---")
print(len(subscription_created_features))

print("\n--- Created Feature Names ---")
for col in subscription_created_features:
    print(col)

print("\n--- Summary Statistics for Created Features ---")
display(subscription_feature_df[subscription_created_features].describe(include="all"))

print("\n--- Preview of Subscription Feature Table ---")
display(
    subscription_feature_df[
        ["subscription_id", "account_id", "target_churn_subscription"] + subscription_created_features
    ].head()
)

=== 12.14.2 Subscription-Level Feature Engineering ===

--- Number of Created Subscription-Level Features ---
19

--- Created Feature Names ---
usage_count_per_event
usage_duration_per_event
error_rate_per_usage
feature_breadth_ratio
mrr_per_seat
arr_per_seat
has_zero_revenue
is_paid_subscription
account_ticket_pressure_per_subscription
account_escalation_rate
account_ticket_per_seat
account_refund_per_churn_event
account_upgrade_before_churn_rate
account_downgrade_before_churn_rate
account_reactivation_related_churn_rate
has_account_churn_history
usage_minus_errors
support_friction_interaction
trial_low_value_interaction

--- Summary Statistics for Created Features ---


,usage_count_per_event,usage_duration_per_event,error_rate_per_usage,feature_breadth_ratio,mrr_per_seat,arr_per_seat,has_zero_revenue,is_paid_subscription,account_ticket_pressure_per_subscription,account_escalation_rate,account_ticket_per_seat,account_refund_per_churn_event,account_upgrade_before_churn_rate,account_downgrade_before_churn_rate,account_reactivation_related_churn_rate,has_account_churn_history,usage_minus_errors,support_friction_interaction,trial_low_value_interaction
count,4967.000000,4967.000000,4967.000000,4967.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,4928.000000,5000.000000,3528.000000,3528.000000,3528.000000,3528.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,10.005137,3039.364071,0.058290,0.951737,76.739600,920.875200,0.155600,0.844400,3.979600,0.048895,0.273027,13.848776,0.208749,0.083385,0.087632,0.705600,47.283600,11.261400,0.155600
std,1.564209,1029.367189,0.056686,0.092049,79.840247,958.082967,0.362512,0.362512,1.846474,0.117346,0.444912,32.543245,0.349614,0.236242,0.229260,0.455818,22.366753,12.410233,0.362512
min,3.000000,48.000000,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9.000000,2384.888889,0.017241,0.919872,19.000000,228.000000,0.000000,1.000000,3.000000,0.000000,0.083333,0.000000,0.000000,0.000000,0.000000,0.000000,31.000000,2.000000,0.000000
50%,10.000000,2997.250000,0.047619,1.000000,49.000000,588.000000,0.000000,1.000000,4.000000,0.000000,0.153846,0.000000,0.000000,0.000000,0.000000,1.000000,45.000000,8.000000,0.000000
75%,11.000000,3652.166667,0.085366,1.000000,199.000000,2388.000000,0.000000,1.000000,5.000000,0.000000,0.285714,11.130000,0.333333,0.000000,0.000000,1.000000,62.000000,16.000000,0.000000
max,20.000000,10146.000000,0.666667,1.000000,199.000000,2388.000000,1.000000,1.000000,11.000000,1.000000,8.000000,277.540000,1.000000,1.000000,1.000000,1.000000,141.000000,91.000000,1.000000



--- Preview of Subscription Feature Table ---


,subscription_id,account_id,target_churn_subscription,usage_count_per_event,usage_duration_per_event,error_rate_per_usage,feature_breadth_ratio,mrr_per_seat,arr_per_seat,has_zero_revenue,...,account_escalation_rate,account_ticket_per_seat,account_refund_per_churn_event,account_upgrade_before_churn_rate,account_downgrade_before_churn_rate,account_reactivation_related_churn_rate,has_account_churn_history,usage_minus_errors,support_friction_interaction,trial_low_value_interaction
0,S-8cec59,A-3c1a3f,1,10.500000,4403.000000,0.031746,1.0,199.0,2388.0,0,...,0.0,0.285714,NaN,NaN,NaN,NaN,0,61.0,8.0,0.0
1,S-0f6f44,A-9b9fe9,0,9.000000,1704.750000,0.166667,1.0,49.0,588.0,0,...,0.0,0.235294,13.960000,0.000000,0.0,0.000000,1,30.0,24.0,0.0
2,S-51c0d1,A-659280,0,8.333333,4625.666667,0.080000,1.0,0.0,0.0,1,...,0.0,0.064516,13.875000,0.000000,0.0,0.000000,1,23.0,8.0,1.0
3,S-f81687,A-e7a1e2,1,9.285714,3277.571429,0.076923,1.0,199.0,2388.0,0,...,0.0,0.200000,2.066667,0.333333,0.0,0.333333,1,60.0,5.0,0.0
4,S-cff5a2,A-ba6516,0,NaN,NaN,NaN,NaN,199.0,2388.0,0,...,0.0,0.222222,0.000000,0.000000,0.0,0.000000,1,0.0,0.0,0.0


In [87]:
subscription_feature_df.to_csv("subscription_feature_engineering.csv", index=False)
print("subscription_feature_engineering.csv exported successfully.")

subscription_feature_engineering.csv exported successfully.


### 12.14.2 Interpretation — Subscription-Level Feature Engineering

This step successfully created the first **subscription-level engineered feature set**, expanding the analytical value of the subscription table beyond its raw commercial and usage aggregates. A total of **19 new engineered features** were added, covering behavioral engagement, product friction, commercial value, support context, and account-level churn-history context.

#### 1. The subscription-level feature set is now richer and more behaviorally informative  
The new variables improve the analytical representation of each subscription by introducing measures such as:

- `usage_count_per_event`
- `usage_duration_per_event`
- `error_rate_per_usage`
- `feature_breadth_ratio`
- `mrr_per_seat`
- `arr_per_seat`

This is important because it shifts the dataset away from simple totals and toward more normalized behavioral indicators. At the subscription level, this is especially useful because different subscriptions under the same account may vary substantially in size, monetization, and engagement intensity.

---

#### 2. Product usage appears well distributed, but not uniform  
The behavioral features suggest that usage at the subscription level is active and measurable:

- the average `usage_count_per_event` is approximately **10**
- the average `usage_duration_per_event` is around **3,039 seconds**
- the average `error_rate_per_usage` is about **5.8%**
- `feature_breadth_ratio` is very close to **1.0** on average

These values suggest that many subscriptions interact with the product consistently, and that the subscription-level representation is capturing real behavioral variation rather than empty or overly sparse records. At the same time, the distributions still show meaningful spread, which is useful for later churn modeling.

---

#### 3. Commercial features remain highly informative  
The commercial engineered variables are especially valuable at this level:

- `mrr_per_seat`
- `arr_per_seat`
- `has_zero_revenue`
- `is_paid_subscription`

These features allow the project to distinguish clearly between paid and zero-revenue subscriptions, and to evaluate commercial value relative to subscription size. This is analytically stronger than the account-level version because revenue and trial behavior often differ across subscriptions within the same customer account.

The summary confirms this heterogeneity: the mean `mrr_per_seat` is about **76.7**, but the distribution ranges from **0** to **199**, showing that subscriptions differ substantially in monetization intensity.

---

#### 4. Support and churn context are now attached at subscription level  
The feature set also adds account-level support and churn-history context to each subscription, including:

- `account_ticket_pressure_per_subscription`
- `account_escalation_rate`
- `account_ticket_per_seat`
- `account_refund_per_churn_event`
- `account_upgrade_before_churn_rate`
- `account_downgrade_before_churn_rate`
- `account_reactivation_related_churn_rate`
- `has_account_churn_history`

This is a useful compromise analytically: even though support tickets and churn events are not stored directly at the subscription level, attaching account-level context allows each subscription to inherit a realistic business environment. This may help modeling later by giving each subscription both its own direct behavior and the broader risk context of the account it belongs to.

---

#### 5. Missing values are limited and structurally expected  
A few of the new features contain missing values, particularly:

- `usage_count_per_event`
- `usage_duration_per_event`
- `error_rate_per_usage`
- `feature_breadth_ratio`
- `account_refund_per_churn_event`
- `account_upgrade_before_churn_rate`
- `account_downgrade_before_churn_rate`
- `account_reactivation_related_churn_rate`

These missing values are not necessarily quality problems. They are mostly business-expected:

- subscriptions with no usage activity will naturally have undefined usage ratios
- subscriptions belonging to accounts with no churn history will naturally have undefined churn-history rates

This means the feature table remains analytically sound, but these variables will require explicit handling later through imputation or controlled missing-value treatment.

---

### Final Conclusion for Step 12.14.2

This step successfully transformed the raw subscription-level analytical table into a **much more informative subscription-level feature table**. The new engineered variables provide stronger visibility into:

- per-subscription engagement
- per-subscription value
- product friction
- support burden context
- account-level instability signals

The most important takeaway is:

> **The subscription-level pipeline now has a richer and potentially more predictive feature base than the earlier account-level pipeline, because it preserves variation that was previously averaged away.**

This means the project is now ready for the next stage:


**12.14.3 Feature Quality Review at Subscription Level**

where the new feature table should be checked for missingness, validity, skewness, and redundancy before moving into subscription-level modeling.

## 12.14.3 Feature Quality Review at Subscription Level

After creating the subscription-level engineered feature set, the next step is to evaluate its overall quality before moving into modeling.

The purpose of this step is to verify that the engineered subscription-level variables are analytically reliable and ready for downstream churn modeling. In particular, this review checks for:

- missing values in the newly created subscription-level features
- infinite values or division-related artifacts
- invalid or suspicious ranges
- constant or near-constant features with limited analytical value
- duplicated columns or structural inconsistencies

This step is important because subscription-level feature engineering introduced several ratio-based and context-based variables, and these may contain undefined values or sparse patterns that require review before the next modeling stage.

In [88]:
# =========================================================
# 12.14.3 Feature Quality Review at Subscription Level
# =========================================================

import pandas as pd
import numpy as np

subscription_feature_df = subscription_feature_df.copy()

print("=== 12.14.3 Feature Quality Review at Subscription Level ===")

# ---------------------------------------------------------
# A) List of engineered subscription-level features
# ---------------------------------------------------------
subscription_engineered_features = [
    "usage_count_per_event",
    "usage_duration_per_event",
    "error_rate_per_usage",
    "feature_breadth_ratio",
    "mrr_per_seat",
    "arr_per_seat",
    "has_zero_revenue",
    "is_paid_subscription",
    "account_ticket_pressure_per_subscription",
    "account_escalation_rate",
    "account_ticket_per_seat",
    "account_refund_per_churn_event",
    "account_upgrade_before_churn_rate",
    "account_downgrade_before_churn_rate",
    "account_reactivation_related_churn_rate",
    "has_account_churn_history",
    "usage_minus_errors",
    "support_friction_interaction",
    "trial_low_value_interaction"
]

subscription_engineered_features = [
    col for col in subscription_engineered_features if col in subscription_feature_df.columns
]

print("\nNumber of engineered subscription-level features found:", len(subscription_engineered_features))

# ---------------------------------------------------------
# B) Missing values check
# ---------------------------------------------------------
print("\n--- Missing Values in Subscription-Level Engineered Features ---")
missing_summary = subscription_feature_df[subscription_engineered_features].isna().sum().sort_values(ascending=False)
missing_nonzero = missing_summary[missing_summary > 0]

if len(missing_nonzero) > 0:
    display(missing_nonzero)
else:
    print("No missing values found.")

# ---------------------------------------------------------
# C) Infinite values check
# ---------------------------------------------------------
print("\n--- Infinite Values Check ---")
infinite_summary = pd.Series({
    col: np.isinf(pd.to_numeric(subscription_feature_df[col], errors="coerce")).sum()
    for col in subscription_engineered_features
})
infinite_nonzero = infinite_summary[infinite_summary > 0]

if len(infinite_nonzero) > 0:
    display(infinite_nonzero)
else:
    print("No infinite values found.")

# ---------------------------------------------------------
# D) Descriptive range review
# ---------------------------------------------------------
print("\n--- Descriptive Range Review ---")
display(subscription_feature_df[subscription_engineered_features].describe(include="all").T)

# ---------------------------------------------------------
# E) Constant / near-constant feature detection
# ---------------------------------------------------------
print("\n--- Constant / Near-Constant Features ---")

nunique_summary = subscription_feature_df[subscription_engineered_features].nunique(dropna=False).sort_values()
display(nunique_summary)

constant_features = nunique_summary[nunique_summary == 1].index.tolist()
near_constant_features = nunique_summary[nunique_summary <= 2].index.tolist()

print("\nConstant features:")
print(constant_features if constant_features else "None")

print("\nNear-constant features (2 or fewer unique values):")
print(near_constant_features if near_constant_features else "None")

# ---------------------------------------------------------
# F) Duplicate column name check
# ---------------------------------------------------------
print("\n--- Duplicate Column Name Check ---")
duplicate_columns = subscription_feature_df.columns[subscription_feature_df.columns.duplicated()].tolist()
print(duplicate_columns if duplicate_columns else "No duplicated column names found.")

# ---------------------------------------------------------
# G) Bounded feature validity checks
# ---------------------------------------------------------
print("\n--- Bounded Feature Validity Checks ---")

bounded_checks = {}

if "error_rate_per_usage" in subscription_feature_df.columns:
    bounded_checks["error_rate_per_usage < 0"] = (subscription_feature_df["error_rate_per_usage"] < 0).sum(skipna=True)
    bounded_checks["error_rate_per_usage > 1"] = (subscription_feature_df["error_rate_per_usage"] > 1).sum(skipna=True)

if "account_escalation_rate" in subscription_feature_df.columns:
    bounded_checks["account_escalation_rate < 0"] = (subscription_feature_df["account_escalation_rate"] < 0).sum(skipna=True)
    bounded_checks["account_escalation_rate > 1"] = (subscription_feature_df["account_escalation_rate"] > 1).sum(skipna=True)

if "account_upgrade_before_churn_rate" in subscription_feature_df.columns:
    bounded_checks["account_upgrade_before_churn_rate < 0"] = (subscription_feature_df["account_upgrade_before_churn_rate"] < 0).sum(skipna=True)
    bounded_checks["account_upgrade_before_churn_rate > 1"] = (subscription_feature_df["account_upgrade_before_churn_rate"] > 1).sum(skipna=True)

if "account_downgrade_before_churn_rate" in subscription_feature_df.columns:
    bounded_checks["account_downgrade_before_churn_rate < 0"] = (subscription_feature_df["account_downgrade_before_churn_rate"] < 0).sum(skipna=True)
    bounded_checks["account_downgrade_before_churn_rate > 1"] = (subscription_feature_df["account_downgrade_before_churn_rate"] > 1).sum(skipna=True)

if "account_reactivation_related_churn_rate" in subscription_feature_df.columns:
    bounded_checks["account_reactivation_related_churn_rate < 0"] = (subscription_feature_df["account_reactivation_related_churn_rate"] < 0).sum(skipna=True)
    bounded_checks["account_reactivation_related_churn_rate > 1"] = (subscription_feature_df["account_reactivation_related_churn_rate"] > 1).sum(skipna=True)

bounded_checks_series = pd.Series(bounded_checks)
display(bounded_checks_series)

# ---------------------------------------------------------
# H) Final quick quality snapshot
# ---------------------------------------------------------
print("\n--- Final Subscription Feature Quality Snapshot ---")

quality_snapshot = pd.DataFrame({
    "missing_count": subscription_feature_df[subscription_engineered_features].isna().sum(),
    "nunique": subscription_feature_df[subscription_engineered_features].nunique(dropna=False),
    "min": subscription_feature_df[subscription_engineered_features].min(numeric_only=False),
    "max": subscription_feature_df[subscription_engineered_features].max(numeric_only=False),
})

display(quality_snapshot.sort_values("missing_count", ascending=False))


=== 12.14.3 Feature Quality Review at Subscription Level ===

Number of engineered subscription-level features found: 19

--- Missing Values in Subscription-Level Engineered Features ---


account_reactivation_related_churn_rate    1472
account_downgrade_before_churn_rate        1472
account_upgrade_before_churn_rate          1472
account_refund_per_churn_event             1472
account_escalation_rate                      72
usage_duration_per_event                     33
usage_count_per_event                        33
feature_breadth_ratio                        33
error_rate_per_usage                         33
dtype: int64


--- Infinite Values Check ---
No infinite values found.

--- Descriptive Range Review ---


,count,mean,std,min,25%,50%,75%,max
usage_count_per_event,4967.0,10.005137,1.564209,3.000000,9.000000,10.000000,11.000000,20.000000
usage_duration_per_event,4967.0,3039.364071,1029.367189,48.000000,2384.888889,2997.250000,3652.166667,10146.000000
error_rate_per_usage,4967.0,0.058290,0.056686,0.000000,0.017241,0.047619,0.085366,0.666667
feature_breadth_ratio,4967.0,0.951737,0.092049,0.333333,0.919872,1.000000,1.000000,1.000000
mrr_per_seat,5000.0,76.739600,79.840247,0.000000,19.000000,49.000000,199.000000,199.000000
arr_per_seat,5000.0,920.875200,958.082967,0.000000,228.000000,588.000000,2388.000000,2388.000000
has_zero_revenue,5000.0,0.155600,0.362512,0.000000,0.000000,0.000000,0.000000,1.000000
is_paid_subscription,5000.0,0.844400,0.362512,0.000000,1.000000,1.000000,1.000000,1.000000
account_ticket_pressure_per_subscription,5000.0,3.979600,1.846474,0.000000,3.000000,4.000000,5.000000,11.000000
account_escalation_rate,4928.0,0.048895,0.117346,0.000000,0.000000,0.000000,0.000000,1.000000



--- Constant / Near-Constant Features ---


trial_low_value_interaction                    2
has_account_churn_history                      2
has_zero_revenue                               2
is_paid_subscription                           2
mrr_per_seat                                   4
arr_per_seat                                   4
account_reactivation_related_churn_rate        7
account_downgrade_before_churn_rate            8
account_upgrade_before_churn_rate              8
account_escalation_rate                       12
account_ticket_pressure_per_subscription      12
feature_breadth_ratio                         26
support_friction_interaction                  57
account_refund_per_churn_event               122
usage_minus_errors                           129
usage_count_per_event                        240
account_ticket_per_seat                      422
error_rate_per_usage                         549
usage_duration_per_event                    4630
dtype: int64


Constant features:
None

Near-constant features (2 or fewer unique values):
['trial_low_value_interaction', 'has_account_churn_history', 'has_zero_revenue', 'is_paid_subscription']

--- Duplicate Column Name Check ---
No duplicated column names found.

--- Bounded Feature Validity Checks ---


error_rate_per_usage < 0                       0
error_rate_per_usage > 1                       0
account_escalation_rate < 0                    0
account_escalation_rate > 1                    0
account_upgrade_before_churn_rate < 0          0
account_upgrade_before_churn_rate > 1          0
account_downgrade_before_churn_rate < 0        0
account_downgrade_before_churn_rate > 1        0
account_reactivation_related_churn_rate < 0    0
account_reactivation_related_churn_rate > 1    0
dtype: int64


--- Final Subscription Feature Quality Snapshot ---


,missing_count,nunique,min,max
account_reactivation_related_churn_rate,1472,7,0.000000,1.000000
account_downgrade_before_churn_rate,1472,8,0.000000,1.000000
account_upgrade_before_churn_rate,1472,8,0.000000,1.000000
account_refund_per_churn_event,1472,122,0.000000,277.540000
account_escalation_rate,72,12,0.000000,1.000000
usage_duration_per_event,33,4630,48.000000,10146.000000
usage_count_per_event,33,240,3.000000,20.000000
feature_breadth_ratio,33,26,0.333333,1.000000
error_rate_per_usage,33,549,0.000000,0.666667
is_paid_subscription,0,2,0.000000,1.000000


## 12.14.4 Prepare the Subscription-Level Feature Set for Modeling

After reviewing the quality of the subscription-level engineered feature table, the next step is to prepare the final modeling structure.

The purpose of this step is to define which variables will be used as candidate predictors for subscription-level churn modeling and which variables should be excluded. This includes removing identifiers, direct target-like fields, and columns that are not suitable for direct baseline modeling in their current form.

This step separates the dataset into:

- the subscription-level churn target
- candidate numerical features
- candidate categorical features
- excluded columns

The objective is to produce a clean and well-defined subscription-level feature matrix that can be used in the next modeling stage.

In [89]:
# =========================================================
# 12.14.4 Prepare the Subscription-Level Feature Set for Modeling
# =========================================================

import pandas as pd
import numpy as np

sub_model_df = subscription_feature_df.copy()

print("=== 12.14.4 Prepare the Subscription-Level Feature Set for Modeling ===")

# ---------------------------------------------------------
# A) Define target
# ---------------------------------------------------------
target_col_sub = "target_churn_subscription"

# ---------------------------------------------------------
# B) Define columns to exclude
# ---------------------------------------------------------
exclude_cols_sub = [
    # identifiers
    "subscription_id",
    "account_id",

    # temporal fields
    "start_date",
    "end_date",
    "signup_date",

    # direct target-like / raw churn fields
    "churn_flag",
    "churn_flag_num",
    "target_churn_subscription",
]

# keep only existing columns
exclude_cols_sub = [col for col in exclude_cols_sub if col in sub_model_df.columns]

# ---------------------------------------------------------
# C) Separate numeric and categorical features
# ---------------------------------------------------------
numeric_cols_sub = sub_model_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols_sub = sub_model_df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

numeric_features_sub = [col for col in numeric_cols_sub if col not in exclude_cols_sub]
categorical_features_sub = [col for col in categorical_cols_sub if col not in exclude_cols_sub]

# ---------------------------------------------------------
# D) Review near-constant numeric / binary features
# ---------------------------------------------------------
near_constant_numeric_sub = [
    col for col in numeric_features_sub
    if sub_model_df[col].nunique(dropna=False) <= 2
]

print("\n--- Near-Constant Numeric/Binary Features ---")
print(near_constant_numeric_sub if near_constant_numeric_sub else "None")

# ---------------------------------------------------------
# E) Missing value profile for candidate features
# ---------------------------------------------------------
print("\n--- Missing Values in Candidate Features ---")

candidate_features_sub = numeric_features_sub + categorical_features_sub
missing_candidates_sub = sub_model_df[candidate_features_sub].isna().sum().sort_values(ascending=False)
missing_candidates_sub_nonzero = missing_candidates_sub[missing_candidates_sub > 0]

if len(missing_candidates_sub_nonzero) > 0:
    display(missing_candidates_sub_nonzero)
else:
    print("No missing values found.")

# ---------------------------------------------------------
# F) Final feature group summary
# ---------------------------------------------------------
baseline_numeric_features_sub = [col for col in numeric_features_sub if col != target_col_sub]
baseline_categorical_features_sub = categorical_features_sub.copy()

print("\n--- Final Feature Group Summary ---")
print("Target column:", target_col_sub)
print("Number of numeric candidate features:", len(numeric_features_sub))
print("Number of categorical candidate features:", len(categorical_features_sub))
print("Number of baseline numeric features:", len(baseline_numeric_features_sub))
print("Number of baseline categorical features:", len(baseline_categorical_features_sub))

# ---------------------------------------------------------
# G) Preview feature lists
# ---------------------------------------------------------
print("\n--- Baseline Numeric Features ---")
for col in baseline_numeric_features_sub:
    print(col)

print("\n--- Baseline Categorical Features ---")
for col in baseline_categorical_features_sub:
    print(col)

print("\n--- Excluded Columns ---")
for col in exclude_cols_sub:
    print(col)

# ---------------------------------------------------------
# H) Build X / y objects
# ---------------------------------------------------------
X_sub_numeric = sub_model_df[baseline_numeric_features_sub].copy()
y_sub = pd.to_numeric(sub_model_df[target_col_sub], errors="coerce").fillna(0).astype(int)

print("\n--- Subscription-Level Modeling Shapes ---")
print("X_sub_numeric shape:", X_sub_numeric.shape)
print("y_sub shape:", y_sub.shape)

print("\nPreview of X_sub_numeric:")
display(X_sub_numeric.head())

print("\nPreview of y_sub:")
display(y_sub.head())

=== 12.14.4 Prepare the Subscription-Level Feature Set for Modeling ===

--- Near-Constant Numeric/Binary Features ---
['is_trial', 'upgrade_flag', 'downgrade_flag', 'auto_renew_flag', 'is_trial_num', 'upgrade_flag_num', 'downgrade_flag_num', 'auto_renew_flag_num', 'has_zero_revenue', 'is_paid_subscription', 'has_account_churn_history', 'trial_low_value_interaction']

--- Missing Values in Candidate Features ---


account_reactivation_related_churn_rate    1472
account_downgrade_before_churn_rate        1472
account_upgrade_before_churn_rate          1472
account_refund_per_churn_event             1472
account_escalation_rate                      72
usage_count_per_event                        33
usage_duration_per_event                     33
error_rate_per_usage                         33
feature_breadth_ratio                        33
dtype: int64


--- Final Feature Group Summary ---
Target column: target_churn_subscription
Number of numeric candidate features: 52
Number of categorical candidate features: 5
Number of baseline numeric features: 52
Number of baseline categorical features: 5

--- Baseline Numeric Features ---
seats
mrr_amount
arr_amount
is_trial
upgrade_flag
downgrade_flag
auto_renew_flag
is_trial_num
upgrade_flag_num
downgrade_flag_num
auto_renew_flag_num
seats_num
mrr_amount_num
arr_amount_num
usage_event_count
total_usage_count
total_usage_duration_secs
total_error_count
unique_features_used
beta_feature_usage_rate
avg_usage_count
avg_usage_duration_secs
avg_error_count
account_ticket_count
account_escalated_ticket_count
account_avg_resolution_time_hours
account_avg_first_response_time_minutes
account_avg_satisfaction_score
account_churn_event_count
account_total_refund_amount_usd
account_reactivation_related_churn_count
account_preceding_upgrade_churn_count
account_preceding_downgrade_churn_count
usage_count_pe

,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,auto_renew_flag,is_trial_num,upgrade_flag_num,downgrade_flag_num,...,account_escalation_rate,account_ticket_per_seat,account_refund_per_churn_event,account_upgrade_before_churn_rate,account_downgrade_before_churn_rate,account_reactivation_related_churn_rate,has_account_churn_history,usage_minus_errors,support_friction_interaction,trial_low_value_interaction
0,14,2786,33432,0,0,0,1,0.0,0.0,0.0,...,0.0,0.285714,NaN,NaN,NaN,NaN,0,61.0,8.0,0.0
1,17,833,9996,0,0,0,1,0.0,0.0,0.0,...,0.0,0.235294,13.960000,0.000000,0.0,0.000000,1,30.0,24.0,0.0
2,62,0,0,1,1,0,0,1.0,1.0,0.0,...,0.0,0.064516,13.875000,0.000000,0.0,0.000000,1,23.0,8.0,1.0
3,5,995,11940,0,0,0,1,0.0,0.0,0.0,...,0.0,0.200000,2.066667,0.333333,0.0,0.333333,1,60.0,5.0,0.0
4,27,5373,64476,0,0,0,1,0.0,0.0,0.0,...,0.0,0.222222,0.000000,0.000000,0.0,0.000000,1,0.0,0.0,0.0



Preview of y_sub:


0    1
1    0
2    0
3    1
4    0
Name: target_churn_subscription, dtype: int32

### 12.14.4 Interpretation — Prepare the Subscription-Level Feature Set for Modeling

This step successfully prepared the subscription-level feature table for downstream churn modeling. The resulting structure is clear and well defined, with:

- **`target_churn_subscription`** as the modeling target
- **52 numeric candidate features**
- **5 categorical candidate features**
- a final numeric feature matrix of shape **(5,000 × 52)**

This confirms that the subscription-level pipeline now has a substantially larger and more detailed modeling base than the earlier account-level version.

#### 1. The subscription-level modeling structure is now clearly defined
The modeling input is correctly aligned at the **subscription level**, where each row represents one subscription and the target captures whether that subscription churned. This is analytically important because subscription-level churn is often more behaviorally specific than account-level churn and may therefore be easier to predict.

#### 2. The feature space is broad and multi-domain
The candidate numeric features cover multiple analytical domains, including:

- **commercial variables** such as revenue, seats, trial exposure, upgrade/downgrade behavior, and billing structure
- **usage variables** such as usage volume, usage duration, error activity, and feature breadth
- **support-context variables** such as account-level support burden, escalation behavior, and satisfaction context
- **churn-context variables** inherited from the parent account, such as refund and pre-churn transition rates

This means the subscription-level model will not rely on one narrow class of predictors. Instead, it is positioned to test churn as a multi-factor outcome using both direct subscription behavior and account-level context.

#### 3. Some binary and raw status features are near-constant or structurally simple
Several fields were identified as near-constant or binary, including:

- `is_trial`
- `upgrade_flag`
- `downgrade_flag`
- `auto_renew_flag`
- `has_zero_revenue`
- `is_paid_subscription`
- `has_account_churn_history`
- `trial_low_value_interaction`

These variables are not invalid, but they are structurally simple and may not contribute strong standalone signal. Their value, if any, will likely depend on interaction with other richer variables rather than isolated predictive strength.

#### 4. Missing values remain limited and explainable
The remaining missing values are concentrated in:

- account-level churn-context rate features
- account-level escalation rate
- usage-ratio features such as `usage_count_per_event`, `usage_duration_per_event`, `error_rate_per_usage`, and `feature_breadth_ratio`

These missing values are still **business-expected**, not technical failures. For example:

- subscriptions tied to accounts with no churn history cannot have churn-context rates
- subscriptions with no usage records cannot produce usage-based ratios
- subscriptions tied to accounts without support escalation cannot produce escalation-rate measures

This means that modeling can still proceed safely, provided that missing-value handling is applied consistently in the next step.

### Final Conclusion for Step 12.14.4

This step confirms that the subscription-level feature table is now **ready for modeling preparation and baseline churn prediction**. The structure is richer and more granular than the earlier account-level pipeline, and the candidate feature set preserves both direct subscription behavior and broader account context.

The most important takeaway is:

> **The subscription-level pipeline now has a strong structural foundation for testing whether churn becomes more predictable when modeled at the subscription level rather than at the aggregated account level.**

## 12.14.5 Build the First Subscription-Level Baseline Model

This step builds the first baseline churn model at the subscription level using Logistic Regression.

The purpose of this model is to test whether the subscription-level analytical unit produces stronger churn signal than the earlier account-level pipeline. Since subscription-level churn may preserve more specific commercial and behavioral variation, it is important to evaluate whether this more granular representation improves predictive modeling performance.

In this step, the workflow will:

- use the subscription-level numeric feature matrix
- handle missing values through median imputation
- standardize numeric variables
- split the data into training and testing subsets
- fit a Logistic Regression classifier
- evaluate model performance using classification metrics and ROC-AUC
- inspect coefficient strength to identify the most influential subscription-level features

This step provides the first direct benchmark for subscription-level churn prediction.
``

In [90]:
# =========================================================
# 12.14.5 Build the First Subscription-Level Baseline Model
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=== 12.14.5 Build the First Subscription-Level Baseline Model ===")

# ---------------------------------------------------------
# A) Prepare X and y
# ---------------------------------------------------------
X_sub_final = X_sub_numeric.copy()
y_sub_final = y_sub.copy()

# Safety checks
X_sub_final = X_sub_final.loc[:, ~X_sub_final.columns.duplicated()].copy()
X_sub_final = X_sub_final.apply(pd.to_numeric, errors="coerce")
y_sub_final = pd.to_numeric(y_sub_final, errors="coerce").fillna(0).astype(int)

print("\n--- Subscription-Level Input Shapes ---")
print("X_sub_final shape:", X_sub_final.shape)
print("y_sub_final shape:", y_sub_final.shape)

print("\nTarget distribution:")
print(y_sub_final.value_counts(dropna=False))

# ---------------------------------------------------------
# B) Train-test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_sub_final,
    y_sub_final,
    test_size=0.20,
    random_state=42,
    stratify=y_sub_final
)

print("\n--- Train/Test Split ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# ---------------------------------------------------------
# C) Build baseline logistic regression pipeline
# ---------------------------------------------------------
sub_logreg_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# ---------------------------------------------------------
# D) Fit model
# ---------------------------------------------------------
sub_logreg_model.fit(X_train, y_train)

# ---------------------------------------------------------
# E) Predictions
# ---------------------------------------------------------
y_pred = sub_logreg_model.predict(X_test)
y_pred_proba = sub_logreg_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# F) Evaluation metrics
# ---------------------------------------------------------
print("\n--- Subscription-Level Baseline Classification Metrics ---")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_summary)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
display(cm_df)

# ---------------------------------------------------------
# G) Coefficient importance
# ---------------------------------------------------------
print("\n--- Subscription-Level Logistic Regression Coefficients ---")

log_reg_model = sub_logreg_model.named_steps["log_reg"]

coef_df = pd.DataFrame({
    "feature": X_sub_final.columns,
    "coefficient": log_reg_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 15 Features by Absolute Coefficient Strength:")
display(coef_df.head(15))

print("\nTop Positive Coefficients (associated with higher churn probability):")
display(coef_df.sort_values("coefficient", ascending=False).head(10))

print("\nTop Negative Coefficients (associated with lower churn probability):")
display(coef_df.sort_values("coefficient", ascending=True).head(10))

=== 12.14.5 Build the First Subscription-Level Baseline Model ===

--- Subscription-Level Input Shapes ---
X_sub_final shape: (5000, 52)
y_sub_final shape: (5000,)

Target distribution:
target_churn_subscription
0    4514
1     486
Name: count, dtype: int64

--- Train/Test Split ---
X_train shape: (4000, 52)
X_test shape: (1000, 52)
y_train distribution:
target_churn_subscription
0    3611
1     389
Name: count, dtype: int64
y_test distribution:
target_churn_subscription
0    903
1     97
Name: count, dtype: int64

--- Subscription-Level Baseline Classification Metrics ---


,Metric,Value
0,Accuracy,0.551000
1,Precision,0.101810
2,Recall,0.463918
3,F1 Score,0.166976
4,ROC-AUC,0.489183



Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.56      0.69       903
           1       0.10      0.46      0.17        97

    accuracy                           0.55      1000
   macro avg       0.50      0.51      0.43      1000
weighted avg       0.83      0.55      0.64      1000


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,506,397
Actual 1,52,45



--- Subscription-Level Logistic Regression Coefficients ---

Top 15 Features by Absolute Coefficient Strength:


,feature,coefficient,abs_coefficient
14,usage_event_count,-0.956881,0.956881
18,unique_features_used,0.788953,0.788953
17,total_error_count,0.468602,0.468602
46,account_downgrade_before_churn_rate,0.344933,0.344933
35,error_rate_per_usage,-0.302379,0.302379
31,account_preceding_upgrade_churn_count,-0.238543,0.238543
32,account_preceding_downgrade_churn_count,-0.236778,0.236778
45,account_upgrade_before_churn_rate,0.234827,0.234827
50,support_friction_interaction,-0.232180,0.232180
36,feature_breadth_ratio,-0.200355,0.200355



Top Positive Coefficients (associated with higher churn probability):


,feature,coefficient,abs_coefficient
18,unique_features_used,0.788953,0.788953
17,total_error_count,0.468602,0.468602
46,account_downgrade_before_churn_rate,0.344933,0.344933
45,account_upgrade_before_churn_rate,0.234827,0.234827
30,account_reactivation_related_churn_count,0.192771,0.192771
16,total_usage_duration_secs,0.190393,0.190393
27,account_avg_satisfaction_score,0.103456,0.103456
20,avg_usage_count,0.093754,0.093754
29,account_total_refund_amount_usd,0.089183,0.089183
28,account_churn_event_count,0.088443,0.088443



Top Negative Coefficients (associated with lower churn probability):


,feature,coefficient,abs_coefficient
14,usage_event_count,-0.956881,0.956881
35,error_rate_per_usage,-0.302379,0.302379
31,account_preceding_upgrade_churn_count,-0.238543,0.238543
32,account_preceding_downgrade_churn_count,-0.236778,0.236778
50,support_friction_interaction,-0.232180,0.232180
36,feature_breadth_ratio,-0.200355,0.200355
47,account_reactivation_related_churn_rate,-0.192916,0.192916
44,account_refund_per_churn_event,-0.144502,0.144502
49,usage_minus_errors,-0.139440,0.139440
34,usage_duration_per_event,-0.100978,0.100978


### 12.14.5 Interpretation — Build the First Subscription-Level Baseline Model

This first subscription-level Logistic Regression model **does not produce a strong predictive improvement** over the earlier account-level models. Although the analytical unit is now more granular, the model still shows **weak overall discrimination**.

#### 1. Overall predictive performance remains weak
The baseline subscription-level model achieved:

- **Accuracy = 0.551**
- **Precision = 0.102**
- **Recall = 0.464**
- **F1 Score = 0.167**
- **ROC-AUC = 0.489**

The most important metric here is **ROC-AUC = 0.489**, which is effectively **below or around random discrimination**. This means that, in its current form, the model is still unable to separate churned and non-churned subscriptions reliably.

---

#### 2. The model detects some churn, but with many false positives
The confusion matrix shows:

- **45 true positives**
- **52 false negatives**
- **397 false positives**
- **506 true negatives**

This means the model is able to capture **some churned subscriptions** (**recall = 0.464**), but it does so at the cost of a very large number of incorrect churn predictions. The **precision of only 0.102** confirms that most subscriptions predicted as churned are actually non-churned.

So although the subscription-level model is sensitive to churn to some extent, it is still **too noisy** for reliable churn targeting.

---

#### 3. Moving from account-level to subscription-level did not solve the problem by itself
One of the main goals of this alternative pipeline was to test whether churn becomes easier to predict when modeled at the subscription level rather than the account level.

The result suggests that:

> **Changing the analytical unit alone was not enough to unlock strong predictive performance.**

This means that the weak signal was **not caused only by account-level aggregation**. Some information may indeed be preserved better at the subscription level, but the overall churn signal still remains too weak or too complex to be captured effectively by a simple linear model.

---

#### 4. The coefficient structure is still analytically useful
Even though performance is weak, the coefficient ranking still gives useful directional insight.

##### Positive coefficients (associated with higher churn probability) include:
- `unique_features_used`
- `total_error_count`
- `account_downgrade_before_churn_rate`
- `account_upgrade_before_churn_rate`
- `account_reactivation_related_churn_count`
- `total_usage_duration_secs`

##### Negative coefficients (associated with lower churn probability) include:
- `usage_event_count`
- `error_rate_per_usage`
- `account_preceding_upgrade_churn_count`
- `account_preceding_downgrade_churn_count`
- `support_friction_interaction`
- `feature_breadth_ratio`

These results suggest that the model is still drawing signal from the same broad domains that appeared earlier:

- **usage behavior**
- **error / friction**
- **support context**
- **commercial or churn-history context**

So while prediction quality is weak, the model still confirms that these are the most relevant analytical axes in the churn problem.

---

### Final Conclusion for Step 12.14.5

The first subscription-level baseline model shows that **subscription-level modeling alone does not produce a strong churn classifier** in the current setup. The model is able to detect some churned subscriptions, but its precision is very low and its ROC-AUC indicates that class separation remains weak.

The most important takeaway is:

> **The churn problem remains difficult even at the subscription level, at least with the current feature set and a simple Logistic Regression classifier.**

This means the next logical step is to move beyond a linear baseline and test a more flexible subscription-level model, such as:

## **12.14.6 Build a Subscription-Level Tree-Based Model (Random Forest / XGBoost)**

That step will help determine whether the remaining churn signal is nonlinear or interaction-driven and may therefore be better captured by a more flexible classifier.

## 12.14.6 Build a Subscription-Level Tree-Based Model (Random Forest)

After testing the first subscription-level Logistic Regression baseline, the next step is to evaluate a more flexible tree-based model using Random Forest.

The purpose of this step is to test whether subscription-level churn is driven by nonlinear relationships or feature interactions that were not captured effectively by the earlier linear baseline.

In this step, the workflow will:

- use the subscription-level numeric feature matrix
- handle missing values through median imputation
- split the data into training and testing subsets
- fit a Random Forest classifier
- evaluate model performance using classification metrics and ROC-AUC
- inspect feature importance to identify the strongest subscription-level predictive signals

This step provides a stronger nonlinear benchmark for the subscription-level churn pipeline.

In [91]:
# =========================================================
# 12.14.6 Build a Subscription-Level Tree-Based Model (Random Forest)
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=== 12.14.6 Build a Subscription-Level Tree-Based Model (Random Forest) ===")

# ---------------------------------------------------------
# A) Prepare X and y
# ---------------------------------------------------------
X_sub_rf = X_sub_numeric.copy()
y_sub_rf = y_sub.copy()

# Safety checks
X_sub_rf = X_sub_rf.loc[:, ~X_sub_rf.columns.duplicated()].copy()
X_sub_rf = X_sub_rf.apply(pd.to_numeric, errors="coerce")
y_sub_rf = pd.to_numeric(y_sub_rf, errors="coerce").fillna(0).astype(int)

print("\n--- Subscription-Level RF Input Shapes ---")
print("X_sub_rf shape:", X_sub_rf.shape)
print("y_sub_rf shape:", y_sub_rf.shape)

print("\nTarget distribution:")
print(y_sub_rf.value_counts(dropna=False))

# ---------------------------------------------------------
# B) Train-test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_sub_rf,
    y_sub_rf,
    test_size=0.20,
    random_state=42,
    stratify=y_sub_rf
)

print("\n--- Train/Test Split ---")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(dropna=False))
print("y_test distribution:")
print(y_test.value_counts(dropna=False))

# ---------------------------------------------------------
# C) Build Random Forest pipeline
# ---------------------------------------------------------
sub_rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# ---------------------------------------------------------
# D) Fit model
# ---------------------------------------------------------
sub_rf_model.fit(X_train, y_train)

# ---------------------------------------------------------
# E) Predictions
# ---------------------------------------------------------
y_pred = sub_rf_model.predict(X_test)
y_pred_proba = sub_rf_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# F) Evaluation metrics
# ---------------------------------------------------------
print("\n--- Subscription-Level Random Forest Classification Metrics ---")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_summary)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
display(cm_df)

# ---------------------------------------------------------
# G) Feature importance
# ---------------------------------------------------------
print("\n--- Subscription-Level Random Forest Feature Importance ---")

rf_fitted = sub_rf_model.named_steps["rf"]

importance_df = pd.DataFrame({
    "feature": X_sub_rf.columns,
    "importance": rf_fitted.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 Features by Importance:")
display(importance_df.head(15))

# ---------------------------------------------------------
# H) Majority-class baseline reference
# ---------------------------------------------------------
print("\n--- Majority-Class Baseline Reference ---")

majority_class_accuracy = (y_test == y_test.mode()[0]).mean()

baseline_compare = pd.DataFrame({
    "Model": ["Majority Class Baseline", "Subscription-Level Random Forest"],
    "Accuracy": [majority_class_accuracy, accuracy],
    "ROC-AUC": [0.50, roc_auc]
})

display(baseline_compare)

=== 12.14.6 Build a Subscription-Level Tree-Based Model (Random Forest) ===

--- Subscription-Level RF Input Shapes ---
X_sub_rf shape: (5000, 52)
y_sub_rf shape: (5000,)

Target distribution:
target_churn_subscription
0    4514
1     486
Name: count, dtype: int64

--- Train/Test Split ---
X_train shape: (4000, 52)
X_test shape: (1000, 52)
y_train distribution:
target_churn_subscription
0    3611
1     389
Name: count, dtype: int64
y_test distribution:
target_churn_subscription
0    903
1     97
Name: count, dtype: int64

--- Subscription-Level Random Forest Classification Metrics ---


,Metric,Value
0,Accuracy,0.903000
1,Precision,0.000000
2,Recall,0.000000
3,F1 Score,0.000000
4,ROC-AUC,0.530054



Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       903
           1       0.00      0.00      0.00        97

    accuracy                           0.90      1000
   macro avg       0.45      0.50      0.47      1000
weighted avg       0.82      0.90      0.86      1000


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,903,0
Actual 1,97,0



--- Subscription-Level Random Forest Feature Importance ---

Top 15 Features by Importance:


,feature,importance
26,account_avg_first_response_time_minutes,0.051635
21,avg_usage_duration_secs,0.050984
34,usage_duration_per_event,0.050177
16,total_usage_duration_secs,0.047483
25,account_avg_resolution_time_hours,0.045336
33,usage_count_per_event,0.043031
43,account_ticket_per_seat,0.042560
20,avg_usage_count,0.040415
15,total_usage_count,0.039571
35,error_rate_per_usage,0.038505



--- Majority-Class Baseline Reference ---


,Model,Accuracy,ROC-AUC
0,Majority Class Baseline,0.903,0.500000
1,Subscription-Level Random Forest,0.903,0.530054


### 12.14.6 Interpretation — Subscription-Level Tree-Based Model (Random Forest)

This subscription-level Random Forest model does **not** provide a meaningful improvement in churn prediction. Although the model achieved an apparently high **accuracy of 0.903**, this result is misleading because the subscription-level target is highly imbalanced, and the model simply predicted the majority class.

#### 1. The model failed completely on the churn class
The most important result is that:

- **Precision = 0.000**
- **Recall = 0.000**
- **F1 Score = 0.000**

The confusion matrix confirms that the model predicted **every subscription as non-churned**:

- **903 true negatives**
- **97 false negatives**
- **0 true positives**
- **0 false positives**

This means the model completely failed to identify any churned subscriptions.

---

#### 2. Accuracy is misleading in this case
The majority-class baseline already achieved:

- **Accuracy = 0.903**
- **ROC-AUC = 0.500**

The Random Forest achieved:

- **Accuracy = 0.903**
- **ROC-AUC = 0.530**

So even though the Random Forest slightly exceeded random discrimination in ROC-AUC, it did **not** improve the actual classification outcome in a meaningful operational sense. In practice, it behaved exactly like a majority-class classifier.

---

#### 3. The feature importance ranking is still analytically useful
Even though the classifier failed as a churn detector, the feature importance output still highlights the subscription-level variables the model considered most informative, including:

- `account_avg_first_response_time_minutes`
- `avg_usage_duration_secs`
- `usage_duration_per_event`
- `total_usage_duration_secs`
- `account_avg_resolution_time_hours`
- `usage_count_per_event`
- `account_ticket_per_seat`
- `avg_usage_count`
- `total_usage_count`
- `error_rate_per_usage`

This suggests that the model is still focusing on meaningful domains such as:

- usage intensity
- support response burden
- subscription-level engagement
- product friction

So while prediction failed, the analytical narrative remains consistent: the strongest candidate signals still come from usage, support effort, and friction-related behavior.

---

### Final Conclusion for Step 12.14.6

This result is important because it shows that:

> **moving to the subscription level and switching to a tree-based model still did not produce a usable churn classifier in the current setup.**

The model’s inability to detect even a single churned subscription suggests that:

- the churn signal remains weak,
- the class imbalance is still a major challenge,
- and the current feature set is not yet sufficient for reliable subscription-level prediction.

At this point, the project has strong descriptive and feature-engineering value, but the predictive modeling results remain weak at both the account and subscription levels.



## 12.15 Final Modeling Conclusion and Project Recommendation

The modeling workflow was completed successfully at both the account and subscription levels, including feature engineering, leakage-aware baselines, imbalance-aware experiments, and linear and tree-based classifiers.

Despite these efforts, predictive performance remained weak, which indicates that the current dataset supports strong descriptive churn analysis more than high-confidence predictive churn classification in its present form.

At this point, the technical modeling phase is considered complete, and the project moves to final reporting and recommendation writing.